# Rediseño zero-shot multilingüe · validación contra la rúbrica y NLP del corpus
### Proyecto Twitter/Tesis · WC2026 · notebook 4

La v3 puso el instrumento zero-shot **frente a una segunda medición del mismo corpus**, las
calificaciones de EvaluadorTweets. Esta v4 usa algo que la v3 tenía delante y no miraba: la
**polaridad** que la rúbrica declara en tres de sus cuatro criterios —el nivel 1 es odio o
desprecio, el 5 es fascinación—. Todo el análisis NLP de la v3 partía de *presencia contra
ausencia* y metía en el mismo grupo a los 76 tuits que admiran México y a los 45 que lo desprecian.

### Qué cambia respecto de la v3

| | v3 | v4 |
|---|---|---|
| Polaridad de la rúbrica | sin usar | `polaridad` por tuit en la sección 2, con partición comprobada: **76 positivo · 45 negativo · 19 ambivalente · 4 sin polaridad = 144** |
| Saliencia de violencia | sus 25 tuits iban bajo «nivel 1» en la tabla resumen, junto al odio | columna `presente` propia: **presencia no es valencia** |
| Vocabulario | un contraste, calificados × no calificados | **cinco nubes de palabras** al inicio de la sección 16: dos descriptivas por frecuencia y tres de polaridad por z de log-odds |
| `log_odds` | rótulos cableados a «calificados» | rótulos parametrizados: sirve para cualquier par de grupos |
| Biblioteca de figuras | ECharts 5.6 | ECharts 5.6 + `echarts-wordcloud` 2.1, con **aviso propio** si sólo falla la extensión |

### Qué cambia respecto de la v2

| | v2 | v3 |
|---|---|---|
| Entrada | `turistas_traducido.csv` (54 `id` redondeados por Excel) | `tweets_calificados_anclada.csv`: el mismo corpus reparado + 16 columnas de rúbrica |
| Primera sección de análisis | — | **2 · Resultados de la rúbrica**, con los conteos contrastados contra el HTML original |
| Validación | «descriptivo, NO validado» | **14 · Validación convergente** zero-shot × rúbrica (AUC, ρ, plano con niveles) |
| Criterios | — | **15 · Asociación entre criterios** (Fisher, φ, OR, Mantel–Haenszel) |
| Tuits sin calificar | — | **16 · NLP**: clusters, tópicos, entidades, grafos, auditoría, candidatos a falso negativo |
| Descarga | se había perdido | **17 · Descarga de resultados**, recuperada de la v1 y ampliada |
| Iterar sin GPU | — | `CARGAR_ZEROSHOT` restaura una corrida previa sin cargar modelos |

### Entradas en `/content`

1. **`tweets_calificados_anclada.csv`** (obligatoria).
2. **`graficas_calificaciones_anclada.html`** (opcional): el HTML de EvaluadorTweets, para contrastar
   conteos y verlo tal cual.
3. **`salida/zeroshot_corrida.json.gz`** (opcional): el volcado de una corrida anterior, para saltarte
   la GPU.

Las tres se declaran en la **primera celda de la sección 2**.

### Orden

```
 1  Instalación                          10 Marcos de datos
 2  Resultados de la rúbrica   (CPU)     11 Sesgo entre idiomas
 3  Definición de las dimensiones        12 Tabla resumen
 4  Carga de modelos           (GPU)     13 Figuras del instrumento
 5  Método actual                        14 Validación convergente   (CPU)
 6  Rediseño zero-shot                   15 Asociación entre criterios (CPU)
 7  Configuración                        16 Análisis NLP             (GPU opcional)
 8  Los textos a evaluar                 17 Descarga de resultados
                                         (la 16 abre con las nubes de palabras)
 9  Verificación del ancla               18 Cómo leer esto
```

### Primera corrida y siguientes

**Primera corrida:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)` y ejecuta todo. El
zero-shot sobre los 2 631 tuits tarda unos 11 minutos. Descarga `salida/`.

**Siguientes** (ajustar el NLP, releer la validación): sube `salida/zeroshot_corrida.json.gz`, ponlo en
`CARGAR_ZEROSHOT` y vuelve a ejecutar. No se carga ningún modelo NLI.

**Dos advertencias que atraviesan todo el notebook:**
- La rúbrica **también es un modelo de lenguaje** (Gemini). Contrastar contra ella es validez
  convergente entre dos instrumentos automáticos, **no** validación contra codificación humana.
- Los «candidatos a falso negativo» son material para revisión manual, **no** errores de la rúbrica.
- Las nubes de polaridad describen el vocabulario de **la rúbrica aplicada por un modelo**, sobre
  grupos de 19 a 76 tuits. Son descriptivas: no se concluye nada de ellas.

## 1. Instalación

⚠️ Si la celda te pide reiniciar el entorno, hazlo (`Entorno de ejecución → Reiniciar sesión`)
y vuelve a correr desde aquí.

In [ ]:
import importlib.metadata as md_
import importlib.util
import subprocess, sys

try:
    v = md_.version("transformers")
except md_.PackageNotFoundError:
    v = None
print("transformers instalado:", v)

necesita_reinicio = False
if v is None or int(v.split(".")[0]) >= 5:
    print("instalando transformers 4.x (rama del Dockerfile del proyecto)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5"],
        check=True,
    )
    necesita_reinicio = (v is not None)
else:
    print("version 4.x correcta, no se instala nada")

# ---------------------------------------------------------------------------
# Dependencias de las secciones 14 a 16. Solo se instala lo que falta, y
# SIEMPRE junto con la restriccion de transformers: sentence-transformers
# arrastra la version mas nueva si se le deja, y eso rompe el pipeline de la
# v2, que esta fijado a la rama 4.x.
# ---------------------------------------------------------------------------
_DEPENDENCIAS = {                 # modulo importable -> requisito de pip
    "spacy": "spacy>=3.7,<4",
    "sentence_transformers": "sentence-transformers>=3",
    "umap": "umap-learn",
    "hdbscan": "hdbscan",         # si no compila, la seccion 16 usa sklearn.cluster.HDBSCAN
    "statsmodels": "statsmodels",
    "sklearn": "scikit-learn>=1.3",
}
_faltan = [req for mod, req in _DEPENDENCIAS.items() if importlib.util.find_spec(mod) is None]
if _faltan:
    print("instalando:", ", ".join(_faltan))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_faltan,
                        "transformers>=4.44,<5"])
    if r.returncode != 0:
        print("  aviso: alguna dependencia no se instalo (codigo {}). Si fue hdbscan, la "
              "seccion 16 usa la implementacion de scikit-learn.".format(r.returncode))
else:
    print("dependencias de NLP y estadistica ya presentes")

# El modelo de spaCy de la seccion 16. Si cambias MODELO_SPACY alli, la
# seccion lo descarga sola; este es solo el valor por defecto.
_MODELO_SPACY_DEFECTO = "es_core_news_lg"
if importlib.util.find_spec(_MODELO_SPACY_DEFECTO) is None:
    print("descargando el modelo de spaCy {} (~560 MB)...".format(_MODELO_SPACY_DEFECTO))
    subprocess.run([sys.executable, "-m", "spacy", "download", _MODELO_SPACY_DEFECTO])

# Comprobacion final: nada de lo anterior debe haber movido transformers a 5.x.
try:
    v2_ = md_.version("transformers")
    if int(v2_.split(".")[0]) >= 5:
        print("transformers quedo en", v2_, "-> se reinstala 4.x")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5"])
        necesita_reinicio = True
    elif v2_ != v and v is not None:
        necesita_reinicio = True
except md_.PackageNotFoundError:
    pass

import torch
print("torch :", torch.__version__)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO -- sera MUY lento")

if necesita_reinicio:
    print()
    print("*" * 70)
    print("REINICIA EL ENTORNO AHORA: Entorno de ejecucion -> Reiniciar sesion")
    print("*" * 70)

## 2. Resultados de la rúbrica (EvaluadorTweets)

Antes de medir nada con el instrumento zero-shot, esta sección muestra **lo que ya dice la otra
medición**: los 2 631 tuits calificados por EvaluadorTweets con la rúbrica de imagen de país
(Gemini, cuatro evaluadores aislados). No necesita GPU.

### Las entradas del notebook se declaran aquí

La primera celda declara **todas las entradas**: el CSV calificado, el HTML original de
EvaluadorTweets para contrastar, y —opcional— el volcado de una corrida zero-shot previa. Las
secciones siguientes las leen de aquí.

### Una sola codificación de la ausencia

`rubrica.json` define el nivel **0 como ausencia** en los cuatro criterios («el tuit no expresa /
no evalúa / no menciona…»). El motor de calificación añadió además una bandera `aplicable` que
codifica **lo mismo** por otra vía, y el modelo casi siempre usó la bandera en lugar del 0. Aquí
se resuelve con una regla única:

| `aplicable` | nivel en el CSV | clase | nivel numérico |
|---|---|---|---|
| `NO` | vacío | ausencia | 0 |
| `SI` | 0 | ausencia | 0 |
| `SI` | 1–5 | valencia (R1–R3) · presencia (R4) | 1–5 |
| `BLOQUEADO` | vacío | **faltante** | — |

`BLOQUEADO` es el filtro de contenido del proveedor: **no** es ausencia ni nivel bajo, así que
queda fuera de numeradores y denominadores. Se informa además de dónde viene cada ausencia
(bandera o 0 explícito), para que la colisión quede documentada sin afectar ningún cálculo.

### La escala se declara por criterio

R1–R3 son de **valencia** (1 muy negativo … 5 muy positivo, 3 neutro): paleta divergente. R4 es
de **presencia** (solo 1 = menciona violencia): lleva un color propio. El HTML original pintaba
el 1 de R4 con el rojo de «muy negativo» y lo presentaba como una distribución de valencia de un
único nivel; aquí no.

### Qué reproduce

Las mismas figuras que `graficas_calificaciones_anclada.html` —volumen diario, saliencia,
valencia diaria por criterio, distribución global y por período— con día cortado en
`America/Mexico_City` y los tres períodos de EvaluadorTweets. Si el HTML original está en
`/content`, los conteos se **contrastan automáticamente** contra sus datos embebidos.

**Grupo calificado** = algún criterio con nivel > 0. Es la partición que usan las secciones de
validación, asociación y NLP.

In [ ]:
# === BLOQUE: configuracion-entrada ===
# ===========================================================================
# ENTRADAS DEL NOTEBOOK
#
# Todo lo que el notebook LEE se declara aqui. Las secciones del instrumento,
# la validacion, el NLP y la descarga toman estas rutas de aqui y de ningun
# otro lugar.
# ===========================================================================

# El corpus calificado de EvaluadorTweets: las 52 columnas del corpus mas 16
# de rubrica (nivel, aplicable, puntaje y justificacion por criterio). Trae
# ademas los 54 identificadores que Excel habia redondeado ya reparados.
RUTA_ENTRADA = "/content/tweets_calificados_anclada.csv"

# El HTML de graficas que genero EvaluadorTweets para esta misma corrida.
# Opcional: si esta, sus conteos se contrastan contra los que se recalculan
# aqui, y se puede mostrar tal cual.
RUTA_HTML_RUBRICA = "/content/graficas_calificaciones_anclada.html"
MOSTRAR_HTML_ORIGINAL = True

# Volcado de una corrida zero-shot previa (lo escribe la seccion 8 al
# terminar de evaluar y lo completa la 17). Con una ruta aqui el notebook NO carga ni ejecuta los
# modelos: restaura la corrida y sigue. Sirve para iterar la validacion y el
# NLP sin volver a pagar la GPU. Ejemplo:
#   CARGAR_ZEROSHOT = "/content/salida/zeroshot_corrida.json.gz"
CARGAR_ZEROSHOT = None

# --- como se leen las fechas ------------------------------------------------
# 'created_at' viene en UTC y los cortes son eventos locales: un tuit de las
# 19:00 del 10 de junio en CDMX es la 01:00 del 11 en UTC. Mismos valores que
# config.toml de EvaluadorTweets.
ZONA_HORARIA = "America/Mexico_City"

# Intervalos [desde, hasta): inferior INCLUSIVO, superior EXCLUSIVO. Solo el
# ultimo puede quedar abierto. El 6 de julio no es arbitrario: el ultimo
# partido en territorio mexicano fueron los octavos del 5 de julio.
PERIODOS = [
    {"nombre": "PREVIO A MUNDIAL",              "desde": "2026-05-31", "hasta": "2026-06-11"},
    {"nombre": "DURANTE MUNDIAL EN MEXICO",     "desde": "2026-06-11", "hasta": "2026-07-06"},
    {"nombre": "DESPUES DE PARTIDOS EN MEXICO", "desde": "2026-07-06", "hasta": None},
]

VENTANA_SUAVIZADO = 7          # dias de la media movil (arranca apagada)
UMBRAL_MUESTRA_RUBRICA = 100   # menos tuits en la seleccion -> aviso escrito

# --- columnas del archivo que usa esta seccion ------------------------------
COL_ID = "id"
COL_FECHA = "created_at"
COL_LANG = "lang"
COL_TEXTO = "text"
COL_TRADUCCION = "traducción"
COL_BASE = "base"

# ---------------------------------------------------------------------------
# LOS CRITERIOS. Se detectan del encabezado (columnas `<slug>_nivel`); aqui
# solo se DECLARA lo que el encabezado no dice: nombre corto y tipo de escala.
#
#   valencia   1 muy negativo ... 5 muy positivo, 3 neutro. Paleta divergente.
#   presencia  binaria: 1 = presente. Color propio, nunca el de un negativo.
#
# Un criterio que aparezca en el archivo y no este aqui se trata como de
# valencia y se avisa.
# ---------------------------------------------------------------------------
CRITERIOS_RUBRICA = {
    "rubrica_1_atmosfera_de_mexic": {
        "corto": "atmosfera", "etiqueta": "Atmósfera", "escala": "valencia",
        "nombre": "ATMÓSFERA DE MÉXICO: DIMENSIÓN SIMPÁTICA / EMOCIONAL"},
    "rubrica_2_imagen_cultural_de": {
        "corto": "imagen_cultural", "etiqueta": "Imagen cultural", "escala": "valencia",
        "nombre": "IMAGEN CULTURAL DE MÉXICO: DIMENSIÓN ESTÉTICA"},
    "rubrica_3_perspectiva_politi": {
        "corto": "perspectiva_politica", "etiqueta": "Perspectiva política", "escala": "valencia",
        "nombre": "PERSPECTIVA POLÍTICA DE MÉXICO: DIMENSIÓN FUNCIONAL"},
    "rubrica_4_saliencia_de_viole": {
        "corto": "violencia", "etiqueta": "Saliencia de violencia", "escala": "presencia",
        "nombre": "SALIENCIA DE VIOLENCIA EN MÉXICO"},
}

# Paletas de EvaluadorTweets (viz.py), para que las figuras se lean igual.
PALETA_VALENCIA = ["#c62f2e", "#e8716f", "#c3c2b7", "#6da7ec", "#1c5cab"]  # niveles 1..5
COLOR_PRESENCIA = "#4a3aa7"
COLOR_AUSENCIA = "#898781"
PALETA_IDIOMAS = ["#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7", "#898781"]

# Codigos con que la API marca "no hay texto del cual inferir idioma".
CODIGOS_SIN_IDIOMA_RUBRICA = {"und", "zxx", "qme", "qam", "qct", "qht", "qst", "art"}
ETIQUETA_SIN_IDIOMA_RUBRICA = "(sin idioma)"

CDN_ECHARTS = "https://cdnjs.cloudflare.com/ajax/libs/echarts/5.6.0/echarts.min.js"
# Las nubes de palabras son una EXTENSION aparte: echarts-wordcloud 2.x envuelve
# wordcloud2.js y registra el tipo de serie `wordCloud`. Segundo punto de falla,
# con su propio aviso en la pagina.
CDN_ECHARTS_WORDCLOUD = ("https://cdnjs.cloudflare.com/ajax/libs/"
                         "echarts-wordcloud/2.1.0/echarts-wordcloud.min.js")

import os as _os_ent
print("entrada              :", RUTA_ENTRADA,
      "" if _os_ent.path.exists(RUTA_ENTRADA) else "  <-- NO EXISTE: subelo a /content")
print("HTML de contraste    :", RUTA_HTML_RUBRICA,
      "" if _os_ent.path.exists(RUTA_HTML_RUBRICA) else "  (no esta: se omite el contraste)")
print("corrida previa       :", CARGAR_ZEROSHOT or "ninguna: se evaluara con los modelos")
print("zona horaria         :", ZONA_HORARIA)
for _p in PERIODOS:
    print("   periodo {:<32} [{} , {})".format(_p["nombre"], _p["desde"], _p["hasta"] or "..."))

In [ ]:
# ===========================================================================
# INFRAESTRUCTURA DE LAS FIGURAS NUEVAS
#
# Las secciones 2, 14, 15 y 16 dibujan con Apache ECharts sobre la misma
# pagina base. Cada seccion registra sus DATOS (ya agregados en Python) y su
# JavaScript con `registrar_seccion`; `mostrar_seccion` la dibuja aqui, y la
# seccion 17 junta todas en un solo archivo autocontenido.
#
# Una pagina, dos destinos, igual que en la seccion 13: la misma funcion
# construye lo que se ve en el notebook y lo que se escribe a disco.
# ===========================================================================
import json
import math
from IPython.display import HTML, display

SECCIONES_V3 = {}

PLANTILLA_V3 = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITULO__</title>
<script src="__CDN__"></script>
<script src="__CDN_NUBE__"></script>
<style>
  :root{
    color-scheme: light;
    --sup:#fcfcfb; --plano:#f9f9f7; --tinta:#0b0b0b; --tinta2:#52514e;
    --mudo:#898781; --grid:#e1e0d9; --eje:#c3c2b7; --alerta:#d03b3b; --ok:#0ca30c;
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--plano);color:var(--tinta);
       font:14px/1.55 system-ui,-apple-system,"Segoe UI",sans-serif}
  .wrap{max-width:1120px;margin:0 auto;padding-block:24px;padding-left:16px;padding-right:16px}
  h1{font-size:22px;margin:0 0 4px}
  h2{font-size:18px;margin:30px 0 8px;padding-bottom:6px;border-bottom:1px solid var(--grid)}
  h3{font-size:15px;margin:0 0 2px}
  p{margin:6px 0}
  .sub{color:var(--tinta2);margin:0 0 14px}
  .fig{background:var(--sup);border:1px solid var(--grid);border-radius:8px;padding:16px;margin:14px 0}
  .preg{color:var(--tinta2);font-style:italic;margin:0 0 8px}
  .n{display:inline-block;font-size:11px;color:var(--tinta2);background:var(--plano);
     border:1px solid var(--grid);border-radius:99px;padding:1px 9px;margin:0 6px 8px 0;
     font-variant-numeric:tabular-nums}
  .aviso{background:#fff6e8;border-left:3px solid #eda100;padding:8px 11px;font-size:13px;
         color:var(--tinta2);margin:8px 0}
  .alerta{background:#fdf0f0;border-left:3px solid var(--alerta);padding:8px 11px;font-size:13px;
          color:var(--tinta2);margin:8px 0}
  .bien{background:#f1f8f1;border-left:3px solid var(--ok);padding:8px 11px;font-size:13px;
        color:var(--tinta2);margin:8px 0}
  .banner{background:#fff6e8;border:1px solid #eda100;border-radius:8px;padding:10px 14px;margin:0 0 14px}
  .nota{font-size:13px;color:var(--tinta2);margin-top:10px}
  .lienzo{width:100%}
  .controles{display:flex;flex-wrap:wrap;gap:10px 22px;align-items:center;margin:8px 0 4px;font-size:13px}
  .controles .grupo{display:flex;flex-wrap:wrap;gap:6px;align-items:center}
  .controles .grupo > span{color:var(--tinta2);font-weight:600;margin-right:4px}
  .chip{border:1px solid var(--eje);background:var(--sup);border-radius:99px;padding:2px 9px;
        font-size:12px;cursor:pointer;color:var(--tinta)}
  .chip[aria-pressed="false"]{opacity:.45;text-decoration:line-through}
  .chip small{color:var(--tinta2)}
  .rejilla{display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:12px}
  details{margin-top:10px}
  summary{cursor:pointer;font-size:13px;color:var(--tinta2)}
  table{border-collapse:collapse;width:100%;font-size:12px;margin-top:8px;font-variant-numeric:tabular-nums}
  th,td{border-bottom:1px solid var(--grid);padding:4px 7px;text-align:left;vertical-align:top}
  th{color:var(--tinta2);font-weight:600}
  td.num{text-align:right}
  .tabla-scroll{overflow-x:auto}
  .fallo{background:var(--sup);border:1px solid var(--alerta);border-radius:8px;padding:20px}
  code{background:var(--plano);border:1px solid var(--grid);border-radius:3px;padding:0 4px;font-size:12px}
</style>
</head>
<body>
<div class="wrap" id="raiz"></div>
<script>
window.__ERRORES__ = [];
window.addEventListener('error', e => { window.__ERRORES__.push(String(e.message)); });
const RAIZ = document.getElementById('raiz');
function el(h){const t=document.createElement('template');t.innerHTML=h.trim();return t.content.firstChild;}
function esc(s){return String(s==null?'':s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));}
function fix(v,n){return (v==null||(typeof v==='number'&&isNaN(v)))?'—':Number(v).toFixed(n==null?3:n);}
function pct(v,n){return (v==null||isNaN(v))?'—':(v*100).toFixed(n==null?1:n)+'%';}
function tabla(cab,filas){
  return '<div class="tabla-scroll"><table><thead><tr>'+cab.map(c=>'<th>'+esc(c)+'</th>').join('')
    +'</tr></thead><tbody>'+filas.map(f=>'<tr>'+f.map(v=>typeof v==='number'
      ?'<td class="num">'+(Number.isInteger(v)?v.toLocaleString('es'):fix(v))+'</td>'
      :'<td>'+esc(v)+'</td>').join('')+'</tr>').join('')+'</tbody></table></div>';
}
const C = {tinta:'#0b0b0b',tinta2:'#52514e',mudo:'#898781',grid:'#e1e0d9',eje:'#c3c2b7',sup:'#fcfcfb'};
const TOOLBOX = {right:6,top:-2,feature:{saveAsImage:{type:'svg',title:'guardar SVG',name:'figura',
  backgroundColor:'#fcfcfb'}}};
const EJE_X = {axisLine:{lineStyle:{color:C.eje}},axisTick:{show:false},axisLabel:{color:C.tinta2,fontSize:11},
  nameTextStyle:{color:C.tinta2}};
const EJE_Y = {axisLine:{show:false},axisTick:{show:false},axisLabel:{color:C.tinta2,fontSize:11},
  splitLine:{lineStyle:{color:C.grid,width:1}},nameTextStyle:{color:C.tinta2}};
const GRAFICAS = [];
function init(dom,opt){
  const c=echarts.init(dom,null,{renderer:'svg'}); c.setOption(opt,true); GRAFICAS.push(c); return c;
}
window.addEventListener('resize',()=>GRAFICAS.forEach(g=>g.resize()));
/* Un panel: titulo, pregunta, n, avisos, lienzo y tabla de respaldo. Devuelve el lienzo. */
function panel(cont,o){
  const s=el('<section class="fig"></section>');
  let h='<h3>'+o.titulo+'</h3>';
  if(o.pregunta) h+='<p class="preg">'+o.pregunta+'</p>';
  (o.chips||[]).forEach(t=>h+='<span class="n">'+t+'</span>');
  (o.avisos||[]).forEach(t=>h+='<p class="aviso">'+t+'</p>');
  (o.alertas||[]).forEach(t=>h+='<p class="alerta">'+t+'</p>');
  if(o.controles) h+='<div class="controles">'+o.controles+'</div>';
  if(o.alto!==0) h+='<div class="lienzo" style="height:'+(o.alto||340)+'px"></div>';
  if(o.html) h+=o.html;
  if(o.nota) h+='<p class="nota">'+o.nota+'</p>';
  if(o.tabla) h+='<details><summary>ver los datos de esta figura</summary>'+o.tabla+'</details>';
  s.innerHTML=h; cont.appendChild(s);
  return {seccion:s, lienzo:s.querySelector('.lienzo')};
}
</script>
__SECCIONES__
<script>
/* DOS modos de falla, no uno. Si falta ECharts no hay pagina; si falta solo la
   extension de nubes, el resto de las figuras dibuja bien y lo unico que se
   pierde son las nubes. Sin esta distincion la pagina se veria sana con cinco
   paneles vacios en silencio. */
window.__HAY_NUBE__ = (function(){
  if (typeof echarts === 'undefined') return false;
  try {
    /* La prueba es el tipo de serie, no la etiqueta <script>: ECharts no lanza
       con un `type` que no conoce, simplemente no crea la serie. */
    const d = document.createElement('div');
    d.style.cssText = 'width:64px;height:64px;position:absolute;left:-9999px;top:0';
    document.body.appendChild(d);
    const ch = echarts.init(d);
    ch.setOption({series:[{type:'wordCloud', data:[{name:'x', value:1}]}]});
    const ok = ch.getModel().getSeriesByType('wordCloud').length > 0;
    ch.dispose(); d.remove();
    return ok;
  } catch (e) { return false; }
})();
window.__AVISO_NUBE__ = 'No se pudo cargar <code>echarts-wordcloud</code> desde '
  + '<code>__CDN_NUBE__</code>. El resto de la página sí dibuja; sólo faltan las nubes.';

if (typeof echarts === 'undefined') {
  RAIZ.innerHTML = '<div class="fallo"><h1>No se pudo cargar la biblioteca de gráficas</h1>'
    + '<p>Esta página dibuja con Apache ECharts desde <code>__CDN__</code> y la petición no llegó. '
    + 'Los datos sí están dentro del archivo: recupera la conexión y recarga.</p></div>';
} else {
  (window.__DIBUJAR__||[]).forEach(f => { try { f(); } catch (e) {
    window.__ERRORES__.push(String(e && e.stack || e));
    RAIZ.appendChild(el('<p class="alerta">Error al dibujar una sección: '+esc(e)+'</p>'));
  }});
}
</script>
</body>
</html>
"""

_BLOQUE_SECCION = """<script>
(window.__DIBUJAR__ = window.__DIBUJAR__ || []).push(function(){
  const DATOS = __DATOS__;
  const RAIZ_SEC = document.createElement('div');
  RAIZ.appendChild(RAIZ_SEC);
__JS__
});
</script>"""


def _json_seguro(obj):
    """JSON embebible en <script>: sin NaN (no es JSON valido) y sin cerrar la etiqueta."""
    def limpiar(o):
        if isinstance(o, float):
            return None if (math.isnan(o) or math.isinf(o)) else o
        if isinstance(o, dict):
            return {str(k): limpiar(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return [limpiar(v) for v in o]
        if hasattr(o, "item"):          # escalares de numpy
            return limpiar(o.item())
        return o
    return json.dumps(limpiar(obj), ensure_ascii=False).replace("</", "<\\/")


def registrar_seccion(clave, titulo, datos, js):
    """Guarda una seccion para mostrarla aqui y para el archivo de la seccion 17."""
    SECCIONES_V3[clave] = {"titulo": titulo, "datos": datos, "js": js}


def pagina_v3(claves=None, titulo="Figuras v3"):
    claves = list(SECCIONES_V3) if claves is None else claves
    bloques = []
    for k in claves:
        s = SECCIONES_V3[k]
        bloques.append(_BLOQUE_SECCION.replace("__DATOS__", _json_seguro(s["datos"]))
                       .replace("__JS__", s["js"]))
    # __CDN_NUBE__ va PRIMERO: sustituir __CDN__ antes dejaria el otro marcador
    # a medio reemplazar si alguien le cambia el nombre.
    return (PLANTILLA_V3.replace("__TITULO__", titulo)
            .replace("__CDN_NUBE__", CDN_ECHARTS_WORDCLOUD)
            .replace("__CDN__", CDN_ECHARTS)
            .replace("__SECCIONES__", "\n".join(bloques)))


def mostrar_seccion(clave):
    pagina = pagina_v3([clave], SECCIONES_V3[clave]["titulo"])
    print("figuras de la seccion: {:.0f} KB embebidos".format(len(pagina.encode("utf-8")) / 1024))
    display(HTML(pagina))
    return pagina


print("infraestructura de figuras lista")

In [ ]:
# === BLOQUE: rubrica-y-polaridad ===
import os
import re
import numpy as np
import pandas as pd

# ===========================================================================
# EL MARCO DE LA RUBRICA
#
#   RUBRICA       una fila por tuit x criterio   -> clase, nivel_num, origen
#   RUBRICA_TUIT  una fila por tuit              -> fecha local, periodo,
#                 idioma, autosuficiencia, grupo, y nivel/presencia por criterio
#
# Todas las secciones nuevas leen de aqui. La regla de codificacion vive en
# UNA funcion, `clasificar_calificacion`, y en ningun otro lugar.
# ===========================================================================

RUBRICA = None
RUBRICA_TUIT = None
CRITERIOS = []          # [{slug, corto, etiqueta, nombre, escala}] en orden de la rubrica
HAY_RUBRICA = False


def clasificar_calificacion(aplicable, nivel_txt, escala):
    """(clase, nivel_num, origen_ausencia) para una calificacion.

    La rubrica define el nivel 0 como ausencia. La bandera `aplicable` del
    motor codifica lo mismo por otra via, asi que NO y nivel 0 son la MISMA
    clase. BLOQUEADO es el filtro del proveedor: no dice nada del tuit, es un
    dato faltante y nunca se cuenta como ausencia.
    """
    ap = str(aplicable or "").strip().upper()
    nv = str(nivel_txt or "").strip()
    if ap == "BLOQUEADO":
        return "faltante", np.nan, None
    if nv == "":
        if ap == "NO":
            return "ausencia", 0.0, "aplicable_no"
        # aplicable SI (o vacio) sin nivel: no hay calificacion que leer
        return "faltante", np.nan, "inconsistente"
    try:
        n = float(nv)
    except ValueError:
        return "faltante", np.nan, "inconsistente"
    if n == 0:
        return "ausencia", 0.0, ("aplicable_no" if ap == "NO" else "nivel_0")
    return ("presencia" if escala == "presencia" else "valencia"), n, None


# Niveles de la escala de valencia. La rubrica ancla el 1 en odio, desprecio o
# juicio fuertemente negativo y el 5 en fascinacion o elogio superlativo; el 3
# es ambivalencia declarada ("emocion mixta", "sin valencia clara").
NIVELES_NEGATIVOS = (1.0, 2.0)
NIVEL_AMBIVALENTE = 3.0
NIVELES_POSITIVOS = (4.0, 5.0)

POLARIDADES = ("positivo", "negativo", "ambivalente", "sin_polaridad")


def polaridad_de_tuit(niveles_valencia):
    """Polaridad de un tuit a partir de los niveles de sus criterios de VALENCIA.

    Los criterios de PRESENCIA no entran. Un tuit que menciona violencia no es
    un tuit negativo: es un tuit que la menciona. De los 25 con saliencia de
    violencia en el corpus, 4 son positivos o ambivalentes -- "Mexico es
    maravilloso si evitas que te estafen" --, y sumar la presencia los
    arrastraria a negativo.

    Los polos van PRIMERO y la ambivalencia es el residuo. Definir ambivalente
    como "algun criterio en 3" parte en dos a tres tuits que tienen polo claro
    y ambivalencia en un criterio secundario, y la particion deja de sumar.

    Devuelve "conflicto" cuando hay un criterio en 1-2 y otro en 4-5 a la vez.
    Los datos de hoy no traen ninguno y la asercion de abajo lo caza: la
    rubrica no lo impide, asi que no se supone, se comprueba.
    """
    vs = [v for v in niveles_valencia if v == v and v > 0]   # faltante y ausencia fuera
    if not vs:
        return "sin_polaridad"
    hay_pos = any(v in NIVELES_POSITIVOS for v in vs)
    hay_neg = any(v in NIVELES_NEGATIVOS for v in vs)
    if hay_pos and not hay_neg:
        return "positivo"
    if hay_neg and not hay_pos:
        return "negativo"
    if all(v == NIVEL_AMBIVALENTE for v in vs):
        return "ambivalente"
    return "conflicto"


def _a_utc(serie):
    """El corpus mezcla '2026-06-03T23:59:50' con '2026-07-04 7:02:11'. Las
    marcas sin huso son UTC, que es lo que entrega la API. Igual que viz.py."""
    for extra in ({"format": "mixed"}, {"format": "ISO8601"}, {}):
        try:
            return pd.to_datetime(serie, errors="coerce", utc=True, **extra)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(serie, errors="coerce", utc=True)


def _es_verdadero(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "si", "sí"])


def _periodo(fecha_local_dia):
    per = pd.Series(pd.NA, index=fecha_local_dia.index, dtype="object")
    for p in PERIODOS:
        dentro = fecha_local_dia >= pd.Timestamp(p["desde"])
        if p["hasta"]:
            dentro &= fecha_local_dia < pd.Timestamp(p["hasta"])
        per = per.mask(dentro & per.isna(), p["nombre"])
    return per


def construir_rubrica(ruta):
    global RUBRICA, RUBRICA_TUIT, CRITERIOS, HAY_RUBRICA, RUBRICA_ORIGEN_DF
    df = pd.read_csv(ruta, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    RUBRICA_ORIGEN_DF = df

    slugs = [m.group(1) for c in df.columns
             for m in [re.match(r"^(rubrica_\d+_.+)_nivel$", c)] if m]
    slugs = sorted(slugs, key=lambda s: int(re.match(r"rubrica_(\d+)_", s).group(1)))
    if not slugs:
        HAY_RUBRICA = False
        print("El archivo no trae columnas de rubrica (<slug>_nivel). Esta seccion y")
        print("las de validacion, asociacion y grupos del NLP se OMITEN. El instrumento")
        print("zero-shot corre igual.")
        return

    CRITERIOS = []
    for s in slugs:
        dec = CRITERIOS_RUBRICA.get(s)
        if dec is None:
            print("   aviso: criterio '{}' no declarado en CRITERIOS_RUBRICA; se trata "
                  "como de valencia".format(s))
            dec = {"corto": s, "etiqueta": s, "nombre": s, "escala": "valencia"}
        faltan = [c for c in (s + "_nivel", s + "_aplicable") if c not in df.columns]
        if faltan:
            raise KeyError("El criterio {} no trae {}".format(s, faltan))
        CRITERIOS.append(dict(dec, slug=s))

    ids = df[COL_ID].astype(str).str.strip()
    if ids.duplicated().any():
        raise ValueError("{} identificadores repetidos en la entrada: la rubrica se une por "
                         "identificador y no puede haber dos filas con el mismo."
                         .format(int(ids.duplicated().sum())))

    # --- una fila por tuit x criterio --------------------------------------
    filas = []
    for c in CRITERIOS:
        s = c["slug"]
        ap = df[s + "_aplicable"].tolist()
        nv = df[s + "_nivel"].tolist()
        just = df[s + "_justificacion"].tolist() if s + "_justificacion" in df else [""] * len(df)
        for i in range(len(df)):
            clase, nnum, origen = clasificar_calificacion(ap[i], nv[i], c["escala"])
            filas.append((ids.iat[i], s, c["corto"], c["escala"], ap[i], nv[i],
                          clase, nnum, origen, just[i]))
    RUBRICA = pd.DataFrame(filas, columns=["id", "slug", "corto", "escala", "aplicable",
                                           "nivel_txt", "clase", "nivel_num",
                                           "origen_ausencia", "justificacion"])
    RUBRICA["presente"] = RUBRICA["nivel_num"] > 0          # NaN > 0 es False

    # --- una fila por tuit -------------------------------------------------
    t = pd.DataFrame({"id": ids})
    dt_utc = _a_utc(df[COL_FECHA])
    try:
        dt_local = dt_utc.dt.tz_convert(ZONA_HORARIA)
        zona = ZONA_HORARIA
    except Exception as exc:
        print("   aviso: no se pudo convertir a {} ({}); se agrupa en UTC y los cortes "
              "quedan corridos hasta 6 h".format(ZONA_HORARIA, exc))
        dt_local, zona = dt_utc, "UTC"
    t["fecha_utc"] = dt_utc
    t["fecha"] = dt_local.dt.normalize().dt.tz_localize(None)
    t["fecha_str"] = t["fecha"].dt.strftime("%Y-%m-%d")
    t["periodo"] = _periodo(t["fecha"])
    lang = df[COL_LANG].astype(str).str.strip()
    t["lang"] = lang
    # Mismo criterio que EvaluadorTweets: el codigo se conserva y se marca;
    # solo el vacio se renombra.
    t["idioma"] = lang.where(lang != "", ETIQUETA_SIN_IDIOMA_RUBRICA)
    t["sin_idioma"] = t["idioma"].isin(CODIGOS_SIN_IDIOMA_RUBRICA | {ETIQUETA_SIN_IDIOMA_RUBRICA})
    # Autosuficiencia: la misma regla que corpus.detectar_contexto_incompleto.
    es_reply = (df["in_reply_to_user_id"].astype(str).str.strip() != "") \
        if "in_reply_to_user_id" in df else pd.Series(False, index=df.index)
    es_reply |= df[COL_TEXTO].astype(str).str.lstrip().str.startswith("@")
    es_rt = _es_verdadero(df["is_retweet"]) if "is_retweet" in df else False
    es_qt = _es_verdadero(df["is_quote"]) if "is_quote" in df else False
    t["autosuficiente"] = ~(es_reply | es_rt | es_qt)
    t["base"] = df[COL_BASE] if COL_BASE in df else ""

    for c in CRITERIOS:
        sub = RUBRICA[RUBRICA.slug == c["slug"]].set_index("id")
        t["nivel_" + c["corto"]] = t["id"].map(sub["nivel_num"])
        t["clase_" + c["corto"]] = t["id"].map(sub["clase"])
        t["presente_" + c["corto"]] = t["id"].map(sub["presente"]).astype(bool)
    pres = t[["presente_" + c["corto"] for c in CRITERIOS]]
    falt = t[["clase_" + c["corto"] for c in CRITERIOS]] == "faltante"
    t["calificado"] = pres.any(axis=1)
    t["con_faltante"] = falt.any(axis=1)
    t["grupo"] = np.where(t["calificado"], "calificado", "no_calificado")

    # --- polaridad ---------------------------------------------------------
    # Los criterios de valencia se DERIVAN de la escala declarada, nunca del
    # nombre: anadir o quitar un criterio a la rubrica no debe obligar a tocar
    # esta linea.
    cortos_valencia = [c["corto"] for c in CRITERIOS if c["escala"] == "valencia"]
    cols_valencia = ["nivel_" + k for k in cortos_valencia]
    t["polaridad"] = [polaridad_de_tuit(fila) for fila in t[cols_valencia].to_numpy()]
    # Sin ningun criterio presente no hay nada que polarizar: esos tuits no
    # son "sin polaridad", son otro grupo.
    t.loc[~t["calificado"], "polaridad"] = "no_calificado"

    # La particion es disjunta por construccion -- polaridad_de_tuit devuelve
    # una sola etiqueta por tuit --, asi que lo que hay que comprobar es que
    # sea EXHAUSTIVA y que no aparezca el conflicto que la rubrica no impide.
    _conflicto = t.index[t["polaridad"] == "conflicto"]
    if len(_conflicto):
        _detalle = "\n".join(
            "     {}   {}".format(
                t.at[i, "id"],
                ", ".join("{}={}".format(k, t.at[i, "nivel_" + k]) for k in cortos_valencia))
            for i in _conflicto)
        raise AssertionError(
            "{} tuit(s) con un criterio de valencia en 1-2 y otro en 4-5 a la vez. "
            "La particion de polaridad deja de tener una lectura unica; decide la "
            "regla antes de seguir:\n{}".format(len(_conflicto), _detalle))
    _en_particion = int(t["polaridad"].isin(POLARIDADES).sum())
    _calificados = int(t["calificado"].sum())
    assert _en_particion == _calificados, (
        "La particion de polaridad cubre {} tuits y los calificados son {}."
        .format(_en_particion, _calificados))

    RUBRICA_TUIT = t
    HAY_RUBRICA = True

    # --- reporte -----------------------------------------------------------
    print("=" * 78)
    print("RUBRICA · {} tuits x {} criterios   ({})".format(len(t), len(CRITERIOS), ruta))
    print("=" * 78)
    print("  {:<24} {:>9} {:>9} {:>9} {:>9}   ausencia por: {:>6} {:>7}".format(
        "criterio", "ausencia", "val/pres", "faltante", "incons.", "NO", "nivel 0"))
    for c in CRITERIOS:
        g = RUBRICA[RUBRICA.slug == c["slug"]]
        print("  {:<24} {:>9} {:>9} {:>9} {:>9}   {:>20} {:>7}".format(
            c["etiqueta"], int((g.clase == "ausencia").sum()),
            int(g.clase.isin(["valencia", "presencia"]).sum()),
            int((g.clase == "faltante").sum()),
            int((g.origen_ausencia == "inconsistente").sum()),
            int((g.origen_ausencia == "aplicable_no").sum()),
            int((g.origen_ausencia == "nivel_0").sum())))
        niveles = g[g.presente]["nivel_num"].value_counts().sort_index()
        print("  {:<24} niveles > 0: {}   escala: {}".format(
            "", ", ".join("{}:{}".format(int(k), v) for k, v in niveles.items()) or "ninguno",
            c["escala"]))
    print()
    print("  GRUPOS (calificado = algun criterio con nivel > 0)")
    print("     calificados           {:>6}".format(int(t.calificado.sum())))
    print("     no calificados        {:>6}   de ellos con algun criterio faltante: {}".format(
        int((~t.calificado).sum()), int((~t.calificado & t.con_faltante).sum())))
    print()
    print("  POLARIDAD (solo criterios de {}; la presencia no entra)".format(
        ", ".join(cortos_valencia)))
    for _p in POLARIDADES:
        _nota = "   sin ningun criterio de valencia presente" if _p == "sin_polaridad" else ""
        print("     {:<22}{:>6}{}".format(_p, int((t.polaridad == _p).sum()), _nota))
    print()
    ilegibles = int(t["fecha"].isna().sum())
    print("  dia cortado en {} · fechas ilegibles: {} · fuera de todo periodo: {}".format(
        zona, ilegibles, int(t["periodo"].isna().sum())))
    print("  autosuficientes {} de {} (respuestas, retweets y citas no lo son)".format(
        int(t.autosuficiente.sum()), len(t)))
    for p in PERIODOS:
        s = t[t.periodo == p["nombre"]]
        print("     {:<32} {:>3} dias {:>6} tuits".format(
            p["nombre"], s["fecha_str"].nunique(), len(s)))
    print("=" * 78)


RUBRICA_ORIGEN_DF = None
if os.path.exists(RUTA_ENTRADA):
    construir_rubrica(RUTA_ENTRADA)
else:
    print("No existe {}. Subelo al panel de archivos de Colab o corrige RUTA_ENTRADA."
          .format(RUTA_ENTRADA))

In [ ]:
import json
import os
import pandas as pd

# ===========================================================================
# AGREGADOS DE LA RUBRICA Y CONTRASTE CONTRA EL HTML ORIGINAL
#
# Se agrega ANTES de dibujar y se embebe solo el agregado (dia x criterio x
# clase x nivel x idioma x autosuficiencia), como hace EvaluadorTweets: los
# filtros de la pagina son reducciones sobre un arreglo chico.
# ===========================================================================


def agregar_rubrica():
    v = RUBRICA.merge(RUBRICA_TUIT[["id", "fecha_str", "idioma", "autosuficiente"]],
                      on="id", how="left")
    v = v[v["fecha_str"].notna()].copy()
    v["nivel"] = v.apply(lambda r: str(int(r["nivel_num"]))
                         if r["clase"] in ("valencia", "presencia") else "", axis=1)
    agg = (v.groupby(["fecha_str", "slug", "clase", "nivel", "idioma", "autosuficiente"])
             .size().reset_index(name="n"))
    tv = RUBRICA_TUIT[RUBRICA_TUIT["fecha_str"].notna()]
    vol = (tv.groupby(["fecha_str", "idioma", "autosuficiente"]).size().reset_index(name="n"))
    por_idioma = tv.groupby("idioma").size().sort_values(ascending=False)
    periodos = []
    for p in PERIODOS:
        s = tv[tv.periodo == p["nombre"]]
        d = int(s["fecha_str"].nunique())
        periodos.append({"nombre": p["nombre"], "dias": d, "tuits": int(len(s)),
                         "por_dia": round(len(s) / max(d, 1), 1)})
    crits = []
    for c in CRITERIOS:
        g = RUBRICA[RUBRICA.slug == c["slug"]]
        origen = g["origen_ausencia"].value_counts().to_dict()
        crits.append({
            "slug": c["slug"], "etiqueta": c["etiqueta"], "nombre": c["nombre"],
            "escala": c["escala"],
            "niveles": ["1", "2", "3", "4", "5"] if c["escala"] == "valencia" else ["1"],
            "conteo": {"ausencia": int((g.clase == "ausencia").sum()),
                       "faltante": int((g.clase == "faltante").sum()),
                       "presentes": int(g.presente.sum()),
                       "ausencia_por_aplicable_no": int(origen.get("aplicable_no", 0)),
                       "ausencia_por_nivel_0": int(origen.get("nivel_0", 0))},
        })
    return {
        "criterios": crits,
        "fechas": sorted(tv["fecha_str"].unique()),
        "periodos": periodos,
        "cortes": PERIODOS,
        "idiomas": [{"codigo": k, "n": int(n),
                     "sin_idioma": k in CODIGOS_SIN_IDIOMA_RUBRICA or k == ETIQUETA_SIN_IDIOMA_RUBRICA}
                    for k, n in por_idioma.items()],
        "agg": agg.to_dict("records"),
        "volumen": vol.to_dict("records"),
        "umbral_muestra": UMBRAL_MUESTRA_RUBRICA,
        "ventana": VENTANA_SUAVIZADO,
        "paleta_valencia": PALETA_VALENCIA,
        "color_presencia": COLOR_PRESENCIA,
        "color_ausencia": COLOR_AUSENCIA,
        "paleta_idiomas": PALETA_IDIOMAS,
        "zona": ZONA_HORARIA,
        "grupos": {"calificados": int(RUBRICA_TUIT.calificado.sum()),
                   "no_calificados": int((~RUBRICA_TUIT.calificado).sum()),
                   "no_calificados_con_faltante":
                       int((~RUBRICA_TUIT.calificado & RUBRICA_TUIT.con_faltante).sum())},
    }


def datos_html_original(ruta):
    """El bloque de datos que EvaluadorTweets embebe en su pagina."""
    with open(ruta, encoding="utf-8") as fh:
        h = fh.read()
    i = h.find('{"criterios"')
    if i < 0:
        return None
    datos, _ = json.JSONDecoder().raw_decode(h[i:])
    return datos


def contrastar_con_original(mios, suyos):
    """Conteos por criterio x clase x nivel. EvaluadorTweets llama 'valencia' a
    todo nivel graficable y 'fallida' a lo que aqui es faltante."""
    mapa = {"presencia": "valencia", "faltante": "fallida"}
    a = pd.DataFrame(mios["agg"])
    a["clase"] = a["clase"].map(lambda c: mapa.get(c, c))
    a = a.groupby(["slug", "clase", "nivel"])["n"].sum()
    b = pd.DataFrame(suyos["agg"]).groupby(["slug", "clase", "nivel"])["n"].sum()
    comp = pd.concat([a.rename("recalculado"), b.rename("original")], axis=1).fillna(0).astype(int)
    diffs = comp[comp.recalculado != comp.original]
    vol_mio = int(pd.DataFrame(mios["volumen"])["n"].sum())
    vol_suyo = int(pd.DataFrame(suyos["volumen"])["n"].sum())
    per_mio = {p["nombre"]: p["tuits"] for p in mios["periodos"]}
    per_suyo = {p["nombre"]: p["tuits"] for p in suyos.get("periodos", [])}
    return comp, diffs, (vol_mio, vol_suyo), (per_mio, per_suyo)


JS_RUBRICA = r"""
const D = DATOS;
const PALV = D.paleta_valencia, PALI = D.paleta_idiomas;
RAIZ_SEC.appendChild(el('<h2>Resultados de la rúbrica · EvaluadorTweets</h2>'));
RAIZ_SEC.appendChild(el('<p class="sub">'+D.fechas.length+' días · día cortado en '+esc(D.zona)+' · '
  + D.periodos.map(p=>esc(p.nombre)+': '+p.tuits.toLocaleString('es')).join(' · ')
  + ' · calificados (algún nivel &gt; 0): <b>'+D.grupos.calificados+'</b> · no calificados: '
  + D.grupos.no_calificados+' ('+D.grupos.no_calificados_con_faltante+' con algún criterio bloqueado)</p>'));
RAIZ_SEC.appendChild(el('<p class="banner"><b>Codificación.</b> Ausencia = nivel 0 o <code>aplicable=NO</code> '
  + '(la misma clase). <code>BLOQUEADO</code> = faltante, fuera de numeradores y denominadores. '
  + 'Calificaciones hechas por un modelo de lenguaje: NO es codificación humana.</p>'));

const sel = new Set(D.idiomas.map(x=>x.codigo));
let modo='prop', suavizar=false, soloAuto=false;
const ctl = el('<section class="fig"><div class="controles">'
  + '<div class="grupo"><span>Idioma</span><span class="chips"></span></div>'
  + '<div class="grupo"><span>Medida</span><label><input type="radio" name="modo-rub" value="prop" checked> proporción</label>'
  + '<label><input type="radio" name="modo-rub" value="abs"> conteo</label></div>'
  + '<div class="grupo"><span>Opciones</span><label><input type="checkbox" class="suav"> media móvil '+D.ventana+' días</label>'
  + '<label><input type="checkbox" class="auto"> sólo tuits autosuficientes</label></div>'
  + '</div><p class="aviso" hidden></p></section>');
RAIZ_SEC.appendChild(ctl);
const chips = ctl.querySelector('.chips'), aviso = ctl.querySelector('.aviso');
D.idiomas.forEach(x=>{
  const b=el('<button class="chip" aria-pressed="true">'+esc(x.codigo)+' <small>'+x.n.toLocaleString('es')+'</small></button>');
  if(x.sin_idioma) b.title='No es un idioma: la API lo usa cuando no hay texto del cual inferirlo';
  b.onclick=()=>{ if(sel.has(x.codigo)){sel.delete(x.codigo);b.setAttribute('aria-pressed','false');}
                  else {sel.add(x.codigo);b.setAttribute('aria-pressed','true');} pintar(); };
  chips.appendChild(b);
});
ctl.querySelectorAll('input[name=modo-rub]').forEach(r=>r.onchange=e=>{modo=e.target.value;pintar();});
ctl.querySelector('.suav').onchange=e=>{suavizar=e.target.checked;pintar();};
ctl.querySelector('.auto').onchange=e=>{soloAuto=e.target.checked;pintar();};

const pasa = r => sel.has(r.idioma) && (!soloAuto || r.autosuficiente === true);
const fechas = D.fechas, idx = Object.fromEntries(fechas.map((f,i)=>[f,i]));
function media(serie,k){
  if(!suavizar) return serie;
  const m=Math.floor(k/2);
  return serie.map((_,i)=>{let s=0,n=0;for(let j=Math.max(0,i-m);j<=Math.min(serie.length-1,i+m);j++)
    if(serie[j]!==null){s+=serie[j];n++;} return n?s/n:null;});
}
function colorNivel(c,i){ return c.escala==='presencia' ? D.color_presencia : PALV[i%PALV.length]; }
function porCriterio(slug){
  const c=D.criterios.find(x=>x.slug===slug), niv=c.niveles;
  const val=Object.fromEntries(niv.map(n=>[n,fechas.map(()=>0)]));
  const conVal=fechas.map(()=>0), aus=fechas.map(()=>0), fal=fechas.map(()=>0), hay=fechas.map(()=>false);
  D.agg.filter(r=>r.slug===slug && pasa(r)).forEach(r=>{
    const i=idx[r.fecha_str]; if(i===undefined) return; hay[i]=true;
    if((r.clase==='valencia'||r.clase==='presencia') && val[r.nivel]){val[r.nivel][i]+=r.n;conVal[i]+=r.n;}
    else if(r.clase==='ausencia') aus[i]+=r.n; else fal[i]+=r.n;
  });
  // Denominador de la saliencia: valencia/presencia + ausencia. Los faltantes
  // no dicen si el tuit tocaba la dimensión: fuera de ambos lados.
  const base=fechas.map((_,i)=>conVal[i]+aus[i]);
  return {c,niv,val,conVal,aus,fal,base,hay};
}
const ejeX = () => Object.assign({type:'category',data:fechas,boundaryGap:true,
  axisLabel:{color:C.tinta2,fontSize:10,formatter:v=>v.slice(5)}},EJE_X);
const ejeYp = () => Object.assign({type:'value',max:modo==='prop'?1:null,
  axisLabel:{color:C.tinta2,fontSize:11,formatter:modo==='prop'?(v=>Math.round(v*100)+'%'):undefined}},EJE_Y);
const bandas = () => ({silent:true,data:D.cortes.map((p,i)=>([
  {xAxis:p.desde<fechas[0]?fechas[0]:p.desde,itemStyle:{color:i%2?'rgba(11,11,11,.04)':'rgba(11,11,11,.015)'},
   label:{show:true,formatter:p.nombre,color:C.mudo,fontSize:10,position:'insideTop'}},
  {xAxis:p.hasta&&p.hasta<=fechas[fechas.length-1]?p.hasta:fechas[fechas.length-1]}]))});
const fmt = v => v==null?'—':(modo==='prop'?(v*100).toFixed(1)+'%':Math.round(v));
const opc = () => ({tooltip:{trigger:'axis',valueFormatter:fmt},toolbox:TOOLBOX,
  legend:{top:0,left:0,textStyle:{color:C.tinta2,fontSize:11},itemWidth:12,itemHeight:8},
  grid:{left:52,right:16,top:34,bottom:34}});

const pVol = panel(RAIZ_SEC,{titulo:'Volumen diario del corpus',
  pregunta:'Tuits recogidos por día, apilados por idioma. Es el denominador de todo lo demás.',alto:280});
const pSal = panel(RAIZ_SEC,{titulo:'Saliencia dimensional',
  pregunta:'Proporción diaria de tuits que <em>tocan</em> cada criterio: nivel &gt; 0 sobre (nivel &gt; 0 + ausencia). Los bloqueados no cuentan en ningún lado.',alto:300});
const gVol = init(pVol.lienzo,{}), gSal = init(pSal.lienzo,{});
const gSerie = {};
D.criterios.forEach(c=>{
  const p = panel(RAIZ_SEC,{titulo:esc(c.etiqueta)+' <span style="font-weight:400;color:#6b6862">· '+esc(c.nombre)+'</span>',
    pregunta: c.escala==='valencia'
      ? 'Valencia diaria, apilada del más negativo (1) al más positivo (5). La ausencia no entra en el apilado.'
      : 'Criterio de <b>presencia</b>: tuits que mencionan violencia por día (en proporción, sobre los que tienen calificación). No es una escala de valencia.',
    chips:['ausencia '+c.conteo.ausencia.toLocaleString('es')+' (NO: '+c.conteo.ausencia_por_aplicable_no
      +' · nivel 0: '+c.conteo.ausencia_por_nivel_0+')','nivel &gt; 0: '+c.conteo.presentes,'bloqueados: '+c.conteo.faltante],
    alto:260});
  gSerie[c.slug]=init(p.lienzo,{});
});
const valCrits = D.criterios.filter(c=>c.escala==='valencia'), presCrits = D.criterios.filter(c=>c.escala==='presencia');
const pGlo = panel(RAIZ_SEC,{titulo:'Distribución de valencia · todo el corpus',
  pregunta:'Sólo criterios de valencia y sólo tuits con nivel &gt; 0. La ausencia y los bloqueados quedan fuera y se reportan en la tabla.',
  alto:60+valCrits.length*46,html:'<div class="t-global"></div>'});
const gGlo = init(pGlo.lienzo,{});
const pPer = panel(RAIZ_SEC,{titulo:'Distribución por período',
  pregunta:'Misma escala y mismo orden de niveles en los tres períodos. La presencia de violencia se muestra como tasa.',
  alto:0,html:'<div class="rejilla"></div>'});
const gPer = {};
D.criterios.forEach(c=>{
  const d=el('<div><h3 style="font-size:13px">'+esc(c.etiqueta)+'</h3><div class="lienzo" style="height:190px"></div></div>');
  pPer.seccion.querySelector('.rejilla').appendChild(d);
  gPer[c.slug]=init(d.querySelector('.lienzo'),{});
});

function pintar(){
  const nSel = D.idiomas.filter(x=>sel.has(x.codigo)).reduce((a,b)=>a+b.n,0);
  if(sel.size===0){aviso.hidden=false;aviso.textContent='Ningún idioma seleccionado: no hay datos que mostrar.';}
  else if(nSel<D.umbral_muestra){aviso.hidden=false;aviso.textContent='La selección deja '+nSel+' tuits, por debajo de '
    +D.umbral_muestra+'. A esa densidad la serie diaria describe unos pocos tuits, no una tendencia.';}
  else aviso.hidden=true;

  const vol={};
  D.volumen.filter(pasa).forEach(r=>{(vol[r.idioma]=vol[r.idioma]||fechas.map(()=>0))[idx[r.fecha_str]]+=r.n;});
  const ords = D.idiomas.map(x=>x.codigo).filter(c=>vol[c]);
  gVol.setOption(Object.assign(opc(),{tooltip:{trigger:'axis'},xAxis:ejeX(),
    yAxis:Object.assign({type:'value'},EJE_Y),
    series:ords.map((c,i)=>({name:c,type:'bar',stack:'v',data:vol[c],
      itemStyle:{color:PALI[i%PALI.length]}, markArea:i===0?bandas():undefined}))}),true);

  gSal.setOption(Object.assign(opc(),{
    tooltip:{trigger:'axis',valueFormatter:v=>v==null?'—':(v*100).toFixed(1)+'%'},xAxis:ejeX(),
    yAxis:Object.assign({type:'value',axisLabel:{color:C.tinta2,formatter:v=>Math.round(v*100)+'%'}},EJE_Y),
    series:D.criterios.map((c,i)=>{const {conVal,base,hay}=porCriterio(c.slug);
      const s=fechas.map((_,j)=>hay[j]&&base[j]?conVal[j]/base[j]:null);
      return {name:c.etiqueta,type:'line',connectNulls:false,symbolSize:5,data:media(s,D.ventana),
        itemStyle:{color:['#eb6834','#1baf7a','#eda100','#4a3aa7'][i%4]},markArea:i===0?bandas():undefined};})}),true);

  D.criterios.forEach(c=>{
    const {niv,val,hay,base}=porCriterio(c.slug);
    let series;
    if(c.escala==='valencia'){
      const den=fechas.map((_,j)=>niv.reduce((a,n)=>a+val[n][j],0));
      series=niv.map((n,i)=>({name:'nivel '+n,type:'bar',stack:'s',
        data:fechas.map((_,j)=>!hay[j]||(modo==='prop'&&!den[j])?null:(modo==='prop'?val[n][j]/den[j]:val[n][j])),
        itemStyle:{color:colorNivel(c,i)},markArea:i===0?bandas():undefined}));
    } else {
      series=[{name:'presente',type:'bar',
        data:fechas.map((_,j)=>!hay[j]||(modo==='prop'&&!base[j])?null:(modo==='prop'?val['1'][j]/base[j]:val['1'][j])),
        itemStyle:{color:D.color_presencia},markArea:bandas()}];
    }
    // La presencia de violencia ronda pocos puntos porcentuales: con el eje
    // fijo en 100% las barras serian invisibles.
    const ejeY = ejeYp(); if(c.escala==='presencia') ejeY.max = null;
    gSerie[c.slug].setOption(Object.assign(opc(),{xAxis:ejeX(),yAxis:ejeY,series}),true);
  });

  const sumaNiv=(c,filtro)=>{const {niv,val}=porCriterio(c.slug);
    const suma=niv.map(n=>fechas.reduce((a,f,j)=>a+(filtro(f)?val[n][j]:0),0));
    return {suma,t:suma.reduce((a,b)=>a+b,0)};};
  const glob=valCrits.map(c=>Object.assign({c},sumaNiv(c,()=>true)));
  gGlo.setOption(Object.assign(opc(),{grid:{left:130,right:16,top:30,bottom:24},
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'},valueFormatter:fmt},
    xAxis:Object.assign({type:'value',max:modo==='prop'?1:null,
      axisLabel:{color:C.tinta2,formatter:modo==='prop'?(v=>Math.round(v*100)+'%'):undefined}},EJE_Y),
    yAxis:Object.assign({type:'category',data:glob.map(g=>g.c.etiqueta+' ('+g.t+')').reverse()},EJE_X),
    series:['1','2','3','4','5'].map((n,i)=>({name:'nivel '+n,type:'bar',stack:'g',
      data:glob.map(g=>modo==='prop'?(g.t?g.suma[i]/g.t:0):g.suma[i]).reverse(),itemStyle:{color:PALV[i]}}))}),true);

  // tabla de respaldo: la identidad nunca queda sólo en el color.
  // Los criterios de PRESENCIA llevan columna propia. Un «1» de presencia es
  // «hay violencia»; un «1» de valencia es «odio o desprecio». Con los dos en
  // la columna «nivel 1» la columna deja de poder sumarse y quien la lee de
  // arriba abajo suma cosas distintas. La forma de la fila se deriva de la
  // ESCALA declarada, nunca del nombre del criterio.
  const filasT = D.criterios.map(c=>{const {niv,val,aus,fal}=porCriterio(c.slug);
    const tot=n=>fechas.reduce((a,_,j)=>a+val[n][j],0);
    const esPresencia = c.escala==='presencia';
    const niveles = esPresencia ? ['—','—','—','—','—']
      : ['1','2','3','4','5'].map(n=>niv.includes(n)?tot(n):'—');
    const presente = esPresencia ? niv.reduce((a,n)=>a+tot(n),0) : '—';
    return [c.etiqueta,c.escala].concat(niveles).concat(
      [presente,aus.reduce((a,b)=>a+b,0),fal.reduce((a,b)=>a+b,0)]);});
  pGlo.seccion.querySelector('.t-global').innerHTML = tabla(
    ['criterio','escala','nivel 1','nivel 2','nivel 3','nivel 4','nivel 5','presente','ausencia','bloqueados'],filasT);

  D.criterios.forEach(c=>{
    const filas=D.cortes.map(p=>{const dentro=f=>f>=p.desde&&(!p.hasta||f<p.hasta);
      const {niv,val,aus}=porCriterio(c.slug);
      const suma=niv.map(n=>fechas.reduce((a,f,j)=>a+(dentro(f)?val[n][j]:0),0));
      const a=fechas.reduce((x,f,j)=>x+(dentro(f)?aus[j]:0),0);
      return {nombre:p.nombre,suma,t:suma.reduce((x,y)=>x+y,0),aus:a};});
    let series, xmax;
    if(c.escala==='valencia'){
      xmax=modo==='prop'?1:null;
      series=c.niveles.map((n,i)=>({name:'nivel '+n,type:'bar',stack:'p',
        data:filas.map(f=>modo==='prop'?(f.t?f.suma[i]/f.t:0):f.suma[i]).reverse(),itemStyle:{color:PALV[i]}}));
    } else {
      xmax=null;
      series=[{name:'presente',type:'bar',itemStyle:{color:D.color_presencia},
        data:filas.map(f=>modo==='prop'?((f.t+f.aus)?f.t/(f.t+f.aus):0):f.t).reverse()}];
    }
    gPer[c.slug].setOption({tooltip:{trigger:'axis',axisPointer:{type:'shadow'},valueFormatter:fmt},toolbox:TOOLBOX,
      grid:{left:118,right:16,top:10,bottom:24},
      xAxis:Object.assign({type:'value',max:xmax,axisLabel:{color:C.tinta2,fontSize:10,
        formatter:modo==='prop'?(v=>(v*100).toFixed(c.escala==='presencia'?1:0)+'%'):undefined}},EJE_Y),
      yAxis:Object.assign({type:'category',data:filas.map(f=>f.nombre.split(' ')[0]+' ('+f.t+')').reverse()},EJE_X),
      series},true);
  });
}
pintar();
"""

DATOS_RUBRICA = None
if HAY_RUBRICA:
    DATOS_RUBRICA = agregar_rubrica()

    # --- contraste contra el HTML original ---------------------------------
    CONTRASTE_RUBRICA = None
    if RUTA_HTML_RUBRICA and os.path.exists(RUTA_HTML_RUBRICA):
        _suyos = datos_html_original(RUTA_HTML_RUBRICA)
        if _suyos is None:
            print("El HTML original no trae el bloque de datos esperado: contraste omitido.")
        else:
            _comp, _diffs, (_vm, _vs), (_pm, _ps) = contrastar_con_original(DATOS_RUBRICA, _suyos)
            CONTRASTE_RUBRICA = {"diferencias": len(_diffs), "volumen": [_vm, _vs]}
            print("CONTRASTE CONTRA {}".format(RUTA_HTML_RUBRICA))
            if _diffs.empty and _vm == _vs and _pm == _ps:
                print("   coinciden los {} conteos por criterio x clase x nivel, el volumen "
                      "({} tuits) y los tuits por periodo.".format(len(_comp), _vm))
            else:
                print("   ERROR DE REPLICA: la reconstruccion no coincide con el original.")
                if not _diffs.empty:
                    print(_diffs.to_string())
                if _vm != _vs:
                    print("   volumen: recalculado {} · original {}".format(_vm, _vs))
                if _pm != _ps:
                    print("   por periodo: recalculado {} · original {}".format(_pm, _ps))
    else:
        print("HTML original no disponible en {}: contraste NO realizado.".format(RUTA_HTML_RUBRICA))

    registrar_seccion("rubrica", "Resultados de la rúbrica", DATOS_RUBRICA, JS_RUBRICA)
    PAGINA_RUBRICA = mostrar_seccion("rubrica")

    if MOSTRAR_HTML_ORIGINAL and RUTA_HTML_RUBRICA and os.path.exists(RUTA_HTML_RUBRICA):
        print()
        print("HTML ORIGINAL DE EVALUADORTWEETS (tal cual se genero):")
        with open(RUTA_HTML_RUBRICA, encoding="utf-8") as _fh:
            display(HTML(_fh.read()))

## 3. Definición de las dimensiones

Aquí vive todo el diseño. Cada dimensión son **tres hipótesis**: positiva, negativa y una de
irrelevancia. El modelo reparte probabilidad entre las tres y de ahí salen dos números:

```
score      = P(positiva) − P(negativa)        va de −1 a 1
relevancia = 1 − P(irrelevante)               va de 0 a 1
```

La tercera hipótesis es la pieza clave. Sin ella, un tuit sobre fútbol preguntado por
gobierno reparte 50/50 y devuelve 0 — el mismo 0 ambiguo del método actual. Con ella,
`relevancia ≈ 0` te dice que el turista simplemente no habló del tema.

### Las dimensiones

| Dimensión | Qué mide |
|---|---|
| `atmosfera` | el ánimo que se vive: alegre y emocionante contra desagradable y estresante |
| `imagen_cultural` | comida, lugares y cultura: hermosos e impresionantes contra decepcionantes |
| `perspectiva_politica` | el gobierno y sus autoridades: competentes y legales contra corruptos |
| `violence_salience` · `safety_perceived` | violencia y seguridad, con las cuatro etiquetas y las dos fórmulas de `analyzer.py` |

### Dónde va el ancla «México»

La tesis mide la imagen de México, así que el instrumento tiene que estar anclado ahí. Pero
el ancla va en la **plantilla**, no dentro de cada hipótesis, y la diferencia no es cosmética.

`multi_label=False` reparte la probabilidad con `softmax` sobre los logits de las tres
etiquetas, y el softmax es invariante a un desplazamiento **constante**. Metida dentro de las
etiquetas, el ancla no es constante: encarece las dos afirmaciones (`pos`, `neg`) y **abarata
la negación** (`na`), porque es trivialmente cierto que un texto no menciona el ánimo *en
México* si no menciona México. El resultado es masa desplazada hacia `na`, relevancia hundida
y una alerta de «dimensión muda» causada por la redacción y no por el corpus.

En la plantilla el ancla es un prefijo idéntico para las tres etiquetas y se cancela.

**No es una suposición:** `ANCLA_EN_ETIQUETAS` genera la variante alterna a partir de las
mismas frases, y la sección 9 corre la batería con las dos y compara. Si el efecto no
aparece, la decisión se revisa con el número delante.

**Estas frases son el instrumento de medición.** Cambiarlas cambia los resultados, así que
en la tesis van citadas textualmente, igual que se cita un libro de códigos.

In [ ]:
# ===========================================================================
# EL INSTRUMENTO DE MEDICION. Todo el diseno vive aqui.
#
# Idioma de las hipotesis: XNLI es translinguistico, la hipotesis puede ir en
# un idioma y el tuit en otro. Lo importante es usar UNA SOLA para todos los
# textos, para que el instrumento sea identico entre idiomas.
# ===========================================================================
IDIOMA_HIPOTESIS = "es"

# ---------------------------------------------------------------------------
# DONDE VA EL ANCLA GEOGRAFICA. Es una decision de diseno, no un detalle de
# redaccion, y conviene entender por que.
#
# multi_label=False calcula softmax(logit_pos, logit_neg, logit_na) sobre los
# logits de entailment. El softmax es invariante a un desplazamiento CONSTANTE
# entre las tres etiquetas. El ancla, metida DENTRO de cada etiqueta, no lo es:
#
#   pos: "el ambiente EN MEXICO fue alegre"        afirmacion + predicado
#   neg: "el ambiente EN MEXICO fue desagradable"    -> mas dificil de entrañar
#   na : "NO menciona el ambiente ... EN MEXICO"   negacion + predicado
#                                                    -> MAS FACIL de entrañar
#
# Es trivialmente cierto que un texto no menciona el animo *en Mexico* si no
# menciona Mexico. El ancla empuja masa hacia `na`, hunde
# relevancia = 1 - P(na) bajo UMBRAL_MUDEZ y dispara la alerta de "dimension
# muda" por un artefacto de REDACCION, no por el corpus.
#
# En la PLANTILLA el ancla es un prefijo identico para las tres etiquetas y
# se cancela. Por eso ahi es donde va.
#
# ANCLA_EN_ETIQUETAS existe solo para MEDIR ese efecto en vez de suponerlo:
# la seccion de verificacion del ancla corre la bateria con las dos variantes
# y compara las distribuciones de relevancia. No lo dejes en True para una
# corrida de produccion.
# ---------------------------------------------------------------------------
ANCLA_EN_ETIQUETAS = False

_PLANTILLAS = {
    # ancla en la plantilla: constante entre etiquetas, se cancela
    False: {"en": "This text about Mexico says that {}.",
            "es": "Este texto sobre Mexico dice que {}."},
    # ancla dentro de las etiquetas: la plantilla queda neutra
    True:  {"en": "This text says that {}.",
            "es": "Este texto dice que {}."},
}
PLANTILLA = _PLANTILLAS[ANCLA_EN_ETIQUETAS][IDIOMA_HIPOTESIS]

# ---------------------------------------------------------------------------
# LAS DIMENSIONES. Cada una son tres hipotesis que compiten: positiva,
# negativa y de irrelevancia.
#
#   score      = P(pos) - P(neg)     va de -1 a 1
#   relevancia = 1 - P(na)           va de 0 a 1
#
# Las frases van SIN "en Mexico": el ancla la pone la plantilla. La variante
# anclada se deriva abajo, mecanicamente, para la comparacion.
#
# ESTAS FRASES SON EL INSTRUMENTO. Cambiarlas cambia los resultados, asi que
# en la tesis van citadas textualmente, igual que se cita un libro de codigos.
# ---------------------------------------------------------------------------
DIMENSIONES = {
    # `pos` y `neg` son estructuralmente identicas: mismo sujeto, misma
    # construccion, y adjetivos que son antonimos de intensidad equivalente.
    # Lo unico que las separa es la polaridad, que es lo que el softmax de dos
    # etiquetas tiene que contrastar.
    #
    # imagen_cultural NO usa ya un sujeto conjuntivo ("la comida, los lugares
    # Y la cultura"): entrañar una conjuncion exige los tres conjuntos, asi
    # que un tuit que solo habla de comida dejaba `pos` y `neg` a medias por
    # igual y la resta colapsaba a cero. Se sustituye por hiperonimo con
    # ejemplificacion ("la cultura del lugar, como su comida y sus sitios"),
    # que un solo ejemplar ya puede entrañar, conservando los tres sustantivos
    # como anclas lexicas.
    #
    # Desaparecen los atenuantes: "poco atractivos" era un downtoner sobre un
    # adjetivo positivo, y los modelos NLI son insensibles a los mitigadores,
    # asi que anclaba lexicamente hacia el campo positivo.
    #
    # `na` solo se usa con DISENO_RELEVANCIA = "trio". Se reescribe en forma
    # AFIRMATIVA ("se habla de otro tema, ajeno a ...") porque los marcadores
    # de negacion son rasgos hypothesis-only que empujan hacia contradiccion
    # en MNLI/XNLI, y porque "Este texto dice que no menciona X" es una
    # composicion autorreferente que el modelo nunca vio en entrenamiento.
    "atmosfera": {
        "en": {"pos": "the mood people experienced was joyful and exciting",
               "neg": "the mood people experienced was unpleasant and stressful",
               "na":  "it deals with a different topic, apart from the mood and the emotions"},
        "es": {"pos": "el ambiente que se vivio fue alegre y emocionante",
               "neg": "el ambiente que se vivio fue desagradable y estresante",
               "na":  "se habla de otro tema, ajeno al ambiente y a las emociones"},
    },
    "imagen_cultural": {
        "en": {"pos": "the local culture, such as its food and its places, was beautiful and impressive",
               "neg": "the local culture, such as its food and its places, was ugly and disappointing",
               "na":  "it deals with a different topic, apart from the culture, the food or the places"},
        "es": {"pos": "la cultura del lugar, como su comida y sus sitios, resulto hermosa e impresionante",
               "neg": "la cultura del lugar, como su comida y sus sitios, resulto fea y decepcionante",
               "na":  "se habla de otro tema, ajeno a la cultura, la comida o los lugares"},
    },
    "perspectiva_politica": {
        "en": {"pos": "the authorities and the government acted in a competent and honest way",
               "neg": "the authorities and the government acted in an incompetent and corrupt way",
               "na":  "it deals with a different topic, apart from the government and its authorities"},
        "es": {"pos": "las autoridades y el gobierno actuaron de forma competente y honesta",
               "neg": "las autoridades y el gobierno actuaron de forma incompetente y corrupta",
               "na":  "se habla de otro tema, ajeno al gobierno y sus autoridades"},
    },
}

# ---------------------------------------------------------------------------
# LA COMPUERTA DE RELEVANCIA. Es la pieza nueva del rediseño.
#
# POR QUE NO BASTA REESCRIBIR `na`. Con multi_label=False el pipeline hace
# softmax(logit_pos, logit_neg, logit_na) sobre los logits de ENTAILMENT. Si
# un texto es genuinamente ajeno al tema, los tres entailments son debiles
# POR IGUAL -- y el softmax renormaliza de todos modos: tres logits igual de
# malos dan P ~ 1/3 cada uno, o sea relevancia = 1 - 1/3 ~ 0.67 para un texto
# perfectamente mudo. Para que la relevancia baje de 0.30 no alcanza con que
# `na` sea verdadera: tiene que DOMINAR a las otras dos, y una hipotesis
# negada no puede hacerlo. Eso es un piso estructural del diseño de tres en
# competencia, no un problema de redaccion.
#
# La compuerta lo saca de la competencia: con multi_label=True cada etiqueta
# se puntua entailment-vs-contradiccion de forma INDEPENDIENTE. No hay
# softmax conjunto, no hay piso, y no queda ni una sola hipotesis negada en
# todo el instrumento.
#
# La lista disyuntiva ("o") si funciona aqui: la etiqueta actua como bolsa de
# anclas tematicas, no como predicacion conjunta.
#
# El ancla «México» sigue viviendo en la plantilla, por la misma razon de
# siempre.
# ---------------------------------------------------------------------------
_PLANTILLAS_GATE = {
    False: {"en": "This text about Mexico talks about {}.",
            "es": "Este texto sobre Mexico habla de {}."},
    True:  {"en": "This text talks about {}.",
            "es": "Este texto habla de {}."},
}
PLANTILLA_GATE = _PLANTILLAS_GATE[ANCLA_EN_ETIQUETAS][IDIOMA_HIPOTESIS]

# Las etiquetas en español empiezan con "la/las" a proposito: la plantilla
# dice "habla de {}" y no contrae, asi que una etiqueta que empezara con "el"
# produciria "habla de el ambiente". Un error de concordancia en el texto que
# el modelo lee de verdad, y gratuito.
GATE = {
    "en": {"atmosfera":            "the emotions and the mood people experience",
           "imagen_cultural":      "the food, the places or the culture",
           "perspectiva_politica": "the authorities and the government"},
    "es": {"atmosfera":            "las emociones y el ambiente que se viven",
           "imagen_cultural":      "la comida, los lugares o la cultura",
           "perspectiva_politica": "las autoridades y el gobierno"},
}

# ---------------------------------------------------------------------------
# Grupo de violencia/seguridad: se conservan las CUATRO etiquetas y las dos
# formulas del codigo actual (analyzer.py L67), de modo que violence_salience
# y safety_perceived se sigan calculando igual. Respecto del notebook previo
# lo unico que cambia es donde vive el ancla; el texto es el mismo.
# ---------------------------------------------------------------------------
VIOLENCIA = {
    "en": {"violence":   "there was violence or crime",
           "insecurity": "the place felt unsafe or insecure",
           "safety":     "the place felt safe and calm",
           "neutral":    "it does not talk about safety or violence"},
    "es": {"violence":   "hubo violencia o crimen",
           "insecurity": "el lugar se sentia inseguro",
           "safety":     "el lugar se sentia seguro y tranquilo",
           "neutral":    "no habla de seguridad ni violencia"},
}

# ---------------------------------------------------------------------------
# Variante con el ancla DENTRO de las etiquetas. Se DERIVA de las frases de
# arriba en vez de escribirse a mano, para que las dos variantes no puedan
# divergir en el contenido: la unica diferencia entre ellas es el ancla.
# ---------------------------------------------------------------------------
_ANCLA = {"en": " in Mexico", "es": " en Mexico"}


def _con_ancla(etiquetas, idi):
    return {k: v + _ANCLA[idi] for k, v in etiquetas.items()}


if ANCLA_EN_ETIQUETAS:
    DIMENSIONES = {d: {idi: _con_ancla(t[idi], idi) for idi in t}
                   for d, t in DIMENSIONES.items()}
    VIOLENCIA = {idi: _con_ancla(VIOLENCIA[idi], idi) for idi in VIOLENCIA}
    GATE = {idi: _con_ancla(GATE[idi], idi) for idi in GATE}

# --- resumen ---------------------------------------------------------------
_ETIQ = {d: len(t[IDIOMA_HIPOTESIS]) for d, t in DIMENSIONES.items()}
_ETIQ["violencia"] = len(VIOLENCIA[IDIOMA_HIPOTESIS])

print("idioma de las hipotesis :", IDIOMA_HIPOTESIS)
print("ancla geografica        :", "DENTRO de las etiquetas (variante de prueba)"
      if ANCLA_EN_ETIQUETAS else "en la plantilla (se cancela en el softmax)")
print("plantilla               :", PLANTILLA)
print()
print("dimensiones             :", len(DIMENSIONES))
for _d, _t in DIMENSIONES.items():
    print("   {:<22} {} hipotesis".format(_d, len(_t[IDIOMA_HIPOTESIS])))
    for _k, _v in _t[IDIOMA_HIPOTESIS].items():
        print("      {:<4} {}".format(_k, PLANTILLA.format(_v)))
print("   {:<22} {} etiquetas".format("violencia/seguridad", _ETIQ["violencia"]))
for _k, _v in VIOLENCIA[IDIOMA_HIPOTESIS].items():
    print("      {:<11} {}".format(_k, PLANTILLA.format(_v)))
print()
print()
print("compuerta de relevancia (multi_label=True, sin softmax conjunto)")
print("   plantilla            :", PLANTILLA_GATE)
for _d, _v in GATE[IDIOMA_HIPOTESIS].items():
    print("      {:<22} {}".format(_d, PLANTILLA_GATE.format(_v)))
print()
print("Con DISENO_RELEVANCIA='compuerta' la relevancia sale de aqui y de cada")
print("dimension solo se usan pos/neg. Con 'trio' se usa `na` y la compuerta")
print("no se llama. El conteo real de pasadas lo imprime la seccion 8.")

## 4. Carga de modelos

Dos modelos, ~3.3 GB en la máquina de Google. **Con `CARGAR_ZEROSHOT` no se carga ninguno**: la corrida se restaura del volcado en la sección 8. **No se usa BART**: el rediseño lo reemplaza
por XLM-R-XNLI, que es multilingüe de verdad. (BART-MNLI está entrenado solo en inglés, así
que el código actual ya manda los tuits en español a un modelo fuera de su distribución.)

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else -1

if CARGAR_ZEROSHOT:
    # Con una corrida previa no hace falta ningun modelo de esta seccion: la
    # evaluacion (seccion 8) restaura `resultados` del volcado. Se dejan los
    # nombres definidos para que un uso accidental falle con un mensaje claro.
    sentiment_pipe = None
    nli_pipe = None
    print("CARGAR_ZEROSHOT = {}".format(CARGAR_ZEROSHOT))
    print("No se carga ningun modelo: la corrida se restaura del volcado.")
else:
    from transformers import pipeline

    print("1/2 modelo de sentimiento (~1.1 GB)...")
    sentiment_pipe = pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
        tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
        truncation=True, max_length=128, device=DEVICE,
    )

    print("2/2 modelo NLI multilingue (~2.2 GB, tarda)...")
    nli_pipe = pipeline(
        "zero-shot-classification",
        model="joeddav/xlm-roberta-large-xnli",
        truncation=True, device=DEVICE,
    )
    print("listo")

## 5. Método actual (para comparar)

Copiado literal de `analyzer.py`. Solo necesita el modelo de sentimiento: los tres
indicadores léxicos son conteo de palabras puro.

In [ ]:
import math, re

def clamp(x, lo, hi):
    return min(max(x, lo), hi)

# --- analyzer.py L15-24 ----------------------------------------------------
# Copiadas literalmente, incluidas las de hospitalidad: el rediseno ya no mide
# esa dimension, pero la lista se conserva para que esta celda siga siendo una
# reproduccion fiel del codigo actual y se vea que no se toco nada.
HOSPITALITY_POS = {"friendly", "warm", "kind", "welcoming", "helpful", "amable", "hospitalario", "gentil"}
HOSPITALITY_NEG = {"rude", "hostile", "unfriendly", "aggressive", "grosero"}
ATMOSPHERE_POS = {"amazing", "incredible", "beautiful", "electric", "vibrant", "festive", "lively", "increíble", "hermoso", "ambiente"}
ATMOSPHERE_NEG = {"depressing", "unsafe", "scary", "tense", "chaotic", "deprimiente"}
GOVERNANCE_POS = {"police", "security", "safe", "protected", "policia", "seguridad", "officers", "authorities", "controlled"}
GOVERNANCE_NEG = {"corrupt", "impunity", "cartel", "crime", "criminal", "corrupción", "impunidad", "narco"}

# Correspondencia entre las dimensiones que mide el rediseno y las listas de
# palabras del metodo actual.
#
#   atmosfera            <- ATMOSPHERE_*    (analyzer.py las llama atmosphere)
#   perspectiva_politica <- GOVERNANCE_*    (analyzer.py: governance_trust)
#   imagen_cultural      <- NO EXISTE. Por eso ESCALAS la declara con un solo
#                           metodo y las figuras comparativas la excluyen.
#
# Las listas NO se tocan: renombrar la dimension no cambia que palabras busca
# el metodo actual, y el punto de la comparacion es medirlo tal como es.
SETS_LEXICO = {
    "atmosfera":            (ATMOSPHERE_POS, ATMOSPHERE_NEG),
    "perspectiva_politica": (GOVERNANCE_POS, GOVERNANCE_NEG),
}

def sentimiento(text):
    r = sentiment_pipe(text[:512])[0]
    lab = r["label"].lower()
    if lab == "positive":
        return r["score"], r["label"]
    if lab == "negative":
        return -r["score"], r["label"]
    return 0.0, r["label"]

def _mod(tl, pos_set, neg_set):
    pos = sorted(w for w in pos_set if w in tl)
    neg = sorted(w for w in neg_set if w in tl)
    return clamp((len(pos) - len(neg)) * 0.2, -1, 1), pos, neg

def metodo_actual(text):
    """analyzer.py L212-227 -- las dimensiones lexicas + sentimiento.

    Las FORMULAS son las del repositorio, intactas. Lo unico que cambia
    respecto del notebook anterior son las claves con que se devuelven, para
    que coincidan con las dimensiones que mide el rediseno.
    """
    tl = text.lower()
    s, lab = sentimiento(text)
    ma, ap, an = _mod(tl, ATMOSPHERE_POS,  ATMOSPHERE_NEG)
    mg, gp, gn = _mod(tl, GOVERNANCE_POS,  GOVERNANCE_NEG)
    return {
        "overall_sentiment":    s,
        "atmosfera":            clamp(s * 0.7 + ma, -1, 1),
        "perspectiva_politica": clamp(mg * 0.8, -1, 1),
        "_label": lab,
        "_palabras": {"atmosfera": ap + an, "perspectiva_politica": gp + gn},
    }

print("metodo actual listo --", len(SETS_LEXICO), "dimensiones lexicas:",
      ", ".join(SETS_LEXICO))

## 6. Rediseño zero-shot

Una llamada al modelo NLI por dimensión. `multi_label=False` hace que las tres hipótesis
compitan y sumen 1, que es lo que permite restarlas.

In [ ]:
IDI = IDIOMA_HIPOTESIS

# ===========================================================================
# EVALUACION POR LOTES
#
# Con 30 textos da igual. Con el corpus son 34 203 pasadas de un modelo large,
# y hacerlas de a una es la diferencia entre minutos y horas.
#
# BATCH_SIZE vive en la celda de configuracion, que corre DESPUES de esta. Se
# lee en el momento de la llamada, no al definir la funcion, y con un valor de
# respaldo por si alguien ejecuta esta celda suelta.
# ===========================================================================

def _batch():
    return globals().get("BATCH_SIZE", 8)


def _nli_lote(textos, etiquetas):
    """Una llamada por grupo de hipotesis para TODOS los textos del lote.

    Devuelve una lista de {etiqueta_legible: probabilidad}, una por texto, en
    el mismo orden de entrada. La forma por texto es identica a la que
    devolvia la version de a uno, asi que nada de lo que cuelga de aqui cambia.
    """
    frases = list(etiquetas.values())
    claves = list(etiquetas.keys())
    entrada = [t[:512] for t in textos]
    rs = nli_pipe(entrada, candidate_labels=frases,
                  hypothesis_template=PLANTILLA, multi_label=False,
                  batch_size=_batch())
    if isinstance(rs, dict):          # el pipeline devuelve un dict si le dan uno solo
        rs = [rs]
    salida = []
    for r in rs:
        por_frase = dict(zip(r["labels"], r["scores"]))
        salida.append({k: por_frase[etiquetas[k]] for k in claves})
    return salida


def _nli(text, etiquetas):
    """Compatibilidad: un solo texto."""
    return _nli_lote([text], etiquetas)[0]


def _diseno():
    """De donde sale la relevancia. Se lee al llamar, no al definir, porque
    la celda de configuracion corre DESPUES de esta."""
    return globals().get("DISENO_RELEVANCIA", "compuerta")


def _nli_lote_multi(textos, etiquetas):
    """La compuerta de relevancia: multi_label=True.

    Cada etiqueta se puntua entailment-vs-contradiccion de forma
    INDEPENDIENTE. No hay softmax conjunto entre etiquetas, asi que no hay
    piso de 1/3 y las tres dimensiones no se roban masa entre si: un texto
    puede hablar de las tres, o de ninguna, y la compuerta puede decirlo.

    Usa PLANTILLA_GATE, que es afirmativa ('habla de'), no la plantilla de
    discurso reportado ('dice que') de las hipotesis de valencia.
    """
    frases = list(etiquetas.values())
    claves = list(etiquetas.keys())
    entrada = [t[:512] for t in textos]
    rs = nli_pipe(entrada, candidate_labels=frases,
                  hypothesis_template=PLANTILLA_GATE, multi_label=True,
                  batch_size=_batch())
    if isinstance(rs, dict):
        rs = [rs]
    salida = []
    for r in rs:
        por_frase = dict(zip(r["labels"], r["scores"]))
        salida.append({k: por_frase[etiquetas[k]] for k in claves})
    return salida


def _sentimiento_lote(textos):
    rs = sentiment_pipe([t[:512] for t in textos], batch_size=_batch())
    if isinstance(rs, dict):
        rs = [rs]
    salida = []
    for r in rs:
        lab = r["label"].lower()
        if lab == "positive":
            salida.append((r["score"], r["label"]))
        elif lab == "negative":
            salida.append((-r["score"], r["label"]))
        else:
            salida.append((0.0, r["label"]))
    return salida


def metodo_nuevo_lote(textos, retweets=None, likes=None):
    """El rediseno sobre una lista de textos. Formulas identicas a las de la
    version de a uno; lo unico que cambia es que las llamadas van agrupadas."""
    n = len(textos)
    retweets = retweets if retweets is not None else [0] * n
    likes = likes if likes is not None else [0] * n
    if n == 0:
        return []

    sents = _sentimiento_lote(textos)
    salidas = []
    for i in range(n):
        s, lab = sents[i]
        salidas.append({"overall_sentiment": s, "_label": lab,
                        "engagement_weight": math.log1p(retweets[i] + likes[i]),
                        "_crudo": {}})

    # violencia / seguridad -- 4 etiquetas, formulas del repo
    for i, v in enumerate(_nli_lote(textos, VIOLENCIA[IDI])):
        o = salidas[i]
        o["_crudo"]["violencia"] = v
        o["violence_salience"] = clamp(v["violence"] + 0.5 * v["insecurity"], 0, 1)
        o["safety_perceived"] = clamp(v["safety"] - v["violence"], -1, 1)

    # --- relevancia ------------------------------------------------------
    # "compuerta": una pasada multi_label=True por dimension, independiente.
    # "trio":      el diseño anterior, relevancia = 1 - P(na) dentro del
    #              softmax de tres. Se conserva para poder MEDIR la diferencia
    #              sobre la bateria en vez de suponerla.
    _dis = _diseno()
    puertas = None
    if _dis == "compuerta":
        print("   NLI: compuerta", flush=True)
        puertas = _nli_lote_multi(textos, GATE[IDI])
        for i, g in enumerate(puertas):
            salidas[i]["_crudo"]["compuerta"] = g

    # --- valencia --------------------------------------------------------
    for dim, etiquetas in DIMENSIONES.items():
        print("   NLI:", dim, flush=True)
        et = etiquetas[IDI]
        if _dis == "compuerta":
            # sin `na`: softmax de DOS etiquetas, contraste puro de polaridad
            et = {k: v for k, v in et.items() if k in ("pos", "neg")}
        for i, p in enumerate(_nli_lote(textos, et)):
            o = salidas[i]
            o["_crudo"][dim] = p
            o[dim] = clamp(p["pos"] - p["neg"], -1, 1)
            if _dis == "compuerta":
                o[dim + "_relevancia"] = clamp(puertas[i][dim], 0, 1)
            else:
                o[dim + "_relevancia"] = clamp(1 - p["na"], 0, 1)

    return salidas


def metodo_nuevo(text, retweets=0, likes=0):
    """Compatibilidad: un solo texto."""
    return metodo_nuevo_lote([text], [retweets], [likes])[0]


print("rediseno listo -- pasadas de NLI por tuit segun DISENO_RELEVANCIA:")
print("   compuerta: {} = compuerta {} + pos/neg {} + violencia {}".format(
    len(GATE[IDI]) + 2 * len(DIMENSIONES) + len(VIOLENCIA[IDI]),
    len(GATE[IDI]), 2 * len(DIMENSIONES), len(VIOLENCIA[IDI])))
print("   trio     : {} = pos/neg/na {} + violencia {}".format(
    sum(len(t[IDI]) for t in DIMENSIONES.values()) + len(VIOLENCIA[IDI]),
    sum(len(t[IDI]) for t in DIMENSIONES.values()), len(VIOLENCIA[IDI])))
print("   (en lotes de BATCH_SIZE, que se lee al llamar)")

## 7. Configuración de las figuras

Los umbrales con los que se leen las figuras, todos en un solo lugar.

El importante es `UMBRAL_MUDEZ`. Marca dónde dejamos de leer un `score` como juicio y
empezamos a leerlo como «no habló del tema». **Es arbitrario**, y por eso la figura de
ausencia dibuja la distribución completa de `relevancia` con el corte superpuesto, en vez
de esconderlo: así se ve qué tan sensible es la lectura al valor elegido. En la tesis este
corte se justifica o se somete a análisis de sensibilidad; no se hereda de aquí.

También se declara aquí la **escala de cada dimensión** —valencia de −1 a 1, o prominencia
de 0 a 1— en vez de inferirla de los valores observados. Con 24 textos, concluir «esta
dimensión va de 0 a 1 porque no vi negativos» es exactamente el error que se quiere evitar.

In [ ]:
# ===========================================================================
# UMBRALES DE LECTURA. Ninguno es sagrado; todos se cambian aqui y solo aqui.
# ===========================================================================

# UMBRAL_MUDEZ es ARBITRARIO, y conviene decirlo en voz alta porque de el
# depende como se lee cada figura. Marca donde dejamos de leer un score como
# juicio y empezamos a leerlo como "no hablo del tema".
#
# En la tesis este corte hay que JUSTIFICARLO o someterlo a un analisis de
# sensibilidad; no se hereda de este notebook. Por eso la figura de ausencia
# dibuja la distribucion COMPLETA de relevancia con el corte superpuesto:
# para que se vea que tan sensible es la lectura al valor elegido.
UMBRAL_MUDEZ = 0.30

# ---------------------------------------------------------------------------
# DE DONDE SALE LA RELEVANCIA. Es la decision de diseño del rediseño de
# hipotesis, y se deja como interruptor para poder MEDIR la diferencia sobre
# la bateria en vez de suponerla.
#
#   "compuerta"  DOS ETAPAS. La relevancia sale de una pasada aparte con
#                multi_label=True (sin softmax conjunto, sin hipotesis
#                negada) y la valencia de un softmax de SOLO pos/neg.
#                Desaparece la desigualdad |valencia| <= relevancia, que era
#                un artefacto del acoplamiento y no una propiedad del
#                constructo. Los dos ejes quedan de verdad ortogonales.
#
#   "trio"       el diseño anterior: pos/neg/na en un softmax de tres y
#                relevancia = 1 - P(na). Se conserva para la comparacion.
#
# CRITERIO DE FALSACION, sobre la bateria y NUNCA sobre el corpus: si con
# "trio" y las hipotesis nuevas el control mudo sigue por encima de 0.50 de
# relevancia, el piso estructural del softmax queda confirmado en este mismo
# setup y se adopta "compuerta" con el numero delante.
DISENO_RELEVANCIA = "compuerta"

# Por debajo de esta relevancia la valencia NO se interpreta. Con "compuerta"
# los dos ejes son independientes, asi que este corte es lo unico que decide
# que observaciones tienen juicio legible. Se calibra contra la bateria y se
# congela ANTES de mirar un solo promedio del corpus.
UMBRAL_RELEVANCIA = 0.50

# Proporcion de observaciones mudas por encima de la cual una dimension se
# declara muda EN ESTE CORPUS y se dispara una alerta escrita. No dice que la
# dimension sea mala: dice que aqui casi nadie hablo de eso, y que por tanto
# su promedio esta dominado por textos que no opinaron.
UMBRAL_DIMENSION_MUDA = 0.70

# Por debajo de esto una figura se dibuja igual, pero con advertencia encima.
# Con la bateria de ejemplo (30 textos) varias figuras caen aqui, y es correcto
# que lo digan.
N_MINIMO_FIGURA = 20

# Minimo de observaciones para que un idioma se dibuje como comparable en las
# figuras que comparan idiomas. Por debajo se informa su n y se excluye; con
# el corpus real hay idiomas con una sola observacion.
N_MINIMO_IDIOMA = 3

# TODO lo que el notebook escribe va aqui, y nada de lo que lee. No es orden
# por gusto: con las salidas en el directorio de entrada, la autodeteccion del
# CSV encuentra dos archivos en la segunda corrida y el notebook se bloquea a
# si mismo.
SALIDA_DIR = "salida"

SALIDA_HTML = "figuras_dimensiones.html"

# CSV de resultados. El ancho trae UNA FILA POR TEXTO con las columnas
# originales del archivo mas una columna por dimension y metodo: es el que
# abres en Excel o le pasas a alguien. El largo trae una fila por texto,
# metodo y dimension: es el que se analiza.
SALIDA_CSV = "resultados_enriquecidos.csv"
SALIDA_CSV_LARGO = "resultados_largo.csv"
EXPORTAR_LARGO = True
# Probabilidades crudas de cada hipotesis como columnas. Hacen auditable cada
# valor: sin ellas hay que creerle a la resta.
EXPORTAR_PROBABILIDADES = True

# --- rutas compuestas, que es lo que usan las celdas de escritura ---------
import os as _os_cfg

_os_cfg.makedirs(SALIDA_DIR, exist_ok=True)
RUTA_HTML = _os_cfg.path.join(SALIDA_DIR, SALIDA_HTML)
RUTA_CSV_ANCHO = _os_cfg.path.join(SALIDA_DIR, SALIDA_CSV)
RUTA_CSV_LARGO = _os_cfg.path.join(SALIDA_DIR, SALIDA_CSV_LARGO)
CDN_ECHARTS = "https://cdnjs.cloudflare.com/ajax/libs/echarts/5.6.0/echarts.min.js"
# Las nubes de palabras son una EXTENSION aparte: echarts-wordcloud 2.x envuelve
# wordcloud2.js y registra el tipo de serie `wordCloud`. Segundo punto de falla,
# con su propio aviso en la pagina.
CDN_ECHARTS_WORDCLOUD = ("https://cdnjs.cloudflare.com/ajax/libs/"
                         "echarts-wordcloud/2.1.0/echarts-wordcloud.min.js")

# ===========================================================================
# DE DONDE SALEN LOS TEXTOS
#
#   "bateria"  la bateria de prueba del instrumento. No es una muestra de
#              nada: sirve para medir el instrumento, no el corpus.
#   "propios"  textos pegados a mano en la celda de la seccion 8.
#   "corpus"   el CSV del proyecto.
# ===========================================================================
# La v3 existe para contrastar el instrumento contra la rubrica, que solo
# esta sobre el corpus: por eso arranca en "corpus". La bateria sigue
# disponible para medir el instrumento, pero las secciones 14 a 16 no tienen
# contra que unirla.
FUENTE = "corpus"

# --- configuracion del corpus ----------------------------------------------
# RUTA_CSV = None autodetecta el unico .csv del directorio de trabajo. Si hay
# cero o mas de uno, la carga se detiene y los lista, en vez de elegir por ti.
# Se declara UNA vez, en la primera celda de la seccion 2.
RUTA_CSV = RUTA_ENTRADA

# Sobre que columna corre el MODELO. El metodo lexico corre sobre las dos
# (ver COMPARAR_TRADUCCION), porque es emparejamiento de strings y no cuesta
# pasadas de modelo.
COLUMNA_TEXTO = "text"

# El lexico sobre la traduccion responde la objecion "traduce todo y sigue con
# el lexico" sin consumir una sola pasada de modelo. OJO al interpretarlo:
# mide el lexico JUNTO CON la calidad de una traduccion automatica, y los dos
# efectos no se separan sin traducciones de referencia humanas.
COMPARAR_TRADUCCION = True

# Mapeo declarado, no adivinado. Se valida contra el encabezado real ANTES de
# cargar ningun modelo: si falta una columna, el error enumera las que si
# estan. Pon None en las opcionales que tu archivo no traiga.
COLUMNAS = {
    "texto":       "text",
    "traduccion":  "traducción",
    "tweet_id":    "id",
    "created_at":  "created_at",
    "lang":        "lang",
    "likes":       "like_count",
    "retweets":    "retweet_count",
    "estrato":     "base",          # con que consulta se recolecto
    "modalidad":   "corpus_type",   # volume / consolidated
    "es_retweet":  "is_retweet",
    "es_quote":    "is_quote",
}

# Tope de textos a evaluar. Pon None para el corpus completo. Empieza con 200,
# cronometra, extrapola, y solo entonces sueltalo: son 13 pasadas de un modelo
# large por texto.
# Con el corpus calificado el valor por defecto es None: cualquier tope deja
# tuits calificados fuera de la validacion, y el aviso lo cuenta.
N_MUESTRA = None

# El corpus trae filas repetidas por haber sido recolectadas por mas de una
# consulta. Se deduplica por identificador: no hacerlo las cuenta dos veces en
# cada promedio.
DEDUPE = True

# Corre la bateria con las DOS variantes del ancla y compara las
# distribuciones de relevancia. Cuesta 13 pasadas extra por texto (390 sobre la
# bateria), asi que viene apagado. NO lo actives con el corpus.
# Apagado, la decision de poner el ancla en la plantilla queda solo por el
# argumento mecanico de la seccion 3, sin medicion que la respalde.
COMPARAR_ANCLA = False

# Textos por lote en las llamadas al modelo NLI. Con la bateria da igual; con
# el corpus es la diferencia entre minutos y horas. Bajalo si te quedas sin
# memoria de GPU.
BATCH_SIZE = 16

# Codigos con que Twitter marca "no se detecto idioma". No son idiomas y no se
# dibujan como si lo fueran: se agrupan en una categoria explicita.
CODIGOS_SIN_IDIOMA = {"und", "qme", "qam", "qst", "qct", "qht", "zxx", "art"}
ETIQUETA_SIN_IDIOMA = "sin_idioma"

# Que consultas buscaban expresamente que dimension. Sirve para avisar cuando
# un promedio mezcla estratos dirigidos con estratos que no lo son: promediar
# violencia sobre un corpus donde el 40% se busco POR violencia mide la
# consulta tanto como el fenomeno. Las claves se buscan en minusculas dentro
# del nombre del estrato.
CONSULTAS_DIRIGIDAS = {
    "violence_salience": ("victimizacion", "victimizacion_ja", "violencia", "crimen"),
    "safety_perceived":  ("victimizacion", "victimizacion_ja", "violencia", "seguridad"),
}

# ---------------------------------------------------------------------------
# La escala de cada dimension se DECLARA, no se infiere de los valores
# observados. Con 30 textos, inferir "esta dimension va de 0 a 1 porque no vi
# negativos" es exactamente el error que se quiere evitar.
#
#   valencia    -1..1   el cero es un juicio neutro
#   prominencia  0..1   el cero es ausencia
#
# relevancia : si la dimension trae su propio eje de ausencia (la 3a hipotesis)
# metodos    : que metodos la producen
# grupo      : con que llamada al modelo NLI se obtiene (None = no usa NLI)
#
# imagen_cultural declara UN SOLO metodo: no existe lista de palabras
# equivalente en analyzer.py. Las figuras que comparan metodos la excluyen a
# partir de esta declaracion y lo explican, en vez de descubrirlo por ausencia
# de datos y dibujar un hueco.
# ---------------------------------------------------------------------------
_LEXICOS = ("actual", "traducido") if COMPARAR_TRADUCCION else ("actual",)

ESCALAS = {
    "atmosfera":            {"escala": "valencia",    "relevancia": True,
                             "metodos": _LEXICOS + ("nuevo",), "grupo": "atmosfera"},
    "imagen_cultural":      {"escala": "valencia",    "relevancia": True,
                             "metodos": ("nuevo",),   "grupo": "imagen_cultural"},
    "perspectiva_politica": {"escala": "valencia",    "relevancia": True,
                             "metodos": _LEXICOS + ("nuevo",), "grupo": "perspectiva_politica"},
    "violence_salience":    {"escala": "prominencia", "relevancia": False,
                             "metodos": ("nuevo",),   "grupo": "violencia"},
    "safety_perceived":     {"escala": "valencia",    "relevancia": False,
                             "metodos": ("nuevo",),   "grupo": "violencia"},
    "overall_sentiment":    {"escala": "valencia",    "relevancia": False,
                             "metodos": _LEXICOS + ("nuevo",), "grupo": None},
}

DIMS_CON_RELEVANCIA = [d for d, s in ESCALAS.items() if s["relevancia"]]

# Metodos que son emparejamiento de palabras, y sobre que columna corre cada
# uno. El rediseno lee siempre la columna que declare COLUMNA_TEXTO.
METODOS_LEXICOS = {"actual": "texto", "traducido": "traduccion"}

# ---------------------------------------------------------------------------
# Donde el metodo ACTUAL es un escalon puro.
#
#   perspectiva_politica = clamp(mg * 0.8)   con   mg = clamp((pos - neg) * 0.2)
#
# No incorpora el sentimiento, asi que solo puede tomar 11 valores discretos.
# Eso tiene una consecuencia util: su valor es EXACTO y reproducible sin GPU,
# que es lo que convierte la demostracion de la seccion 11 en un numero citable
# en vez de en el resultado de una corrida.
#
# atmosfera = clamp(s*0.7 + ma) SI incorpora el sentimiento, y el sentimiento
# es continuo: no es escalonada, es una nube desplazada a saltos. Marcar
# valores discretos ahi seria dibujar algo falso.
# ---------------------------------------------------------------------------
ESCALONADAS = {
    "perspectiva_politica": sorted({
        round(min(max(k * 0.2, -1.0), 1.0) * 0.8, 4) for k in range(-5, 6)
    })
}

# --- validacion de la configuracion ----------------------------------------
if FUENTE not in ("bateria", "propios", "corpus"):
    raise ValueError(
        "FUENTE = {!r} no es valido. Usa 'bateria', 'propios' o 'corpus'.".format(FUENTE))

_dims_sin_grupo = [d for d, s in ESCALAS.items()
                   if s["grupo"] and s["grupo"] not in DIMENSIONES and s["grupo"] != "violencia"]
if _dims_sin_grupo:
    raise ValueError(
        "ESCALAS declara grupos que no existen en DIMENSIONES: {}. "
        "Las dos celdas tienen que hablar del mismo conjunto.".format(_dims_sin_grupo))

_dims_sin_escala = [d for d in DIMENSIONES if d not in ESCALAS]
if _dims_sin_escala:
    raise ValueError(
        "DIMENSIONES define {} pero ESCALAS no las declara. "
        "Sin escala declarada ninguna figura sabe como dibujarlas.".format(_dims_sin_escala))

if DISENO_RELEVANCIA not in ("compuerta", "trio"):
    raise ValueError(
        "DISENO_RELEVANCIA = {!r} no es valido. Usa 'compuerta' o 'trio'.".format(
            DISENO_RELEVANCIA))

print("diseño de relevancia    ", DISENO_RELEVANCIA,
      "  (compuerta = multi_label=True aparte)" if DISENO_RELEVANCIA == "compuerta"
      else "  (relevancia = 1 - P(na), softmax de tres)")
print("umbral de relevancia    ", UMBRAL_RELEVANCIA, "  (bajo el cual la valencia no se lee)")
print("umbral de mudez         ", UMBRAL_MUDEZ, "  (arbitrario: justificar en la tesis)")
print("umbral dimension muda   ", UMBRAL_DIMENSION_MUDA)
print("n minimo por figura     ", N_MINIMO_FIGURA, "   por idioma:", N_MINIMO_IDIOMA)
print("dimensiones declaradas  ", len(ESCALAS), "  con relevancia:", len(DIMS_CON_RELEVANCIA))
for _d, _v in ESCALONADAS.items():
    print("escalonada: {:<22} {} valores posibles".format(_d, len(_v)))
_solo_nuevo = [d for d, s in ESCALAS.items() if s["metodos"] == ("nuevo",)]
print("solo rediseno           ", _solo_nuevo)
print()
print("fuente de textos        ", FUENTE)
if FUENTE == "corpus":
    print("   columna del modelo   ", COLUMNA_TEXTO)
    print("   lexico sobre traduccion:", COMPARAR_TRADUCCION)
    print("   tope de muestra      ", N_MUESTRA if N_MUESTRA else "sin tope")
    print("   deduplicar por id    ", DEDUPE)
print("tamano de lote          ", BATCH_SIZE)
print("las salidas van a        {}/  (nunca al directorio de entrada)".format(SALIDA_DIR))

## 8. Los textos a evaluar

`FUENTE`, en la sección 7, elige entre tres entradas.

### `"bateria"` — batería de prueba del instrumento (por defecto)

Cinco frases escritas a propósito, cada una en seis idiomas. 30 textos.

**No es una muestra de nada.** Es un instrumento de prueba: el contenido de las seis
versiones de una misma frase es el mismo, así que cualquier diferencia de medición entre
ellas es atribuible al instrumento y no al texto. Ningún número salido de aquí es
reportable como hallazgo sobre el corpus, y las figuras que salen de ella van rotuladas
como tales.

| Frase | Para qué está |
|---|---|
| `ambiente_pos` | única sonda positiva de `atmosfera`. **Es la frase nueva de este rediseño** |
| `imagen_pos` (comida) | única sonda de `imagen_cultural`, la dimensión sin método léxico |
| `politica_neg` | es el caso de la sección 11; reproduce `MISMA_FRASE` textualmente |
| `violencia` | única vía a `violence_salience` y `safety_perceived` |
| **`control_mudo`** (la gente fue amable) | no habla de ninguna de las tres dimensiones |

El control mudo es el que carga el peso. Sin él no hay forma de demostrar que el eje de
relevancia funciona: es el tuit que debe dar `relevancia ≈ 0` en las tres, y **no** un
juicio neutro.

**Ojo con la reasignación.** El texto de los tacos era el control mudo del diseño anterior.
Con `imagen_cultural` en el conjunto dejó de serlo —hablar de comida *es* hablar del tema—,
así que ascendió a sonda positiva de esa dimensión, y el control mudo pasó a ser la frase
sobre el trato de la gente, que no toca ánimo, ni comida o lugares, ni autoridades.

### `"propios"` — textos pegados a mano

Pega tus textos en la lista de la celda de abajo. La validación de relleno sigue en pie: si
dejas alguno sin sustituir, no corre.

### `"corpus"` — el CSV del proyecto

Lee el archivo del directorio de trabajo (autodetectado si hay uno solo). El mapeo de
columnas se declara en la sección 7 y **se valida contra el encabezado antes de cargar
ningún modelo**: si falta una columna, el error enumera las que sí están.

Empieza con `N_MUESTRA = 200`, cronometra, extrapola, y solo entonces suéltalo. Son 13
pasadas de un modelo *large* por texto.

In [ ]:
# ===========================================================================
# BATERIA DE PRUEBA DEL INSTRUMENTO
#
# NO ES UNA MUESTRA DE NADA. Son cinco frases escritas a proposito, cada una
# en los seis idiomas de la prueba. El contenido de las seis versiones de una
# misma frase es EL MISMO, de modo que cualquier diferencia de medicion entre
# ellas es atribuible al instrumento y no al texto. Ningun numero salido de
# aqui es reportable como hallazgo sobre el corpus; las figuras generadas
# sobre la bateria van rotuladas como tales.
#
# TRADUCCIONES: generadas por el asistente durante el diseno del notebook,
# SIN revision de hablante nativo. Hay que declararlo en la tesis. Si una
# version resulta mas entusiasta que otra, parte de la dispersion que se
# atribuye al instrumento es artefacto de traduccion.
#   Atenuante: eso pesa sobre la columna del REDISENO. La dispersion del
#   METODO ACTUAL no depende de la calidad de la traduccion sino de la
#   AUSENCIA de las palabras en el lexico, y eso se verifica a simple vista
#   en la columna "palabras que activo", que sale vacia.
#
# REASIGNACION. Al cambiar el conjunto de dimensiones, cuatro de las cinco
# frases se reutilizan tal cual y solo una es nueva:
#
#   politica_neg    era gobernanza_neg     -> perspectiva_politica + seccion 11
#   imagen_pos      era el control mudo    -> imagen_cultural  (habla de comida)
#   control_mudo    era hospitalidad_pos   -> control mudo del trio nuevo
#   violencia       igual                  -> violence_salience
#   ambiente_pos    NUEVA                  -> atmosfera
#
# El texto de comida dejaba de ser mudo de todas formas: con imagen_cultural
# en el conjunto, hablar de tacos es hablar del tema. Y "la gente fue amable"
# no habla de animo, ni de comida o lugares, ni de autoridades: sirve de
# control mudo del trio nuevo conservando sus seis versiones.
#
# Las seis versiones de politica_neg son las mismas de MISMA_FRASE (seccion 11)
# a proposito: la seccion 11 imprime su dispersion y la figura la recalcula
# desde aqui, asi que ambas tienen que coincidir. La celda de figuras lo
# comprueba y falla si alguien edita una y no la otra.
# ===========================================================================

BATERIA = {
    "ambiente_pos": {
        "objetivo": "atmosfera",
        "que_prueba": "juicio positivo explicito sobre el animo que se vive; unica sonda positiva de atmosfera",
        "nueva": True,
        "textos": {
            "en": "The atmosphere in the streets was festive and lively, I couldn't stop smiling all day.",
            "es": "El ambiente en las calles fue festivo y animado, no paré de sonreír en todo el día.",
            "pt": "O clima nas ruas era festivo e animado, não parei de sorrir o dia todo.",
            "fr": "L'ambiance dans les rues était festive et animée, je n'ai pas arrêté de sourire de la journée.",
            "de": "Die Stimmung auf den Straßen war festlich und ausgelassen, ich habe den ganzen Tag gelächelt.",
            "ar": "كان الجو في الشوارع احتفاليًا ومفعمًا بالحياة، ولم أتوقف عن الابتسام طوال اليوم.",
        },
    },
    "imagen_pos": {
        "objetivo": "imagen_cultural",
        "que_prueba": "habla de comida y de moverse por la ciudad; unica sonda de imagen_cultural, que no tiene metodo lexico equivalente",
        "nueva": False,
        "textos": {
            "en": "I had three tacos al pastor for breakfast and then took the metro downtown.",
            "es": "Desayuné tres tacos al pastor y luego tomé el metro al centro.",
            "pt": "Tomei três tacos al pastor no café da manhã e depois peguei o metrô para o centro.",
            "fr": "J'ai mangé trois tacos al pastor au petit-déjeuner puis j'ai pris le métro au centre-ville.",
            "de": "Ich habe drei Tacos al Pastor zum Frühstück gegessen und dann die U-Bahn ins Zentrum genommen.",
            "ar": "تناولت ثلاثة تاكو آل باستور على الإفطار ثم ركبت المترو إلى وسط المدينة.",
        },
    },
    "politica_neg": {
        "objetivo": "perspectiva_politica",
        "que_prueba": "juicio negativo explicito sobre autoridades; es la dimension donde el metodo actual es escalon puro y donde INVIERTE EL SIGNO entre idiomas (seccion 11)",
        "nueva": False,
        "textos": {
            "en": "The police were corrupt and did nothing, the authorities failed us completely.",
            "es": "La policía fue corrupta y no hizo nada, las autoridades nos fallaron por completo.",
            "pt": "A polícia era corrupta e não fez nada, as autoridades nos falharam completamente.",
            "fr": "La police était corrompue et n'a rien fait, les autorités nous ont complètement laissés tomber.",
            "de": "Die Polizei war korrupt und tat nichts, die Behörden haben uns völlig im Stich gelassen.",
            "ar": "كانت الشرطة فاسدة ولم تفعل شيئًا، وخذلتنا السلطات تمامًا.",
        },
    },
    "violencia": {
        "objetivo": "violence_salience",
        "que_prueba": "unica via a violence_salience y safety_perceived",
        "nueva": False,
        "textos": {
            "en": "There were gunshots near the stadium last night, people ran in panic.",
            "es": "Hubo disparos cerca del estadio anoche, la gente corrió presa del pánico.",
            "pt": "Houve tiros perto do estádio ontem à noite, as pessoas correram em pânico.",
            "fr": "Il y a eu des coups de feu près du stade hier soir, les gens ont couru pris de panique.",
            "de": "Gestern Abend fielen Schüsse in der Nähe des Stadions, die Menschen rannten in Panik.",
            "ar": "سُمع إطلاق نار قرب الملعب الليلة الماضية، وركض الناس في حالة ذعر.",
        },
    },
    # REBAJADA de control mudo a SONDA DE FRONTERA. "La gente fue amable"
    # tiene proximidad semantica real con "el ambiente fue alegre": que la
    # gente sea amable implica, de forma blanda pero legitima, un ambiente
    # agradable. Parte de su relevancia de 0.83 era el `na` roto, pero parte
    # era que la sonda estaba mal elegida. Ya NO es criterio de exito o
    # fracaso: se documenta su valor como caso limite.
    "frontera_amable": {
        "objetivo": None,
        "que_prueba": "CASO LIMITE, no control. Roza atmosfera por inferencia blanda (gente amable -> ambiente agradable). Se espera relevancia intermedia, ~0.3-0.6, y se documenta; no se usa para aceptar ni rechazar el diseño",
        "nueva": False,
        "textos": {
            "en": "The people here were incredibly welcoming, I felt at home the whole trip.",
            "es": "La gente aquí fue increíblemente amable, me sentí como en casa todo el viaje.",
            "pt": "As pessoas aqui foram incrivelmente acolhedoras, me senti em casa a viagem toda.",
            "fr": "Les gens ici ont été incroyablement accueillants, je me suis senti chez moi tout le voyage.",
            "de": "Die Menschen hier waren unglaublich gastfreundlich, ich fühlte mich die ganze Reise wie zu Hause.",
            "ar": "كان الناس هنا مرحبين بشكل لا يصدق، شعرت وكأنني في بيتي طوال الرحلة.",
        },
    },
    # EL CONTROL MUDO DE VERDAD. Enunciado logistico, sin ninguna carga
    # evaluativa y sin proximidad semantica con ninguna de las tres
    # dimensiones: no habla de animo, ni de comida, lugares o cultura, ni de
    # autoridades, ni de seguridad. Es el que carga el peso de demostrar que
    # el eje de relevancia funciona, y el que "la gente fue amable" no podia
    # cargar.
    "control_logistico": {
        "objetivo": None,
        "que_prueba": "CONTROL MUDO. Enunciado logistico sin carga evaluativa: debe dar relevancia BAJA en las tres dimensiones. Si supera el umbral, la compuerta falla y se rediseña la etiqueta, NO el umbral",
        "nueva": True,
        "textos": {
            "en": "The match kicked off at eight in the evening.",
            "es": "El partido empezó a las ocho de la noche.",
            "pt": "A partida começou às oito da noite.",
            "fr": "Le match a commencé à huit heures du soir.",
            "de": "Das Spiel begann um acht Uhr abends.",
            "ar": "بدأت المباراة في الثامنة مساءً.",
        },
    },
}

TRADUCCION_POR = "asistente (automatica), sin revision de hablante nativo"


# ===========================================================================
# TRES ENTRADAS, UNA SOLA FORMA DE SALIDA
#
# Pase lo que pase aqui, abajo quedan siempre:
#
#   TUITS  [(texto, retweets, likes), ...]     lo que consume la comparacion
#   META   [{texto_id, frase, lang, objetivo, tweet_id, created_at,
#            estrato, modalidad, traduccion}, ...]   en la MISMA posicion
#
# La procedencia viaja aparte, en META, para que TUITS conserve la forma que
# ya esperaba la celda de comparacion.
# ===========================================================================

def _meta_vacia(texto_id):
    return {"texto_id": texto_id, "frase": None, "lang": None, "objetivo": None,
            "tweet_id": None, "created_at": None, "estrato": None,
            "modalidad": None, "traduccion": None, "es_retweet": None,
            "es_quote": None}


def _cargar_bateria():
    tuits, meta = [], []
    for frase, spec in BATERIA.items():
        for lang, texto in spec["textos"].items():
            tuits.append((texto, 0, 0))
            m = _meta_vacia("{}:{}".format(frase, lang))
            m.update(frase=frase, lang=lang, objetivo=spec["objetivo"])
            meta.append(m)
    return tuits, meta


def _cargar_propios():
    # TUS TUITS AQUI. Sustituye los textos de relleno; la validacion de abajo
    # no te deja correr con ellos puestos.
    tuits = [
        # ("texto del tuit", retweets, likes)
        ("PEGA AQUI TU PRIMER TUIT", 0, 0),
        ("PEGA AQUI TU SEGUNDO TUIT", 0, 0),
    ]
    meta = [_meta_vacia("tuit_{:03d}".format(i)) for i in range(len(tuits))]
    relleno = [t for t, _, _ in tuits if t.startswith("PEGA AQUI")]
    if relleno:
        raise ValueError(
            "Todavia hay {} texto(s) de relleno en TUITS. Reemplazalos por tus "
            "tuits reales, o pon FUENTE = 'bateria' para correr con la bateria "
            "de ejemplo.".format(len(relleno)))
    return tuits, meta


def _localizar_csv():
    import glob, os
    if RUTA_CSV:
        if not os.path.exists(RUTA_CSV):
            raise FileNotFoundError(
                "RUTA_CSV = {!r} no existe. En Colab sube el archivo al panel "
                "de archivos, o deja RUTA_CSV = None para autodetectarlo."
                .format(RUTA_CSV))
        return RUTA_CSV
    # Los CSV que ESTE notebook escribe no son candidatos a entrada. Sin esto,
    # la segunda corrida encuentra dos archivos y se detiene pidiendo RUTA_CSV:
    # el notebook se bloquearia a si mismo en cuanto exporta una vez. Las
    # salidas viven en SALIDA_DIR; la exclusion es el segundo cinturon por si
    # alguien pone SALIDA_DIR = ".".
    _propios = {os.path.basename(p).lower()
                for p in (globals().get("SALIDA_CSV"), globals().get("SALIDA_CSV_LARGO"),
                          "candidatos_fn.csv", "asociacion_criterios.csv") if p}
    candidatos = [p for p in sorted(glob.glob("*.csv"))
                  if os.path.basename(p).lower() not in _propios]
    if len(candidatos) == 1:
        return candidatos[0]
    if not candidatos:
        raise FileNotFoundError(
            "No hay ningun .csv en el directorio de trabajo ({}). Sube el "
            "corpus al panel de archivos de Colab, o declara RUTA_CSV."
            .format(os.getcwd()))
    raise ValueError(
        "Hay {} archivos .csv y no se cual quieres: {}. Declara RUTA_CSV."
        .format(len(candidatos), candidatos))


def _a_bool(v):
    """'False', 'FALSE' y 'false' son el mismo valor. En el corpus conviven."""
    if v is None:
        return None
    s = str(v).strip().lower()
    if s in ("true", "1", "yes", "si", "sí"):
        return True
    if s in ("false", "0", "no", ""):
        return False
    return None


CORPUS_DF = None       # marco de origen, solo cuando FUENTE == "corpus"
CORPUS_RUTA = None


def _cargar_corpus():
    import pandas as pd

    ruta = _localizar_csv()
    # utf-8-sig: el archivo trae marca de orden de bytes y sin esto la primera
    # columna se llamaria '﻿base' y el mapeo fallaria por una razon
    # invisible al leer el error.
    df = pd.read_csv(ruta, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    print("archivo      :", ruta)
    print("filas leidas :", len(df), " columnas:", len(df.columns))

    # --- validacion del mapeo ANTES de tocar ningun modelo -----------------
    faltan = [k for k, c in COLUMNAS.items() if c and c not in df.columns]
    if faltan:
        raise KeyError(
            "El mapeo COLUMNAS pide columnas que el archivo no tiene: {}.\n"
            "Las {} columnas presentes son:\n  {}\n"
            "Corrige COLUMNAS en la celda de configuracion, o pon None en las "
            "que tu archivo no traiga."
            .format({k: COLUMNAS[k] for k in faltan}, len(df.columns),
                    "\n  ".join(sorted(df.columns))))

    col_modelo = COLUMNA_TEXTO
    if col_modelo not in df.columns:
        raise KeyError(
            "COLUMNA_TEXTO = {!r} no existe en el archivo. Columnas: {}"
            .format(col_modelo, sorted(df.columns)))

    n0 = len(df)
    # --- filas sin texto ---------------------------------------------------
    vacias = df[col_modelo].astype(str).str.strip() == ""
    if vacias.any():
        print("sin texto    :", int(vacias.sum()), "filas descartadas")
        df = df[~vacias]

    # --- deduplicacion -----------------------------------------------------
    col_id = COLUMNAS.get("tweet_id")
    if DEDUPE and col_id:
        antes = len(df)
        df = df.drop_duplicates(subset=[col_id], keep="first")
        if antes != len(df):
            print("duplicados   :", antes - len(df),
                  "filas con el mismo", col_id, "descartadas")

    # --- muestreo ----------------------------------------------------------
    if N_MUESTRA and len(df) > N_MUESTRA:
        print("muestra      :", N_MUESTRA, "de", len(df),
              "({} omitidos: sube N_MUESTRA o ponlo en None)".format(len(df) - N_MUESTRA))
        df = df.head(N_MUESTRA)

    # --- informes sobre lo que falta, en vez de fallar en silencio ---------
    col_fecha = COLUMNAS.get("created_at")
    if col_fecha:
        sin_fecha = (df[col_fecha].astype(str).str.strip() == "").sum()
        if sin_fecha:
            print("sin fecha    :", int(sin_fecha),
                  "filas cargadas con la fecha marcada como no disponible")
    col_trad = COLUMNAS.get("traduccion")
    if COMPARAR_TRADUCCION and col_trad:
        sin_trad = (df[col_trad].astype(str).str.strip() == "").sum()
        if sin_trad:
            print("sin traducir :", int(sin_trad),
                  "filas quedan fuera de la via traducida (no se sustituyen "
                  "por el original)")

    def _num(v):
        try:
            return int(float(str(v).strip() or 0))
        except (TypeError, ValueError):
            return 0

    def _opt(fila, clave):
        c = COLUMNAS.get(clave)
        if not c:
            return None
        v = str(fila.get(c, "") or "").strip()
        return v or None

    # Los booleanos del archivo mezclan capitalizaciones ('False' y 'FALSE'
    # conviven, seguramente por un lote fusionado de otra fuente). Sin
    # normalizar, un value_counts los cuenta como dos categorias distintas.
    for _clave in ("es_retweet", "es_quote"):
        _c = COLUMNAS.get(_clave)
        if _c:
            _vals = sorted(set(df[_c].astype(str).str.strip()))
            if len(_vals) > 2:
                print("booleanos    :", _c, "trae", _vals,
                      "-> se normalizan sin distinguir mayusculas")

    # Se conserva el marco de origen: la descarga devuelve el archivo ENTERO
    # con las columnas de resultados pegadas, y para eso hacen falta todas las
    # columnas originales, no solo las que el modelo consume.
    global CORPUS_DF, CORPUS_RUTA
    CORPUS_DF, CORPUS_RUTA = df.copy(), ruta

    tuits, meta = [], []
    for pos, (_, fila) in enumerate(df.iterrows()):
        texto = str(fila[col_modelo])
        tuits.append((texto,
                      _num(fila.get(COLUMNAS.get("retweets"), 0)) if COLUMNAS.get("retweets") else 0,
                      _num(fila.get(COLUMNAS.get("likes"), 0)) if COLUMNAS.get("likes") else 0))
        tid = _opt(fila, "tweet_id")
        m = _meta_vacia(tid or "fila_{:05d}".format(pos))
        m.update(tweet_id=tid,
                 created_at=_opt(fila, "created_at"),
                 lang=_opt(fila, "lang"),
                 estrato=_opt(fila, "estrato"),
                 modalidad=_opt(fila, "modalidad"),
                 traduccion=_opt(fila, "traduccion"),
                 es_retweet=_a_bool(_opt(fila, "es_retweet")),
                 es_quote=_a_bool(_opt(fila, "es_quote")))
        meta.append(m)
    print("cargados     :", len(tuits), "textos de", n0, "filas del archivo")
    return tuits, meta


# --- construccion ----------------------------------------------------------
if FUENTE == "bateria":
    TUITS, META = _cargar_bateria()
elif FUENTE == "propios":
    TUITS, META = _cargar_propios()
elif FUENTE == "corpus":
    TUITS, META = _cargar_corpus()
else:
    raise ValueError("FUENTE = {!r} no es valido".format(FUENTE))

# Rotulo de procedencia de TRES estados. Antes era binario (bateria si/no) y
# ya no alcanza: el rotulo del corpus no dice "no reportable" sino algo
# distinto y mas facil de pasar por alto -- DESCRIPTIVO PERO NO VALIDADO,
# porque no hay codificacion humana contra la cual contrastar.
PROCEDENCIA = FUENTE
ES_BATERIA = (FUENTE == "bateria")

# --- costo de la corrida, derivado y no escrito a mano ---------------------
# El numero de pasadas sale del numero de etiquetas de cada grupo de
# hipotesis: zero-shot con multi_label=False hace una pasada por etiqueta.
_ETIQUETAS_POR_GRUPO = {"violencia": len(VIOLENCIA[IDIOMA_HIPOTESIS])}
if DISENO_RELEVANCIA == "compuerta":
    # una pasada por dimension para la compuerta, y solo pos/neg en cada una
    _ETIQUETAS_POR_GRUPO["compuerta"] = len(GATE[IDIOMA_HIPOTESIS])
for _d, _t in DIMENSIONES.items():
    _ETIQUETAS_POR_GRUPO[_d] = (
        2 if DISENO_RELEVANCIA == "compuerta" else len(_t[IDIOMA_HIPOTESIS]))
PASADAS_POR_TEXTO = sum(_ETIQUETAS_POR_GRUPO.values())

_ROTULO = {"bateria": "BATERIA DE PRUEBA -- no es una muestra de nada",
           "propios": "textos propios",
           "corpus":  "CORPUS -- descriptivo; contrastado contra la rubrica LLM (seccion 14), "
                      "NO contra codificacion humana"}

print()
print("{} textos listos  ({})".format(len(TUITS), _ROTULO[PROCEDENCIA]))
if ES_BATERIA:
    print("   {} frases x {} idiomas".format(
        len(BATERIA), len(next(iter(BATERIA.values()))["textos"])))
    print("   traducciones:", TRADUCCION_POR)
    _nuevas = [f for f, s in BATERIA.items() if s.get("nueva")]
    if _nuevas:
        print("   redactadas para este rediseno:", ", ".join(_nuevas))
if PROCEDENCIA == "corpus":
    _idiomas = {}
    for _m in META:
        _idiomas[_m["lang"] or "?"] = _idiomas.get(_m["lang"] or "?", 0) + 1
    print("   idiomas: " + ", ".join("{}={}".format(k, v) for k, v in
                                     sorted(_idiomas.items(), key=lambda p: -p[1])[:8]))
    _estratos = {}
    for _m in META:
        _estratos[_m["estrato"] or "?"] = _estratos.get(_m["estrato"] or "?", 0) + 1
    print("   estratos: {} distintos".format(len(_estratos)))
print()
print("costo de la corrida")
print("   pasadas de NLI por texto: {}   {}".format(
    PASADAS_POR_TEXTO,
    " + ".join("{}={}".format(k, v) for k, v in _ETIQUETAS_POR_GRUPO.items())))
print("   total:                    {} pasadas de un modelo large".format(
    len(TUITS) * PASADAS_POR_TEXTO))

try:
    import torch as _torch
    if not _torch.cuda.is_available() and not CARGAR_ZEROSHOT:
        print()
        print("*" * 74)
        print("SIN GPU. Son {} pasadas de un modelo large; en CPU esto puede ser".format(
            len(TUITS) * PASADAS_POR_TEXTO))
        print("impracticable. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4)")
        print("*" * 74)
except Exception:
    pass

In [ ]:
# Dimensiones que tienen metodo lexico, derivadas de ESCALAS en vez de escritas
# a mano: si manana una dimension gana o pierde metodo, esto se entera solo.
LEX = [d for d, s in ESCALAS.items()
       if "actual" in s["metodos"] and d != "overall_sentiment"]

# Metodos lexicos realmente disponibles en esta corrida. "traducido" solo
# existe si hay columna de traduccion poblada.
_HAY_TRADUCCION = any(m.get("traduccion") for m in META)
METODOS_ACTIVOS = ["actual"]
if COMPARAR_TRADUCCION and _HAY_TRADUCCION:
    METODOS_ACTIVOS.append("traducido")


def _texto_de(metodo, texto, meta):
    """Sobre que cadena corre cada metodo lexico.

    La via traducida NO cae de vuelta al original cuando falta la traduccion:
    devuelve None y la fila se omite. Sustituirla en silencio mezclaria dos
    mediciones distintas bajo la misma etiqueta.
    """
    if metodo == "traducido":
        return meta.get("traduccion")
    return texto


# ===========================================================================
# EVALUACION
#
# El rediseno corre POR LOTES: una llamada al modelo por grupo de hipotesis
# para todos los textos, en vez de 13 llamadas por texto. Con la bateria da
# igual; con el corpus es la diferencia entre minutos y horas.
#
# El metodo lexico corre despues, texto por texto, porque es emparejamiento de
# strings. Lo unico que cuesta de la via traducida es UNA pasada del modelo de
# SENTIMIENTO (base, ~1.1 GB) por texto: ninguna pasada de NLI, que es el
# modelo large y el que domina el costo.
# ===========================================================================
import time
import gzip
import hashlib
import datetime as _dt_ev
import json as _json_ev
import os as _os_ev

# ---------------------------------------------------------------------------
# HUELLA DEL INSTRUMENTO. Todo lo que, si cambia, cambia los valores: las
# hipotesis, la compuerta, las plantillas y de donde sale la relevancia. Una
# corrida previa solo se reutiliza si su huella es la misma.
# ---------------------------------------------------------------------------
def huella_instrumento():
    obj = {"idioma": IDIOMA_HIPOTESIS, "plantilla": PLANTILLA,
           "plantilla_gate": PLANTILLA_GATE, "diseno_relevancia": DISENO_RELEVANCIA,
           "ancla_en_etiquetas": ANCLA_EN_ETIQUETAS, "dimensiones": DIMENSIONES,
           "violencia": VIOLENCIA, "gate": GATE}
    txt = _json_ev.dumps(obj, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()[:16]


HUELLA_INSTRUMENTO = huella_instrumento()
RUTA_VOLCADO = _os_ev.path.join(SALIDA_DIR, "zeroshot_corrida.json.gz")


def escribir_volcado(extra=None):
    """La corrida entera, para no volver a pagar la GPU. Se escribe al terminar
    de evaluar y la seccion 17 lo completa con la demostracion y el A/B."""
    v = {"huella": HUELLA_INSTRUMENTO, "fecha": _dt_ev.datetime.now().strftime("%Y-%m-%d %H:%M"),
         "procedencia": PROCEDENCIA, "entrada": CORPUS_RUTA if PROCEDENCIA == "corpus" else None,
         "n_muestra": N_MUESTRA, "ids": [m["texto_id"] for m in META],
         "tiempo_corrida_s": TIEMPO_CORRIDA, "resultados": resultados}
    v.update(extra or {})
    with gzip.open(RUTA_VOLCADO, "wt", encoding="utf-8") as fh:
        _json_ev.dump(v, fh, ensure_ascii=False)
    return RUTA_VOLCADO


def leer_volcado(ruta):
    if not _os_ev.path.exists(ruta):
        raise FileNotFoundError(
            "CARGAR_ZEROSHOT = {!r} no existe. Sube el volcado de la corrida anterior "
            "(salida/zeroshot_corrida.json.gz) o pon CARGAR_ZEROSHOT = None.".format(ruta))
    with gzip.open(ruta, "rt", encoding="utf-8") as fh:
        return _json_ev.load(fh)


_textos = [t for t, _, _ in TUITS]
_rts = [rt for _, rt, _ in TUITS]
_lks = [lk for _, _, lk in TUITS]
_sin_traduccion = 0
VOLCADO_PREVIO = None

if CARGAR_ZEROSHOT:
    VOLCADO_PREVIO = leer_volcado(CARGAR_ZEROSHOT)
    if VOLCADO_PREVIO.get("huella") != HUELLA_INSTRUMENTO:
        raise ValueError(
            "La corrida previa se hizo con OTRO instrumento: huella {} contra la actual {}. "
            "Cambiaron las hipotesis, la compuerta, las plantillas o DISENO_RELEVANCIA "
            "desde que se escribio. Sus valores no corresponden a esta configuracion: "
            "pon CARGAR_ZEROSHOT = None y vuelve a evaluar.".format(
                VOLCADO_PREVIO.get("huella"), HUELLA_INSTRUMENTO))
    _ids_act = [m["texto_id"] for m in META]
    _ids_vol = VOLCADO_PREVIO["ids"]
    if _ids_vol == _ids_act:
        resultados = VOLCADO_PREVIO["resultados"]
    elif len(_ids_vol) == len(_ids_act) and set(_ids_vol) == set(_ids_act):
        _pos = {t: i for i, t in enumerate(_ids_vol)}
        resultados = [VOLCADO_PREVIO["resultados"][_pos[t]] for t in _ids_act]
        print("   el volcado trae los mismos textos en otro orden: se reordena por identificador")
    else:
        _solo_vol = len(set(_ids_vol) - set(_ids_act))
        _solo_act = len(set(_ids_act) - set(_ids_vol))
        raise ValueError(
            "La corrida previa es de OTRO corpus: {} identificadores estan en el volcado y no "
            "en la entrada actual, y {} en la entrada y no en el volcado ({} contra {} textos). "
            "Revisa RUTA_ENTRADA, N_MUESTRA y DEDUPE, o pon CARGAR_ZEROSHOT = None."
            .format(_solo_vol, _solo_act, len(_ids_vol), len(_ids_act)))
    TIEMPO_CORRIDA = VOLCADO_PREVIO.get("tiempo_corrida_s") or 0.0
    for _r in resultados:
        if "traducido" in METODOS_ACTIVOS and _r.get("traducido") is None:
            _sin_traduccion += 1
    print("corrida restaurada de {} ({}): {} textos, huella {}".format(
        CARGAR_ZEROSHOT, VOLCADO_PREVIO.get("fecha"), len(resultados), HUELLA_INSTRUMENTO))
    print("   no se llamo a ningun modelo; el tiempo que se informa es el de la corrida original")
else:
    _t0 = time.perf_counter()

    print("evaluando {} textos con el rediseno (lotes de {})...".format(
        len(_textos), BATCH_SIZE))
    _nuevos = metodo_nuevo_lote(_textos, _rts, _lks)

    print("evaluando el metodo lexico sobre: {}".format(", ".join(METODOS_ACTIVOS)))
    resultados = []
    for _i, (_t, _rt, _lk) in enumerate(TUITS):
        _fila = {"nuevo": _nuevos[_i]}
        for _m in METODOS_ACTIVOS:
            _src = _texto_de(_m, _t, META[_i])
            if _src is None:
                _sin_traduccion += 1
                _fila[_m] = None
            else:
                _fila[_m] = metodo_actual(_src)
        resultados.append(_fila)

    TIEMPO_CORRIDA = time.perf_counter() - _t0

    # La GPU ya se pago: se guarda ANTES de seguir, por si algo de abajo falla.
    print("volcado de la corrida escrito en {} (huella {})".format(
        escribir_volcado(), HUELLA_INSTRUMENTO))

if _sin_traduccion:
    print("   {} textos sin traduccion quedan fuera de la via traducida "
          "(no se sustituyen por el original)".format(_sin_traduccion))


# ===========================================================================
# IMPRESION. Se lee de `resultados`, ya calculado: imprimir no vuelve a
# llamar a ningun modelo.
# ===========================================================================
def comparar(i):
    r = resultados[i]
    texto, rt, lk = TUITS[i]
    a, n = r["actual"], r["nuevo"]
    print("=" * 88)
    print("TUIT: " + texto[:200].replace("\n", " "))
    print("-" * 88)
    print("  sentimiento = {:+.4f}  (label crudo {!r})".format(
        a["overall_sentiment"], a["_label"]))
    print()
    _cols = ["{:>10}".format(m.upper()) for m in METODOS_ACTIVOS]
    print("  {:<22}{} {:>10} {:>12} {:>20}".format(
        "dimension", "".join(_cols), "NUEVO", "relevancia", "palabras que activo"))
    print("  " + "-" * 86)
    for d in LEX:
        vals = ""
        for m in METODOS_ACTIVOS:
            vals += ("{:>+10.4f}".format(r[m][d]) if r[m] else "{:>10}".format("--"))
        pal = a["_palabras"][d]
        print("  {:<22}{} {:>+10.4f} {:>12.2f} {:>20}".format(
            d, vals, n[d], n[d + "_relevancia"], str(pal) if pal else "(ninguna)"))
    for d in ESCALAS:
        if d in LEX or d == "overall_sentiment" or d not in n:
            continue
        _rel = n.get(d + "_relevancia")
        print("  {:<22}{} {:>+10.4f} {:>12}".format(
            d, "{:>10}".format("--") * len(METODOS_ACTIVOS), n[d],
            "{:.2f}".format(_rel) if _rel is not None else "--"))
    print()
    print("  probabilidades crudas del modelo NLI:")
    for grupo, probs in n["_crudo"].items():
        detalle = "  ".join("{}={:.3f}".format(k, v) for k, v in probs.items())
        print("     {:<22} {}".format(grupo, detalle))
    print()
    return a, n


# Con el corpus no se imprimen 2 600 bloques: solo una muestra.
_A_IMPRIMIR = range(len(TUITS)) if PROCEDENCIA == "bateria" else range(min(5, len(TUITS)))
for _i in _A_IMPRIMIR:
    comparar(_i)
if len(_A_IMPRIMIR) < len(TUITS):
    print("... {} bloques mas no impresos (son {} textos). Los marcos y las "
          "figuras los tienen todos.".format(len(TUITS) - len(_A_IMPRIMIR), len(TUITS)))

print("=" * 88)
print("{} textos  ·  {} pasadas de NLI  ·  {:.1f} s  ({:.3f} s/texto){}".format(
    len(TUITS), len(TUITS) * PASADAS_POR_TEXTO, TIEMPO_CORRIDA,
    TIEMPO_CORRIDA / max(len(TUITS), 1),
    "  [corrida original, restaurada]" if CARGAR_ZEROSHOT else ""))
if PROCEDENCIA != "corpus" or (N_MUESTRA and N_MUESTRA < 2000):
    print("Cronometra esto contra el tamano de tu corpus antes de comprometerte.")
    print("   a este ritmo, 2 631 textos tardarian {:.1f} min".format(
        TIEMPO_CORRIDA / max(len(TUITS), 1) * 2631 / 60))
print("=" * 88)

## 9. Verificación del ancla geográfica

La sección 3 afirma que meter «en México» **dentro** de cada hipótesis desplaza probabilidad
hacia `na` y hunde la relevancia. El argumento es mecánico —el softmax cancela un prefijo
constante, no un predicado que encarece las afirmaciones y abarata la negación— pero un
argumento no es una medición.

Esta sección lo mide. Corre la batería **dos veces**, con el ancla en la plantilla y con el
ancla en las etiquetas, y compara las distribuciones de relevancia. La variante alterna se
**deriva** de las mismas frases, así que no pueden diferir en nada más que en el ancla.

Cuesta 13 pasadas extra por texto, así que viene apagada: pon `COMPARAR_ANCLA = True` en la
sección 7. Sobre la batería son 390 pasadas más; no la actives con el corpus.

**Si el efecto no aparece, la decisión hay que revisarla.** Esta celda existe para poder
decirlo con el número delante, no para confirmar lo que ya se escribió.

In [ ]:
# ===========================================================================
# A/B DEL ANCLA GEOGRAFICA
#
# Mide lo que la seccion 3 afirma, en vez de darlo por bueno. La variante
# alterna se DERIVA de las mismas frases (misma funcion _con_ancla que usa la
# celda de dimensiones), asi que la unica diferencia entre las dos corridas es
# donde vive el ancla.
#
# Resultado -> COMPARACION_ANCLA, que la celda de figuras embebe.
# ===========================================================================
COMPARACION_ANCLA = None

if CARGAR_ZEROSHOT:
    COMPARACION_ANCLA = (VOLCADO_PREVIO or {}).get("comparacion_ancla")
    print("Corrida restaurada del volcado: el A/B del ancla no se vuelve a correr.")
    print("   {}".format("se restauro el de la corrida original" if COMPARACION_ANCLA
                       else "la corrida original no lo tenia"))
elif not COMPARAR_ANCLA:
    print("A/B del ancla desactivado (COMPARAR_ANCLA = False).")
    print("La decision de poner el ancla en la plantilla queda por el argumento")
    print("mecanico de la seccion 3, SIN medicion que la respalde. Actívalo para")
    print("respaldarla o para refutarla.")
else:
    _otra = not ANCLA_EN_ETIQUETAS

    def _variante(en_etiquetas):
        """Devuelve (plantilla, dimensiones, violencia) para una variante."""
        pl = _PLANTILLAS[en_etiquetas][IDIOMA_HIPOTESIS]
        # se parte SIEMPRE de las frases sin anclar, vengan como vengan
        base_d = {d: {idi: dict(t[idi]) for idi in t} for d, t in DIMENSIONES.items()}
        base_v = {idi: dict(VIOLENCIA[idi]) for idi in VIOLENCIA}
        if ANCLA_EN_ETIQUETAS:
            # las globales YA vienen ancladas: se les quita para tener la base
            _n = len(_ANCLA[IDIOMA_HIPOTESIS])
            base_d = {d: {idi: {k: v[:-_n] if v.endswith(_ANCLA[idi]) else v
                                for k, v in t[idi].items()} for idi in t}
                      for d, t in base_d.items()}
            base_v = {idi: {k: v[:-_n] if v.endswith(_ANCLA[idi]) else v
                            for k, v in base_v[idi].items()} for idi in base_v}
        if en_etiquetas:
            base_d = {d: {idi: _con_ancla(t[idi], idi) for idi in t}
                      for d, t in base_d.items()}
            base_v = {idi: _con_ancla(base_v[idi], idi) for idi in base_v}
        return pl, base_d, base_v

    def _correr_variante(en_etiquetas, textos):
        global PLANTILLA, DIMENSIONES, VIOLENCIA
        _guardado = (PLANTILLA, DIMENSIONES, VIOLENCIA)
        PLANTILLA, DIMENSIONES, VIOLENCIA = _variante(en_etiquetas)
        try:
            return metodo_nuevo_lote(textos)
        finally:
            PLANTILLA, DIMENSIONES, VIOLENCIA = _guardado

    _txt = [t for t, _, _ in TUITS]
    print("corriendo la bateria con las DOS variantes ({} textos x 2)...".format(len(_txt)))
    _res = {"plantilla": _correr_variante(False, _txt),
            "etiquetas": _correr_variante(True, _txt)}

    _dims = [d for d in DIMENSIONES if d in ESCALAS and ESCALAS[d]["relevancia"]]
    _filas = []
    for dim in _dims:
        fila = {"dimension": dim}
        for var, salidas in _res.items():
            rels = [o[dim + "_relevancia"] for o in salidas]
            media = sum(rels) / len(rels)
            fila[var] = {
                "relevancia_media": round(media, 4),
                "relevancia_min": round(min(rels), 4),
                "relevancia_max": round(max(rels), 4),
                "prop_muda": round(sum(1 for r in rels if r < UMBRAL_MUDEZ) / len(rels), 4),
                "dimension_muda": bool(
                    sum(1 for r in rels if r < UMBRAL_MUDEZ) / len(rels) > UMBRAL_DIMENSION_MUDA),
            }
        fila["delta_relevancia"] = round(
            fila["etiquetas"]["relevancia_media"] - fila["plantilla"]["relevancia_media"], 4)
        _filas.append(fila)

    _mudas_pl = sum(1 for f in _filas if f["plantilla"]["dimension_muda"])
    _mudas_et = sum(1 for f in _filas if f["etiquetas"]["dimension_muda"])
    _delta = sum(f["delta_relevancia"] for f in _filas) / len(_filas)
    # El efecto predicho: el ancla en las etiquetas infla na, luego BAJA la
    # relevancia. delta negativo = la prediccion se cumple.
    _confirmado = _delta < 0

    COMPARACION_ANCLA = {
        "filas": _filas,
        "delta_medio": round(_delta, 4),
        "dimensiones_mudas": {"plantilla": _mudas_pl, "etiquetas": _mudas_et},
        "prediccion_confirmada": bool(_confirmado),
        "umbral_mudez": UMBRAL_MUDEZ,
    }

    print()
    print("=" * 80)
    print("RELEVANCIA SEGUN DONDE VIVA EL ANCLA")
    print("=" * 80)
    print("  {:<22} {:>12} {:>12} {:>10}   {}".format(
        "dimension", "plantilla", "etiquetas", "delta", "% mudo (plant / etiq)"))
    print("  " + "-" * 78)
    for f in _filas:
        print("  {:<22} {:>12.4f} {:>12.4f} {:>+10.4f}   {:>5.0%} / {:<5.0%}".format(
            f["dimension"], f["plantilla"]["relevancia_media"],
            f["etiquetas"]["relevancia_media"], f["delta_relevancia"],
            f["plantilla"]["prop_muda"], f["etiquetas"]["prop_muda"]))
    print()
    print("  delta medio de relevancia: {:+.4f}".format(_delta))
    print("  dimensiones declaradas mudas:  plantilla {} · etiquetas {}".format(
        _mudas_pl, _mudas_et))
    print()
    if _confirmado:
        print("  PREDICCION CONFIRMADA. El ancla dentro de las etiquetas baja la")
        print("  relevancia, que es lo que la seccion 3 argumenta: infla `na` porque")
        print("  anadir el predicado a una negacion la hace mas facil de entrañar.")
        print("  La decision de ponerla en la plantilla queda respaldada por el numero.")
    else:
        print("  " + "!" * 76)
        print("  PREDICCION NO CONFIRMADA. El ancla dentro de las etiquetas NO baja la")
        print("  relevancia en esta corrida. El argumento mecanico de la seccion 3 no")
        print("  se sostiene tal como esta escrito, y la decision de diseño hay que")
        print("  REVISARLA antes de reportar nada. No lo tapes: documentalo.")
        print("  " + "!" * 76)
    print("=" * 80)

## 10. Marcos de datos

Un solo punto de extracción. Se recorre `resultados` una vez y salen tres marcos:

```
LARGO    (texto_id, metodo, dimension)  →  valencia, relevancia, escala
PROBS    (texto_id, grupo, etiqueta)    →  probabilidad
LEXICO   (texto_id, dimension, palabra) →  polaridad, es_subcadena
```

Tres claves distintas quieren tres marcos. Forzar uno solo obligaría a nulos estructurales:
la relevancia no existe para el método actual, las probabilidades crudas cuelgan del **grupo
de llamada NLI** y no de la dimensión reportada, y las palabras del léxico no tienen
equivalente en el rediseño.

Toda figura y toda tabla leen de aquí. Esto recupera lo que la tabla resumen descartaba
—`violence_salience`, `safety_perceived`, el sentimiento y las probabilidades crudas— y de
paso elimina la dependencia oculta que tenía de la sección 11.

**La guardia importante:** `valencia` y `relevancia` no se combinan nunca en un solo número.
El producto es tentador porque reduce todo a un eje, y reconstruye exactamente el defecto
que el rediseño elimina:

```
habló y juzgó neutro    0.02 × 0.91 = 0.018
no habló del tema       0.60 × 0.03 = 0.018
```

El mismo número para dos cosas distintas: el `0.0000` ambiguo con otro nombre. Hay una
aserción que falla si alguien agrega cualquier columna numérica nueva, que es la vía por la
que ese producto entraría sin que nadie lo note.

In [ ]:
import pandas as pd
import re

# ===========================================================================
# UN SOLO PUNTO DE EXTRACCION, TRES FORMAS
#
# Se recorre `resultados` una vez y salen tres marcos con claves limpias:
#
#   LARGO   (texto_id, metodo, dimension)  -> valencia, relevancia, escala
#   PROBS   (texto_id, grupo, etiqueta)    -> probabilidad
#   LEXICO  (texto_id, metodo, dimension, palabra) -> polaridad, es_subcadena
#
# Tres claves distintas quieren tres marcos. Forzar uno solo obligaria a
# nulos estructurales: la relevancia no existe para los metodos lexicos, las
# probabilidades crudas cuelgan del GRUPO de llamada NLI y no de la dimension
# reportada, y las palabras del lexico no tienen equivalente en el rediseno.
#
# La columna `metodo` admite MAS DE DOS valores: 'actual' (lexico sobre el
# texto original), 'traducido' (el mismo lexico sobre la traduccion) y
# 'nuevo'. Es un valor mas de una columna que ya existia, no una columna
# nueva: por eso las figuras lo recogen sin cambios.
#
# Toda figura y toda tabla leen de aqui. Nada vuelve a `resultados`.
# ===========================================================================

_LETRA = r"[^\W\d_]"


def es_subcadena(palabra, texto_lower):
    """True si la palabra SOLO aparece dentro de otra palabra mas larga.

    El metodo actual empareja con `w in tl`, sin frontera de palabra. Eso hace
    que 'safe' se active dentro de 'unsafe', 'corrupt' dentro de 'corrupta' y
    'kind' dentro de 'kinda'. Aqui no se corrige nada: se marca, para poder
    medir cuanto pesa.
    """
    patron = r"(?<!{L}){p}(?!{L})".format(L=_LETRA, p=re.escape(palabra))
    return re.search(patron, texto_lower) is None


def grupo_idioma(lang):
    """Los codigos con que Twitter marca 'no se detecto idioma' no son idiomas.

    Dibujarlos como si lo fueran inventa categorias; meterlos en un cajon de
    sastre sin nombre las esconde. Se agrupan bajo una etiqueta explicita.
    """
    if not lang:
        return None
    return ETIQUETA_SIN_IDIOMA if lang in CODIGOS_SIN_IDIOMA else lang


def construir_frames(resultados, tuits, meta):
    filas_valor, filas_prob, filas_lex = [], [], []

    for i, ((texto, rt, lk), r) in enumerate(zip(tuits, resultados)):
        m = meta[i]
        n = r["nuevo"]
        base = {
            "texto_id":   m["texto_id"],
            "frase":      m["frase"],
            "lang":       m["lang"],
            "lang_grupo": grupo_idioma(m["lang"]),
            "objetivo":   m["objetivo"],
            "texto":      texto,
            # Columnas de procedencia: vacias mientras la entrada sea una lista
            # de textos, pobladas cuando lo sea el corpus.
            "tweet_id":   m["tweet_id"],
            "created_at": m["created_at"],
            # Estratos de recoleccion. Sin estos, un promedio de violencia
            # sobre un corpus donde el 40% se busco POR violencia mide sobre
            # todo la consulta que lo recolecto.
            "estrato":    m["estrato"],
            "modalidad":  m["modalidad"],
            "es_retweet": m.get("es_retweet"),
            "es_quote":   m.get("es_quote"),
        }

        # --- valores por dimension y metodo --------------------------------
        for dim, spec in ESCALAS.items():
            for metodo in spec["metodos"]:
                src = r.get(metodo)
                if src is None or dim not in src:
                    continue
                # La relevancia NO existe para los metodos lexicos. Va como
                # NaN, nunca como 0: un 0 significaria "no hablo del tema",
                # que es justo la informacion que el lexico no tiene.
                relevancia = float("nan")
                if metodo == "nuevo" and spec["relevancia"]:
                    relevancia = float(n[dim + "_relevancia"])
                filas_valor.append(dict(
                    base,
                    metodo=metodo,
                    # Sobre que cadena se calculo esta fila. Sin esto, las dos
                    # vias lexicas serian indistinguibles en el marco.
                    origen_texto=METODOS_LEXICOS.get(metodo, "texto"),
                    dimension=dim,
                    escala=spec["escala"],
                    grupo=spec["grupo"],
                    valencia=float(src[dim]),
                    relevancia=relevancia,
                    engagement_weight=float(n["engagement_weight"]),
                    label_sentimiento=n["_label"],
                ))

        # --- probabilidades crudas, por grupo de llamada NLI ---------------
        # Un grupo puede producir mas de una dimension: 'violencia' tiene
        # cuatro etiquetas y de el salen violence_salience y safety_perceived.
        for grupo, probs in n["_crudo"].items():
            for etiqueta, p in probs.items():
                filas_prob.append({
                    "texto_id": base["texto_id"], "frase": base["frase"],
                    "lang": base["lang"], "lang_grupo": base["lang_grupo"],
                    "estrato": base["estrato"], "grupo": grupo,
                    "etiqueta": etiqueta, "probabilidad": float(p),
                })

        # --- palabras del lexico activadas ---------------------------------
        for metodo in METODOS_LEXICOS:
            src = r.get(metodo)
            if src is None:
                continue
            tl = (texto if metodo == "actual" else m["traduccion"]).lower()
            for dim, palabras in src["_palabras"].items():
                pos_set, neg_set = SETS_LEXICO[dim]
                for w in palabras:
                    filas_lex.append({
                        "texto_id": base["texto_id"], "frase": base["frase"],
                        "lang": base["lang"], "lang_grupo": base["lang_grupo"],
                        "metodo": metodo, "dimension": dim, "palabra": w,
                        "polaridad": "pos" if w in pos_set else "neg",
                        "es_subcadena": es_subcadena(w, tl),
                    })

    largo = pd.DataFrame(filas_valor)
    probs = pd.DataFrame(filas_prob)
    lexico = pd.DataFrame(filas_lex, columns=["texto_id", "frase", "lang",
                                              "lang_grupo", "metodo",
                                              "dimension", "palabra",
                                              "polaridad", "es_subcadena"])
    return largo, probs, lexico


LARGO, PROBS, LEXICO = construir_frames(resultados, TUITS, META)

# ---------------------------------------------------------------------------
# GUARDIA: valencia y relevancia no se combinan nunca en un escalar.
#
# El producto valencia * relevancia es tentador porque reduce todo a un eje,
# y reconstruye exactamente el defecto que el rediseno elimina:
#
#   hablo y juzgo neutro   0.02 * 0.91 = 0.018
#   no hablo del tema      0.60 * 0.03 = 0.018
#
# El mismo numero para dos cosas distintas: el 0.0000 ambiguo con otro nombre.
# Esta asercion falla si alguien agrega cualquier columna numerica nueva, que
# es la via por la que ese producto entraria sin que nadie lo note.
# ---------------------------------------------------------------------------
COLUMNAS_NUMERICAS_PERMITIDAS = {"valencia", "relevancia", "engagement_weight"}
_num = set(LARGO.select_dtypes("number").columns)
assert _num == COLUMNAS_NUMERICAS_PERMITIDAS, (
    "LARGO tiene columnas numericas inesperadas: {}. Si alguna combina valencia "
    "y relevancia en un solo numero, quitala: ese producto reintroduce el cero "
    "ambiguo.".format(sorted(_num - COLUMNAS_NUMERICAS_PERMITIDAS))
)

# ---------------------------------------------------------------------------
# IDIOMAS DEL CORPUS. Se derivan de los datos, no de una lista declarada.
#
# El notebook asumia seis grupos fijos. El corpus real tiene 23 valores de
# lang y CERO en aleman, neerlandes y arabe; el 16% es japones, un idioma que
# el diseno nunca contemplo. Declarar los grupos inventaria los vacios y
# esconderia el japones.
# ---------------------------------------------------------------------------
IDIOMAS_OBSERVADOS = (LARGO[["texto_id", "lang_grupo"]].drop_duplicates()
                      ["lang_grupo"].value_counts().to_dict()
                      if len(LARGO) and LARGO["lang_grupo"].notna().any() else {})
IDIOMAS_COMPARABLES = [l for l, n in IDIOMAS_OBSERVADOS.items() if n >= N_MINIMO_IDIOMA]
IDIOMAS_ESCASOS = {l: n for l, n in IDIOMAS_OBSERVADOS.items() if n < N_MINIMO_IDIOMA}

# Idiomas que el lexico no puede ver: ninguna de sus palabras esta en ese
# idioma. Sobre la bateria eso era una hipotesis; sobre el corpus es un numero.
_con_lexico = set(LEXICO["lang_grupo"].dropna().unique()) if len(LEXICO) else set()
IDIOMAS_SIN_COBERTURA = {l: n for l, n in IDIOMAS_OBSERVADOS.items()
                         if l not in _con_lexico}

print("=" * 74)
print("MARCOS CONSTRUIDOS")
print("=" * 74)
print("  LARGO   {:>6} filas   {} textos x {} dimensiones x metodo".format(
    len(LARGO), LARGO["texto_id"].nunique(), LARGO["dimension"].nunique()))
print("  PROBS   {:>6} filas   {} grupos de hipotesis".format(
    len(PROBS), PROBS["grupo"].nunique()))
print("  LEXICO  {:>6} filas   palabras del lexico activadas".format(len(LEXICO)))
print()
print("  metodos en el marco:", sorted(LARGO["metodo"].unique()))
for _m in sorted(LARGO["metodo"].unique()):
    _sub = LARGO[LARGO.metodo == _m]
    print("     {:<10} sobre {:<10} {} filas, {} dimensiones".format(
        _m, _sub["origen_texto"].iloc[0], len(_sub), _sub["dimension"].nunique()))
print()
_solo_uno = [d for d, s in ESCALAS.items() if len(s["metodos"]) == 1]
print("  dimensiones con un solo metodo (salen de las figuras comparativas):")
for _d in _solo_uno:
    print("     -", _d)
print()
_sin_rel = LARGO[(LARGO.metodo != "nuevo") & LARGO.relevancia.isna()]
print("  relevancia no disponible en {} filas de metodos lexicos (NaN, no 0)".format(
    len(_sin_rel)))
if len(LEXICO):
    _sub = int(LEXICO["es_subcadena"].sum())
    print("  emparejamientos lexicos: {} totales, {} por subcadena".format(
        len(LEXICO), _sub))
else:
    print("  emparejamientos lexicos: ninguno en este corpus")

if IDIOMAS_OBSERVADOS:
    print()
    print("  idiomas observados ({}):".format(len(IDIOMAS_OBSERVADOS)))
    for _l, _n in list(IDIOMAS_OBSERVADOS.items())[:12]:
        _marca = "" if _n >= N_MINIMO_IDIOMA else "  (bajo N_MINIMO_IDIOMA, se excluye)"
        print("     {:<12} {:>5} textos{}".format(_l, _n, _marca))
    if IDIOMAS_SIN_COBERTURA:
        _tot = sum(IDIOMAS_SIN_COBERTURA.values())
        print()
        print("  COBERTURA LEXICA NULA en {}: {} textos ({:.0%} del total).".format(
            sorted(IDIOMAS_SIN_COBERTURA), _tot, _tot / max(sum(IDIOMAS_OBSERVADOS.values()), 1)))
        print("  Para esos textos las dimensiones lexicas valen lo mismo POR")
        print("  CONSTRUCCION: no hay una sola palabra de esos idiomas en las listas.")

_estratos = LARGO["estrato"].dropna().nunique() if len(LARGO) else 0
if _estratos:
    print()
    print("  estratos de recoleccion: {} distintos".format(_estratos))
    for _e, _n in (LARGO[["texto_id", "estrato"]].drop_duplicates()
                   ["estrato"].value_counts().head(8).items()):
        print("     {:<42} {:>5} textos".format(str(_e)[:42], _n))
print("=" * 74)

## 11. La demostración del sesgo entre idiomas

Esta sección es el argumento central, y conviene que vaya en la tesis.

Abajo está **la misma frase** —contenido idéntico, sentido idéntico— escrita en los seis
idiomas de la batería: *la policía era corrupta y las autoridades fallaron*. Un instrumento
de medición válido debería darles aproximadamente el mismo valor, y **negativo**.

No los da. El método actual sale **positivo en inglés y en francés**, negativo en español y
portugués, y exactamente cero en alemán y árabe. La causa en inglés es fácil de ver en la
columna de palabras: `police` y `authorities` están en la lista **positiva** de `analyzer.py`,
así que un texto que denuncia a la policía suma dos positivas contra una negativa.

Eso no es imprecisión entre idiomas. Es la conclusión contraria según el idioma, y por eso
la celda lo señala aparte de la dispersión.

**Un detalle que hace fuerte a esta demostración:** `perspectiva_politica = clamp(mg × 0.8)`
no incorpora el sentimiento, así que su valor no depende de ningún modelo. Es aritmética
sobre listas de palabras. El resultado es exacto, reproducible sin GPU y citable como se
cita una cuenta, no como se cita un experimento.

Compara la columna ACTUAL contra la NUEVO, y mira también `relevancia`: es la información
que hoy no existe.

In [ ]:
# ===========================================================================
# LA MISMA FRASE EN SEIS IDIOMAS
#
# Son los mismos strings que BATERIA["politica_neg"], a proposito: esta celda
# imprime la dispersion y la celda de figuras la recalcula desde la bateria.
# La figura comprueba que coinciden y falla si alguien edita una y no la otra.
#
# Con CARGAR_ZEROSHOT no hay modelos: las filas se restauran del volcado, que
# la seccion 17 completa con ellas. Si el volcado no las trae (la corrida
# original no llego a la seccion 17), la demostracion se omite con aviso en
# vez de detener el notebook.
# ===========================================================================
MISMA_FRASE = [
    ("en", "The police were corrupt and did nothing, the authorities failed us completely."),
    ("es", "La policía fue corrupta y no hizo nada, las autoridades nos fallaron por completo."),
    ("pt", "A polícia era corrupta e não fez nada, as autoridades nos falharam completamente."),
    ("fr", "La police était corrompue et n'a rien fait, les autorités nous ont complètement laissés tomber."),
    ("de", "Die Polizei war korrupt und tat nichts, die Behörden haben uns völlig im Stich gelassen."),
    ("ar", "كانت الشرطة فاسدة ولم تفعل شيئًا، وخذلتنا السلطات تمامًا."),
]

DIMENSION_DEMO = "perspectiva_politica"

filas_demo = []
if CARGAR_ZEROSHOT:
    filas_demo = list((VOLCADO_PREVIO or {}).get("filas_demo") or [])
    if not filas_demo:
        print("El volcado no trae la demostracion entre idiomas (la corrida original no")
        print("llego a la seccion 17). Se omite; ejecutala sin CARGAR_ZEROSHOT para verla.")
else:
    _textos_demo = [f for _, f in MISMA_FRASE]
    _nuevos_demo = metodo_nuevo_lote(_textos_demo)
    for _i, (lang, frase) in enumerate(MISMA_FRASE):
        a = metodo_actual(frase)
        n = _nuevos_demo[_i]
        filas_demo.append({"lang": lang, "actual": a[DIMENSION_DEMO],
                           "nuevo": n[DIMENSION_DEMO],
                           "relevancia": n[DIMENSION_DEMO + "_relevancia"],
                           "palabras": a["_palabras"][DIMENSION_DEMO]})

if filas_demo:
    print("MISMA FRASE, SEIS IDIOMAS -- dimension '{}'{}".format(
        DIMENSION_DEMO, "  (restaurada del volcado)" if CARGAR_ZEROSHOT else ""))
    print()
    print("Contenido identico en las seis: la policia era corrupta y las autoridades")
    print("fallaron. Un instrumento valido deberia darles el mismo valor, y NEGATIVO.")
    print()
    print("  {:<5} {:>10} {:>10} {:>12}   {}".format(
        "idioma", "ACTUAL", "NUEVO", "relevancia", "palabras del lexico"))
    print("  " + "-" * 76)
    for f in filas_demo:
        print("  {:<5} {:>+10.4f} {:>+10.4f} {:>12.2f}   {}".format(
            f["lang"], f["actual"], f["nuevo"], f["relevancia"],
            f["palabras"] if f["palabras"] else "(ninguna)"))

    def _disp(vals):
        m = sum(vals) / len(vals)
        return max(vals) - min(vals), (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5

    _va = [f["actual"] for f in filas_demo]
    _vn = [f["nuevo"] for f in filas_demo]
    _ra, _sa = _disp(_va)
    _rn, _sn = _disp(_vn)

    print()
    print("  DISPERSION ENTRE IDIOMAS -- el numero que se reporta.")
    print("  El contenido no varia, asi que toda variacion es del instrumento.")
    print("  {:<10} rango {:>7.4f}   desv.est. {:>7.4f}".format("ACTUAL", _ra, _sa))
    print("  {:<10} rango {:>7.4f}   desv.est. {:>7.4f}".format("NUEVO", _rn, _sn))

    # -----------------------------------------------------------------------
    # INVERSION DE SIGNO. Es un resultado mas fuerte que la dispersion y merece
    # senalarse aparte: no es que el instrumento sea impreciso entre idiomas,
    # es que llega a la conclusion CONTRARIA segun el idioma.
    #
    # Ojo a la causa en ingles: 'police' y 'authorities' estan en la lista
    # POSITIVA de analyzer.py, asi que un texto que dice que la policia era
    # corrupta suma dos positivas y una negativa, y sale positivo.
    #
    # Este valor es EXACTO: perspectiva_politica = clamp(mg * 0.8) no
    # incorpora el sentimiento, asi que no depende de ninguna corrida ni de
    # ninguna GPU. Se puede citar en la tesis como se cita una cuenta.
    # -----------------------------------------------------------------------
    def _signo(v, eps=1e-9):
        return 0 if abs(v) < eps else (1 if v > 0 else -1)

    for _etiq, _vals in (("ACTUAL", _va), ("NUEVO", _vn)):
        _pos = [f["lang"] for f, v in zip(filas_demo, _vals) if _signo(v) > 0]
        _neg = [f["lang"] for f, v in zip(filas_demo, _vals) if _signo(v) < 0]
        _cero = [f["lang"] for f, v in zip(filas_demo, _vals) if _signo(v) == 0]
        if _pos and _neg:
            print()
            print("  " + "!" * 72)
            print("  INVERSION DE SIGNO en {}: el mismo contenido sale POSITIVO en {}".format(
                _etiq, ", ".join(_pos)))
            print("  y NEGATIVO en {}{}.".format(
                ", ".join(_neg),
                ", y exactamente cero en " + ", ".join(_cero) if _cero else ""))
            print("  No es imprecision entre idiomas: es la conclusion contraria.")
            print("  " + "!" * 72)
        elif _cero and (_pos or _neg):
            print()
            print("  {}: {} idioma(s) dan exactamente cero ({}), que aqui significa".format(
                _etiq, len(_cero), ", ".join(_cero)))
            print("  'el lexico no encontro ninguna palabra', no 'juicio neutro'.")

    _sin_palabras = [f["lang"] for f in filas_demo if not f["palabras"]]
    if _sin_palabras:
        print()
        print("  Sin ninguna palabra del lexico activada: {}. Para esos idiomas el".format(
            ", ".join(_sin_palabras)))
        print("  metodo actual no puede dar otra cosa que cero, pase lo que pase el texto.")

## 12. Tabla resumen

In [ ]:
import pandas as pd

# La tabla sale de LARGO, no de `resultados`. Antes se reconstruia a mano y en
# el camino perdia violence_salience, safety_perceived, el sentimiento y las
# probabilidades crudas. Ademas dependia de `filas_demo`, que nace en la
# seccion de demostracion entre idiomas: ejecutar esta celda sin haber corrido
# aquella reventaba con NameError. Ya no.

_corto = LARGO["texto"].str.slice(0, 38) + LARGO["texto"].str.len().gt(38).map({True: "...", False: ""})
_L = LARGO.assign(tuit=_corto)

# --- comparacion metodo actual contra rediseno -----------------------------
# Comparable = la produce el lexico Y el rediseno. Se pregunta por los metodos
# concretos y no por "cuantos hay": con la via traducida activa una dimension
# puede tener tres, y contar dejaria la tabla vacia sin avisar.
_comparables = [d for d, s in ESCALAS.items()
                if "actual" in s["metodos"] and "nuevo" in s["metodos"]]
_metodos_tabla = [m for m in ("actual", "traducido", "nuevo")
                  if m in set(LARGO["metodo"])]

# Los atributos por texto se unen DESPUES del pivote, no se meten en su indice:
# pivot_table descarta los grupos cuya clave es NaN, y con textos propios `lang`
# viene vacio. Meterlo en el indice vaciaria la tabla entera.
_attrs = _L[["texto_id", "tuit", "lang"]].drop_duplicates(subset=["texto_id"])

_piv = (_L[_L.dimension.isin(_comparables)]
        .pivot_table(index=["texto_id", "dimension"],
                     columns="metodo", values="valencia", aggfunc="first")
        .reset_index())

_rel = (_L[(_L.metodo == "nuevo")][["texto_id", "dimension", "relevancia"]]
        .drop_duplicates(subset=["texto_id", "dimension"]))

tabla = (_piv.merge(_attrs, on="texto_id", how="left")
             .merge(_rel, on=["texto_id", "dimension"], how="left"))
tabla["diferencia"] = tabla["nuevo"] - tabla["actual"]
_cols = ["tuit", "lang", "dimension"] + _metodos_tabla + ["relevancia", "diferencia"]
tabla = tabla[[c for c in _cols if c in tabla.columns]]
tabla = tabla.round({m: 3 for m in _metodos_tabla})
tabla = tabla.round({"relevancia": 2, "diferencia": 3})

print("COMPARACION ENTRE METODOS  (dimensiones que el lexico y el rediseno producen)")
if "traducido" in _metodos_tabla:
    print("  'traducido' es el MISMO lexico sobre la columna de traduccion. Mide el")
    print("  lexico junto con la calidad de una traduccion automatica: si gana, no")
    print("  se sabe cuanto merito es de cada uno.")
display(tabla)

# --- dimensiones que solo produce el rediseno ------------------------------
_solo_nuevo = [d for d, s in ESCALAS.items() if s["metodos"] == ("nuevo",)]
if _solo_nuevo:
    print()
    print("DIMENSIONES QUE SOLO EXISTEN EN EL REDISENO")
    print("(el metodo actual no las produce: no hay columna con que comparar.")
    print(" imagen_cultural esta aqui porque analyzer.py no tiene lista de")
    print(" palabras equivalente, no porque falten datos.)")
    _extra = (_L[_L.dimension.isin(_solo_nuevo)]
              .pivot_table(index="texto_id", columns="dimension",
                           values="valencia", aggfunc="first")
              .reset_index()
              .merge(_attrs, on="texto_id", how="left")
              .drop(columns=["texto_id"])
              .round(3))
    display(_extra)

# --- la seccion 11 sigue imprimiendo lo suyo; aqui solo se repite si existe --
if "filas_demo" in globals():
    print()
    print("Dispersion de la misma frase en 6 idiomas (seccion 11):")
    display(pd.DataFrame(filas_demo))

## 13. Figuras

Siete figuras, cada una respondiendo **una** pregunta, con su umbral y su alerta escrita —en
vez de dejar la conclusión al ojo.

| Figura | Pregunta |
|---|---|
| 1 · dispersión entre idiomas | ¿mide lo mismo cuando el contenido es el mismo? |
| 2 · valencia × relevancia | ¿cuántos de los ceros de hoy son «no habló»? |
| 3 · valores posibles | ¿qué valores puede tomar cada método? |
| 4 · ausencia por dimensión | ¿esta dimensión mide algo en este corpus? |
| 5 · descomposición `pos`/`neg`/`na` | ¿a dónde se va la probabilidad del modelo? |
| 6 · cobertura léxica | ¿en cuántos idiomas llega a activarse el léxico? |
| 7 · desglose por estrato | ¿de qué consulta viene cada número? |

**La ausencia nunca se apila con la escala.** `score` y `relevancia` son ejes ortogonales y
ninguna figura los colapsa. La banda de mudez se **marca**, no se filtra: dibujar esas
observaciones *es* la evidencia.

**Tres métodos, no dos.** Cuando el corpus trae traducción, el léxico corre sobre las dos
columnas y las figuras muestran `léxico (original)`, `léxico (traducido)` y `rediseño`. Se
distinguen por tono **y** por trama, para que sigan siendo legibles impresas en gris. Ojo al
leer la vía traducida: es el mismo léxico sobre otra entrada, así que mide el léxico **junto
con** la calidad de una traducción automática, y los dos efectos no se separan.

**`imagen_cultural` no aparece en las figuras 1, 3 ni 6.** No es que falten datos: no existe
lista de palabras equivalente en `analyzer.py`, así que no hay columna contra la cual
compararla. Las figuras lo dicen en vez de dibujar un hueco.

**Los idiomas salen de los datos.** El diseño anterior declaraba seis grupos fijos; el corpus
real tiene 23 valores de `lang`, cero en alemán, neerlandés y árabe, y un 16% en japonés que
nadie había contemplado. Los idiomas con menos de `N_MINIMO_IDIOMA` observaciones se nombran
y se excluyen, en vez de dibujarse como si fueran comparables.

**El rótulo de procedencia tiene tres estados.** Batería, textos propios y corpus. El del
corpus es el que importa: no dice «no reportable» sino **descriptivo pero no validado**, que
es más fácil de pasar por alto y más caro de olvidar.

**Salida doble.** Las figuras se muestran aquí abajo y además se escriben en un archivo HTML
autocontenido, que es el artefacto que sobrevive al cierre del notebook y va a la tesis. Cada
figura se exporta como SVG desde su barra de herramientas.

In [ ]:
import json
import datetime as _dt
import math
import pandas as pd
from IPython.display import HTML, display

# ===========================================================================
# PREPARACION DE DATOS
#
# Todo se calcula aqui, una vez, en Python. La pagina recibe numeros ya
# agregados y no necesita el kernel para redibujarse: por eso el archivo
# exportado sigue funcionando fuera del notebook.
#
# Las combinaciones sin observaciones se OMITEN, no se emiten como cero.
# Un cero significa "medi y dio cero"; la ausencia es otra cosa.
# ===========================================================================

PALETA = {
    # slots categoricos 1 y 2 del sistema de referencia; par adyacente
    # validado (CVD dE 24.7, vision normal dE 33.6, ambos >= umbral).
    "actual": "#2a78d6",
    "nuevo":  "#eb6834",
    # Tercer metodo: el MISMO lexico sobre la traduccion. Es hermano de
    # "actual", no un metodo independiente, asi que va como tinte claro del
    # mismo azul en vez de como un tercer matiz suelto. En escala de grises se
    # separa por luminosidad, y ademas lleva trama diagonal en la figura.
    "traducido": "#8fb8e8",
    # neutro deliberado para la AUSENCIA: falla el piso de croma del
    # validador a proposito, porque aqui "leerse gris" es el significado.
    "na":     "#898781",
    "neg":    "#e34948",
    "medio":  "#eda100",
    "tinta":  "#0b0b0b",
    "tinta2": "#52514e",
    "mudo":   "#898781",
    "grid":   "#e1e0d9",
    "eje":    "#c3c2b7",
    "sup":    "#fcfcfb",
    "alerta": "#d03b3b",
    "ok":     "#0ca30c",
}

_ALERTAS = []


def _alerta(texto):
    _ALERTAS.append(texto)


# --- comprobacion de coherencia con la seccion 11 ---------------------------
# La bateria reproduce textualmente las seis versiones de MISMA_FRASE. Si
# alguien edita una y no la otra, la figura de dispersion y lo que imprime la
# seccion 11 dejarian de coincidir en silencio. Aqui no.
if ES_BATERIA and "MISMA_FRASE" in globals():
    _b = BATERIA["politica_neg"]["textos"]
    _d = dict(MISMA_FRASE)
    _difieren = [k for k in _d if _b.get(k) != _d[k]]
    assert not _difieren, (
        "Los textos de politica_neg de la BATERIA y de MISMA_FRASE difieren en "
        "{}. Son los mismos a proposito: la seccion 11 imprime su dispersion y "
        "la figura la recalcula. Sincronizalos.".format(_difieren)
    )


def _stats(valores):
    v = [x for x in valores if x is not None and not (isinstance(x, float) and math.isnan(x))]
    if not v:
        return None
    media = sum(v) / len(v)
    var = sum((x - media) ** 2 for x in v) / len(v)
    return {"n": len(v), "rango": max(v) - min(v), "sigma": var ** 0.5, "media": media}


def preparar_datos():
    d = {
        "es_bateria": bool(ES_BATERIA),
        # Rotulo de TRES estados. El del corpus no dice "no reportable" sino
        # algo distinto y mas facil de pasar por alto: descriptivo pero NO
        # VALIDADO, porque no hay codificacion humana contra la cual contrastar.
        "procedencia": PROCEDENCIA,
        "rotulo": {
            "bateria": "BATERIA DE PRUEBA DEL INSTRUMENTO. No es una muestra de "
                       "nada: sus valores no son reportables como hallazgo sobre "
                       "el corpus.",
            "propios": "Textos aportados a mano. Ni bateria ni corpus: lo que "
                       "digan las figuras vale para estos textos y nada mas.",
            "corpus":  "CORPUS REAL. Las figuras son DESCRIPTIVAS. La seccion 14 "
                       "las contrasta contra la rubrica de EvaluadorTweets, que "
                       "tambien es un modelo de lenguaje: NO hay validacion contra "
                       "codificacion humana.",
        }[PROCEDENCIA],
        "n_textos": int(LARGO["texto_id"].nunique()),
        "n_minimo": N_MINIMO_FIGURA,
        "umbral_mudez": UMBRAL_MUDEZ,
        "umbral_relevancia": UMBRAL_RELEVANCIA,
        "diseno_relevancia": DISENO_RELEVANCIA,
        "umbral_dim_muda": UMBRAL_DIMENSION_MUDA,
        "paleta": PALETA,
        # A/B del ancla: la evidencia de la decision de diseno de la seccion 3.
        # None cuando no se corrio, y la pagina lo dice en vez de callarlo.
        "ancla": globals().get("COMPARACION_ANCLA"),
        "ancla_en_etiquetas": bool(ANCLA_EN_ETIQUETAS),
    }

    # ---- ficha tecnica del instrumento ------------------------------------
    d["ficha"] = {
        "idioma_hipotesis": IDIOMA_HIPOTESIS,
        "plantilla": PLANTILLA,
        "modelo_sentimiento": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
        "modelo_nli": "joeddav/xlm-roberta-large-xnli",
        "fecha": _dt.datetime.now().strftime("%Y-%m-%d %H:%M"),
        "pasadas_por_texto": PASADAS_POR_TEXTO,
        "tiempo_corrida_s": round(TIEMPO_CORRIDA, 1) if "TIEMPO_CORRIDA" in globals() else None,
        "huella_instrumento": globals().get("HUELLA_INSTRUMENTO"),
        "corrida_restaurada": CARGAR_ZEROSHOT or None,
        "traduccion": TRADUCCION_POR if ES_BATERIA else None,
        "hipotesis": [
            {"grupo": g, "etiqueta": k, "frase": v}
            for g, t in list(DIMENSIONES.items())
            for k, v in t[IDIOMA_HIPOTESIS].items()
        ] + [
            {"grupo": "violencia", "etiqueta": k, "frase": v}
            for k, v in VIOLENCIA[IDIOMA_HIPOTESIS].items()
        ],
    }

    # ---- textos de la bateria, para que las traducciones sean auditables --
    d["bateria"] = []
    if ES_BATERIA:
        for frase, spec in BATERIA.items():
            for lang, texto in spec["textos"].items():
                d["bateria"].append({"frase": frase, "lang": lang, "texto": texto,
                                     "objetivo": spec["objetivo"],
                                     "que_prueba": spec["que_prueba"]})

    # Los idiomas se DERIVAN de los datos. El notebook asumia seis grupos
    # fijos; el corpus real tiene 23 valores de lang, cero en aleman,
    # neerlandes y arabe, y un 16% en japones que el diseno nunca contemplo.
    # Declararlos inventaria los vacios y esconderia el japones.
    idiomas = [l for l in IDIOMAS_COMPARABLES]
    d["idiomas"] = idiomas
    d["idiomas_n"] = IDIOMAS_OBSERVADOS
    d["idiomas_escasos"] = IDIOMAS_ESCASOS
    d["idiomas_sin_cobertura"] = IDIOMAS_SIN_COBERTURA
    d["n_minimo_idioma"] = N_MINIMO_IDIOMA
    if IDIOMAS_ESCASOS:
        _alerta("IDIOMAS CON MUY POCAS OBSERVACIONES, excluidos de las figuras "
                "que comparan idiomas: {}. Se informan para que la exclusion sea "
                "visible, no para dibujarlos como si fueran comparables."
                .format(", ".join("{} (n={})".format(k, v)
                                  for k, v in sorted(IDIOMAS_ESCASOS.items()))))

    # ---- F1 · dispersion entre idiomas ------------------------------------
    d["dispersion"] = {}
    for dim in DIMS_CON_RELEVANCIA:
        sub = LARGO[(LARGO.dimension == dim) & LARGO.lang.notna()]
        if sub.empty or not idiomas:
            continue
        # Los metodos se leen de ESCALAS. Con la via traducida activa una
        # dimension puede tener tres, y fijar dos aqui la dejaria fuera.
        metodos = [m for m in ESCALAS[dim]["metodos"] if m in set(sub["metodo"])]
        filas = {}
        for metodo in metodos:
            s = sub[sub.metodo == metodo]
            # una frase por idioma: la que apunta a esta dimension si existe
            objetivo = s[s.objetivo == dim]
            s = objetivo if not objetivo.empty else s
            por_lang = s.groupby("lang_grupo")["valencia"].mean()
            filas[metodo] = [round(float(por_lang.get(l)), 4)
                             for l in idiomas if l in por_lang.index]
        langs = [l for l in idiomas
                 if l in sub.groupby("lang_grupo")["valencia"].mean().index]
        if not langs or len(metodos) < 2:
            continue
        d["dispersion"][dim] = {
            "langs": langs,
            "metodos": metodos,
            "series": {m: filas[m] for m in metodos},
            # se conservan las dos claves antiguas para no romper nada que las lea
            "actual": filas.get("actual", []),
            "nuevo": filas.get("nuevo", []),
            "stats": {m: _stats(filas[m]) for m in metodos},
        }
    # dimensiones que no pueden aparecer aqui, con su razon
    d["dispersion_excluidas"] = [
        {"dimension": dim,
         "razon": "solo la produce el rediseno: analyzer.py no tiene lista de "
                  "palabras equivalente, asi que no hay columna contra la cual "
                  "compararla. No es que falten datos."}
        for dim, sp in ESCALAS.items()
        if sp["metodos"] == ("nuevo",) and sp["relevancia"]
    ]

    # ---- F2 · plano valencia x relevancia ---------------------------------
    d["plano"] = {}
    for dim in DIMS_CON_RELEVANCIA:
        sub = LARGO[(LARGO.dimension == dim) & (LARGO.metodo == "nuevo")]
        puntos = {}
        for _, r in sub.iterrows():
            if pd.isna(r["relevancia"]):
                continue
            puntos.setdefault(r["frase"] or "sin frase", []).append({
                "x": round(float(r["relevancia"]), 4),
                "y": round(float(r["valencia"]), 4),
                "lang": r["lang"], "texto": r["texto"][:120],
            })
        if not puntos:
            continue
        mudos = int(sum(1 for g in puntos.values() for p in g
                        if p["x"] < UMBRAL_MUDEZ))
        d["plano"][dim] = {"grupos": puntos, "n": sum(len(g) for g in puntos.values()),
                           "mudos": mudos}
    # dimensiones excluidas de esta figura, con su razon
    d["plano_excluidas"] = [
        {"dimension": dim, "razon": "no produce relevancia: sale del grupo '{}', "
                                    "que no tiene hipotesis de irrelevancia propia "
                                    "por dimension".format(ESCALAS[dim]["grupo"])}
        for dim in ESCALAS if not ESCALAS[dim]["relevancia"] and dim != "overall_sentiment"
    ]

    # ---- F3 · valores posibles de cada metodo -----------------------------
    d["valores"] = {}
    # Comparable = la producen el lexico Y el rediseno. Se pregunta por los
    # metodos concretos y no por cuantos hay: con la via traducida una
    # dimension puede tener tres y el conteo la dejaria fuera en silencio.
    comparables = [dim for dim, sp in ESCALAS.items()
                   if "actual" in sp["metodos"] and "nuevo" in sp["metodos"]
                   and dim != "overall_sentiment"]
    for dim in comparables:
        sub = LARGO[LARGO.dimension == dim]
        # El pivote va solo sobre texto_id: pivot_table descarta los grupos cuya
        # clave es NaN, y con textos propios `lang` viene vacio.
        attrs = sub[["texto_id", "lang", "texto"]].drop_duplicates("texto_id")
        piv = (sub.pivot_table(index="texto_id", columns="metodo",
                               values="valencia", aggfunc="first")
                  .reset_index()
                  .merge(attrs, on="texto_id", how="left"))
        if "actual" not in piv or "nuevo" not in piv:
            continue
        d["valores"][dim] = {
            "puntos": [{"x": round(float(r["actual"]), 4),
                        "y": round(float(r["nuevo"]), 4),
                        "lang": r["lang"], "texto": r["texto"][:120]}
                       for _, r in piv.iterrows()],
            "escalonada": dim in ESCALONADAS,
            "marcas": ESCALONADAS.get(dim, []),
            "formula": ("clamp(mg * 0.8), con mg = clamp((pos - neg) * 0.2). "
                        "No incorpora el sentimiento: solo puede tomar {} valores."
                        ).format(len(ESCALONADAS[dim])) if dim in ESCALONADAS else
                       ("incorpora el sentimiento, que es continuo: no es escalonada, "
                        "es una nube desplazada a saltos"),
        }

    # ---- F4 · distribucion de relevancia y ausencia ------------------------
    d["ausencia"] = {}
    bordes = [i / 10 for i in range(11)]
    for dim in DIMS_CON_RELEVANCIA:
        sub = LARGO[(LARGO.dimension == dim) & (LARGO.metodo == "nuevo")]
        vals = sub["relevancia"].dropna().tolist()
        if not vals:
            continue
        conteo = [0] * 10
        for v in vals:
            idx = min(int(v * 10), 9)
            conteo[idx] += 1
        prop_muda = sum(1 for v in vals if v < UMBRAL_MUDEZ) / len(vals)
        d["ausencia"][dim] = {
            "bordes": bordes, "conteo": conteo, "n": len(vals),
            "prop_muda": round(prop_muda, 4),
        }
        if prop_muda > UMBRAL_DIMENSION_MUDA:
            _alerta("[{}] DIMENSION MAYORITARIAMENTE MUDA ({:.0%} bajo relevancia {}): "
                    "apenas mide nada en este corpus. Su promedio esta dominado por "
                    "textos que no hablaron del tema."
                    .format(dim, prop_muda, UMBRAL_MUDEZ))

    # lectura del control mudo, que es la prueba del eje de relevancia
    d["control_mudo"] = None
    if ES_BATERIA:
        cm = LARGO[(LARGO.frase == "control_logistico") & (LARGO.metodo == "nuevo")
                   & LARGO.dimension.isin(DIMS_CON_RELEVANCIA)]
        if not cm.empty:
            d["control_mudo"] = [
                {"dimension": dim,
                 "relevancia_media": round(float(g["relevancia"].mean()), 4),
                 "valencia_media": round(float(g["valencia"].mean()), 4)}
                for dim, g in cm.groupby("dimension")
            ]

    # ---- F5 · descomposicion de probabilidades ----------------------------
    d["probs"] = {}
    for grupo, g in PROBS.groupby("grupo"):
        etiquetas = g["etiqueta"].unique().tolist()
        clave = "frase" if ES_BATERIA and g["frase"].notna().all() else "texto_id"
        medias = g.pivot_table(index=clave, columns="etiqueta",
                               values="probabilidad", aggfunc="mean")
        d["probs"][grupo] = {
            "etiquetas": etiquetas,
            "filas": medias.index.tolist(),
            "valores": {e: [round(float(medias.loc[i, e]), 4) for i in medias.index]
                        for e in etiquetas if e in medias.columns},
            "agregado_por": clave,
        }
        # la alerta se calcula por observacion, no sobre las medias
        na_lab = "na" if "na" in etiquetas else ("neutral" if "neutral" in etiquetas else None)
        if na_lab:
            por_texto = g.pivot_table(index="texto_id", columns="etiqueta",
                                      values="probabilidad", aggfunc="first")
            if na_lab in por_texto.columns:
                domina = (por_texto[na_lab] > 0.5).mean()
                if domina > 0.8:
                    _alerta("[{}] LA HIPOTESIS DE IRRELEVANCIA DOMINA en {:.0%} de las "
                            "observaciones. Indicio de que las frases de este grupo "
                            "estan mal calibradas, no necesariamente de que nadie "
                            "hable del tema.".format(grupo, domina))

    # ---- F6 · cobertura del lexico por idioma -----------------------------
    # Se calcula siempre. Sin idiomas declarados el desglose cae en un solo
    # cubo: la cobertura y, sobre todo, el emparejamiento por subcadena siguen
    # siendo medibles, y es justo con tuits reales cuando mas importan.
    d["lexico"] = None
    if True:
        por_idioma = bool(LARGO["lang_grupo"].notna().any())
        textos = LARGO[["texto_id", "lang_grupo"]].drop_duplicates()
        textos = textos.rename(columns={"lang_grupo": "lang"})
        textos = textos.assign(
            lang=textos["lang"].fillna("sin idioma declarado")
            if not por_idioma else textos["lang"])
        filas = []
        for lang, g in textos.groupby("lang"):
            ids = set(g["texto_id"])
            sub = LEXICO[LEXICO.texto_id.isin(ids)] if len(LEXICO) else LEXICO
            sub = sub[sub.metodo == "actual"] if len(sub) else sub
            con_match = sub["texto_id"].nunique() if len(sub) else 0
            filas.append({
                "lang": lang, "n_textos": len(ids),
                "cobertura": round(con_match / len(ids), 4) if len(ids) else 0.0,
                "matches": int(len(sub)),
                "subcadenas": int(sub["es_subcadena"].sum()) if len(sub) else 0,
            })
        total_m = sum(f["matches"] for f in filas)
        total_s = sum(f["subcadenas"] for f in filas)
        ejemplos = []
        if len(LEXICO):
            for _, r in LEXICO[LEXICO.es_subcadena].head(8).iterrows():
                t = LARGO[LARGO.texto_id == r["texto_id"]]["texto"]
                ejemplos.append({"palabra": r["palabra"], "dimension": r["dimension"],
                                 "lang": r["lang"],
                                 "texto": (t.iloc[0][:110] if len(t) else "")})
        d["lexico"] = {"filas": filas, "total_matches": total_m,
                       "total_subcadenas": total_s, "ejemplos": ejemplos,
                       "medido": total_m > 0, "por_idioma": por_idioma}
        sin_cobertura = [f["lang"] for f in filas if f["cobertura"] == 0]
        if sin_cobertura and por_idioma:
            _alerta("COBERTURA LEXICA NULA en {}: en esos idiomas el lexico no activa "
                    "ninguna palabra, asi que sus {} dimensiones lexicas valen lo "
                    "mismo por construccion. No es una medicion, es el instrumento."
                    .format(", ".join(sin_cobertura), len(SETS_LEXICO)))

    # ---- estratos de recoleccion ------------------------------------------
    # Un promedio sobre un corpus dirigido por consulta mide sobre todo la
    # consulta. Se desglosa, y se avisa cuando el agregado mezcla estratos que
    # buscaban expresamente la dimension que se esta promediando.
    d["estratos"] = None
    _tiene_estrato = len(LARGO) and LARGO["estrato"].notna().any()
    if _tiene_estrato:
        _textos_estrato = LARGO[["texto_id", "estrato"]].drop_duplicates()
        _n_por_estrato = _textos_estrato["estrato"].value_counts().to_dict()
        _total = sum(_n_por_estrato.values())
        filas_e = []
        _nuevo = LARGO[LARGO.metodo == "nuevo"]
        for est, n_e in sorted(_n_por_estrato.items(), key=lambda p: -p[1]):
            g = _nuevo[_nuevo.estrato == est]
            por_dim = {}
            for dim, gd in g.groupby("dimension"):
                if dim == "overall_sentiment":
                    continue
                por_dim[dim] = {
                    "valencia": round(float(gd["valencia"].mean()), 4),
                    "relevancia": (round(float(gd["relevancia"].mean()), 4)
                                   if gd["relevancia"].notna().any() else None),
                    "n": int(gd["texto_id"].nunique()),
                }
            filas_e.append({"estrato": est, "n": int(n_e),
                            "proporcion": round(n_e / _total, 4),
                            "dimensiones": por_dim})
        d["estratos"] = {"filas": filas_e, "n_estratos": len(filas_e),
                         "dirigidas": CONSULTAS_DIRIGIDAS}

        for dim, claves in CONSULTAS_DIRIGIDAS.items():
            if dim not in set(LARGO["dimension"]):
                continue
            dirigidos = [e for e in _n_por_estrato
                         if any(k in str(e).lower() for k in claves)]
            if not dirigidos:
                continue
            prop = sum(_n_por_estrato[e] for e in dirigidos) / _total
            if 0 < prop < 1:
                _alerta("[{}] AGREGADO DIRIGIDO POR LA CONSULTA: el {:.0%} del "
                        "corpus viene de consultas que buscaban expresamente esta "
                        "dimension ({}). Su promedio sobre el corpus completo mide "
                        "la consulta tanto como el fenomeno, y NO es reportable "
                        "como hallazgo sobre la poblacion. Lee el desglose por "
                        "estrato.".format(dim, prop, ", ".join(sorted(dirigidos))))

    d["alertas"] = _ALERTAS
    return d


DATOS_FIGURAS = preparar_datos()
_payload = json.dumps(DATOS_FIGURAS, ensure_ascii=False)
print("conjunto embebido: {:.1f} KB".format(len(_payload.encode("utf-8")) / 1024))

In [ ]:
PLANTILLA_HTML = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Figuras sobre las dimensiones</title>
<script src="__CDN__"></script>
<style>
  :root{
    color-scheme: light;
    --sup:#fcfcfb; --plano:#f9f9f7; --tinta:#0b0b0b; --tinta2:#52514e;
    --mudo:#898781; --grid:#e1e0d9; --eje:#c3c2b7; --alerta:#d03b3b; --ok:#0ca30c;
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--plano);color:var(--tinta);
       font:14px/1.55 system-ui,-apple-system,"Segoe UI",sans-serif}
  .wrap{max-width:1080px;margin:0 auto;padding-block:28px;padding-left:16px;padding-right:16px}
  h1{font-size:22px;margin:0 0 4px}
  h2{font-size:16px;margin:34px 0 10px;padding-bottom:6px;border-bottom:1px solid var(--grid)}
  h3{font-size:15px;margin:0 0 2px}
  p{margin:6px 0}
  .sub{color:var(--tinta2);margin:0 0 18px}
  .fig{background:var(--sup);border:1px solid var(--grid);border-radius:8px;
       padding:16px;margin:14px 0}
  .preg{color:var(--tinta2);font-style:italic;margin:0 0 8px}
  .n{display:inline-block;font-size:11px;color:var(--tinta2);background:var(--plano);
     border:1px solid var(--grid);border-radius:99px;padding:1px 9px;margin:0 0 8px;
     font-variant-numeric:tabular-nums}
  .aviso{background:#fff6e8;border-left:3px solid #eda100;padding:8px 11px;
         font-size:13px;color:var(--tinta2);margin:8px 0}
  .nota{font-size:13px;color:var(--tinta2);margin-top:10px}
  .cifra{font-variant-numeric:tabular-nums;font-weight:600;color:var(--tinta)}
  .lienzo{width:100%}
  .banner{background:#fff6e8;border:1px solid #eda100;border-radius:8px;
          padding:12px 14px;margin:0 0 16px}
  .banner b{color:#8a5a00}
  .alertas{background:var(--sup);border:1px solid var(--alerta);border-radius:8px;
           padding:12px 14px;margin:14px 0}
  .alertas h3{color:var(--alerta)}
  .alertas li{margin:6px 0;color:var(--tinta2)}
  .bien{background:var(--sup);border:1px solid var(--grid);border-left:3px solid var(--ok);
        border-radius:8px;padding:10px 14px;margin:14px 0;color:var(--tinta2)}
  details{margin-top:10px}
  summary{cursor:pointer;font-size:13px;color:var(--tinta2)}
  table{border-collapse:collapse;width:100%;font-size:12px;margin-top:8px;
        font-variant-numeric:tabular-nums}
  th,td{border-bottom:1px solid var(--grid);padding:4px 7px;text-align:left;vertical-align:top}
  th{color:var(--tinta2);font-weight:600}
  td.num{text-align:right}
  .tabla-scroll{overflow-x:auto}
  .fallo{background:var(--sup);border:1px solid var(--alerta);border-radius:8px;padding:20px}
  code{background:var(--plano);border:1px solid var(--grid);border-radius:3px;padding:0 4px;
       font-size:12px}
  .rtl{direction:rtl;text-align:right;unicode-bidi:isolate}
</style>
</head>
<body>
<div class="wrap" id="raiz"><p>cargando…</p></div>
<script>
const DATOS = __DATOS__;
const raiz = document.getElementById('raiz');

function el(h){const t=document.createElement('template');t.innerHTML=h.trim();return t.content.firstChild;}
function add(h){raiz.appendChild(el(h));}
function esc(s){return String(s==null?'':s).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));}
function rtl(lang,s){return lang==='ar' ? '<span class="rtl">'+esc(s)+'</span>' : esc(s);}
function fix(v,n){return (v==null||isNaN(v))?'—':Number(v).toFixed(n==null?3:n);}

const P = DATOS.paleta;
/* Los tres métodos. 'traducido' es el mismo léxico sobre la columna de
   traducción; se distingue por trama además de por tono, para que siga
   siendo legible impreso en escala de grises. */
const NOMBRES_METODO = {actual:'léxico (original)', traducido:'léxico (traducido)',
                        nuevo:'rediseño zero-shot'};
function NOMBRE_METODO(m){return NOMBRES_METODO[m]||m;}
function COLOR_METODO(m){return m==='nuevo'?P.nuevo:(m==='traducido'?(P.traducido||P.mudo):P.actual);}
const TOOLBOX = {right:6,top:-2,feature:{saveAsImage:{type:'svg',title:'guardar SVG',
  name:'figura',backgroundColor:'#fcfcfb'}}};
const EJE_X = {axisLine:{lineStyle:{color:P.eje}},axisTick:{show:false},
  axisLabel:{color:P.tinta2,fontSize:11}};
const EJE_Y = {axisLine:{show:false},axisTick:{show:false},
  axisLabel:{color:P.tinta2,fontSize:11},
  splitLine:{lineStyle:{color:P.grid,width:1}}};

function init(id,opt){
  const dom=document.getElementById(id);
  const c=echarts.init(dom,null,{renderer:'svg'});
  c.setOption(opt);
  window.addEventListener('resize',()=>c.resize());
  return c;
}

function bloque(id,titulo,pregunta,n,nota,tablaHTML,alto){
  const bajo = (n!==null && n!==undefined && n < DATOS.n_minimo);
  add('<section class="fig">'
    + '<h3>'+titulo+'</h3>'
    + '<p class="preg">'+pregunta+'</p>'
    + ((n!==null&&n!==undefined)?'<p class="n">n = '+n+'</p>':'')
    + (bajo?'<p class="aviso"><b>n = '+n+', por debajo de '+DATOS.n_minimo+'.</b> '
        +'A esta escala la figura no es interpretable. Se dibuja para inspeccionar el '
        +'instrumento, no para reportar un resultado.</p>':'')
    + '<div id="'+id+'" class="lienzo" style="height:'+(alto||340)+'px"></div>'
    + (nota?'<p class="nota">'+nota+'</p>':'')
    + (tablaHTML?'<details><summary>ver los datos de esta figura</summary>'
        +'<div class="tabla-scroll">'+tablaHTML+'</div></details>':'')
    + '</section>');
}

function tabla(cols,filas){
  let h='<table><thead><tr>';
  cols.forEach(c=>h+='<th>'+esc(c)+'</th>');
  h+='</tr></thead><tbody>';
  filas.forEach(f=>{h+='<tr>';f.forEach(v=>{
    const num = (typeof v==='number');
    h+='<td'+(num?' class="num"':'')+'>'+(num?fix(v):v)+'</td>';});h+='</tr>';});
  return h+'</tbody></table>';
}

function dibujar(){
  raiz.innerHTML='';
  add('<h1>Figuras sobre las dimensiones</h1>'
    + '<p class="sub">Comparación del método léxico actual contra el rediseño zero-shot. '
    + DATOS.n_textos+' textos · '+DATOS.ficha.pasadas_por_texto+' pasadas de NLI por texto.</p>');

  /* Rótulo de procedencia: TRES estados, no dos. El del corpus no dice
     "no reportable" sino algo distinto y más fácil de pasar por alto:
     descriptivo pero no validado. */
  if(DATOS.procedencia==='bateria'){
    add('<div class="banner"><b>BATERÍA DE PRUEBA DEL INSTRUMENTO — no es una muestra.</b>'
      + '<p>Estas figuras se construyeron sobre cinco frases escritas a propósito, cada una '
      + 'en seis idiomas. Sirven para ver cómo se comporta el instrumento cuando el contenido '
      + 'no varía. <b>Ningún número de esta página es reportable como hallazgo sobre el corpus.</b></p>'
      + '<p>Traducciones: '+esc(DATOS.ficha.traduccion||'—')+'. Si una versión resulta más '
      + 'enfática que otra, parte de la dispersión atribuida al instrumento es artefacto de '
      + 'traducción; están abajo, completas, para que se puedan auditar.</p></div>');
  } else if(DATOS.procedencia==='corpus'){
    add('<div class="banner"><b>CORPUS REAL — descriptivo, NO validado.</b>'
      + '<p>'+DATOS.n_textos+' textos del corpus del proyecto. A diferencia de la batería, '
      + 'estos números <b>sí</b> hablan del corpus — pero <b>ninguno está validado</b>: el '
      + 'archivo no trae codificación humana, así que el acuerdo de cada método contra '
      + 'codificadores sigue sin medirse.</p>'
      + '<p><b>Es el riesgo de leer esta página demasiado rápido:</b> un corpus de este tamaño '
      + 'invita a tratar sus promedios como hallazgos firmes cuando todavía no lo son.</p></div>');
  } else {
    add('<div class="banner"><b>Textos aportados a mano.</b>'
      + '<p>Ni batería ni corpus: lo que digan estas figuras vale para estos '
      + DATOS.n_textos+' textos y nada más.</p></div>');
  }

  if(DATOS.alertas && DATOS.alertas.length){
    let h='<div class="alertas"><h3>Alertas</h3><ul>';
    DATOS.alertas.forEach(a=>h+='<li>'+esc(a)+'</li>');
    add(h+'</ul></div>');
  } else {
    add('<div class="bien">Ninguna dimensión superó los umbrales configurados '
      + '(mudez '+DATOS.umbral_mudez+', dimensión muda '
      + Math.round(DATOS.umbral_dim_muda*100)+'%).</div>');
  }

  add('<div class="bien" style="border-left-color:var(--eje)"><b>Estas figuras describen; no validan.</b> '
    + 'Muestran cómo se comporta cada instrumento, no cuál mide mejor la realidad. Para eso hace '
    + 'falta codificar a mano una muestra (~200 tuits) y calcular el alfa de Krippendorff de cada '
    + 'método contra esos códigos, con umbral 0.67, como indica la última sección del notebook.</div>');

  seccionDispersion();
  seccionPlano();
  seccionValores();
  seccionAusencia();
  seccionProbs();
  seccionLexico();
  seccionEstratos();
  seccionAncla();
  seccionFicha();
}

/* ---------- verificación del ancla geográfica ---------- */
function seccionAncla(){
  add('<h2>8 · ¿Importa dónde vive el ancla «México»?</h2>');
  const A=DATOS.ancla;
  if(!A){
    add('<div class="aviso"><b>Sin medir.</b> El ancla geográfica vive en la '
      + (DATOS.ancla_en_etiquetas?'<b>etiquetas</b>':'<b>plantilla</b>')
      + ', y esa decisión se apoya <b>solo en el argumento mecánico</b> de la sección 3: '
      + 'el softmax cancela un prefijo constante, no un predicado que encarece las '
      + 'afirmaciones y abarata la negación. Para respaldarla —o refutarla— pon '
      + '<code>COMPARAR_ANCLA = True</code> y vuelve a correr la batería.</div>');
    return;
  }
  const filas=A.filas.map(f=>[f.dimension,
    fix(f.plantilla.relevancia_media,4), fix(f.etiquetas.relevancia_media,4),
    fix(f.delta_relevancia,4),
    Math.round(f.plantilla.prop_muda*100)+'% / '+Math.round(f.etiquetas.prop_muda*100)+'%']);
  const veredicto = A.prediccion_confirmada
    ? '<b>Predicción confirmada.</b> El ancla dentro de las etiquetas baja la relevancia '
      + '(delta medio <span class="cifra">'+fix(A.delta_medio,4)+'</span>), que es lo que '
      + 'la sección 3 argumenta: infla <code>na</code> porque añadir el predicado a una '
      + 'negación la hace más fácil de entrañar.'
    : '<b>Predicción NO confirmada.</b> El ancla dentro de las etiquetas <b>no</b> baja la '
      + 'relevancia aquí (delta medio <span class="cifra">'+fix(A.delta_medio,4)+'</span>). '
      + 'El argumento mecánico de la sección 3 no se sostiene tal como está escrito, y la '
      + 'decisión de diseño hay que revisarla antes de reportar nada.';
  const id='f8';
  bloque(id,'relevancia media según dónde viva el ancla',
    '¿La redacción de las hipótesis, y no el corpus, es lo que hunde la relevancia?',
    A.filas.length, veredicto,
    tabla(['dimensión','ancla en plantilla','ancla en etiquetas','delta','% mudo (plant/etiq)'],filas),
    300);
  init(id,{
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
    legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:22},
    toolbox:TOOLBOX,
    grid:{left:52,right:20,top:18,bottom:52},
    xAxis:Object.assign({type:'category',data:A.filas.map(f=>f.dimension)},EJE_X),
    yAxis:Object.assign({type:'value',min:0,max:1,name:'relevancia media'},EJE_Y),
    series:[
      {name:'ancla en la plantilla',type:'bar',barGap:'12%',
       data:A.filas.map(f=>f.plantilla.relevancia_media),
       itemStyle:{color:P.nuevo,borderRadius:3},
       label:{show:true,position:'top',fontSize:10,color:P.tinta2,
              formatter:p=>Number(p.value).toFixed(2)}},
      {name:'ancla en las etiquetas',type:'bar',
       data:A.filas.map(f=>f.etiquetas.relevancia_media),
       itemStyle:{color:P.mudo,borderRadius:3,
         decal:{symbol:'line',dashArrayX:[1,0],dashArrayY:[2,4],
                rotation:-Math.PI/4,color:'rgba(0,0,0,.22)'}},
       label:{show:true,position:'top',fontSize:10,color:P.tinta2,
              formatter:p=>Number(p.value).toFixed(2)}}
    ]
  });
  add('<div class="nota">Umbral de mudez usado: <span class="cifra">'+A.umbral_mudez
    + '</span>. Dimensiones declaradas mudas — con el ancla en la plantilla: <span class="cifra">'
    + A.dimensiones_mudas.plantilla+'</span>; en las etiquetas: <span class="cifra">'
    + A.dimensiones_mudas.etiquetas+'</span>.</div>');
}

/* ---------- estratos de recolección ---------- */
function seccionEstratos(){
  const E=DATOS.estratos;
  if(!E || !E.filas || !E.filas.length) return;
  add('<h2>7 · ¿De qué consulta viene cada número?</h2>');
  add('<p class="sub">El corpus no es una masa homogénea: se recolectó con consultas distintas, '
    + 'y algunas buscaban expresamente una de las dimensiones que aquí se promedian. Promediar '
    + 'sobre el conjunto mide la consulta tanto como el fenómeno. Por eso se desglosa.</p>');
  const dims=[...new Set(E.filas.flatMap(f=>Object.keys(f.dimensiones)))].sort();
  const filas=E.filas.map(f=>[f.estrato, f.n, Math.round(f.proporcion*100)+'%']
    .concat(dims.map(d=>f.dimensiones[d]?fix(f.dimensiones[d].valencia,3):'—')));
  const id='f7';
  bloque(id,'valencia media por estrato',
    '¿El promedio cambia según con qué consulta se recolectó el texto?',
    E.filas.reduce((a,f)=>a+f.n,0),
    '<b>Un promedio que mezcla estratos no es reportable como hallazgo sobre la población.</b> '
    + 'Las alertas de arriba dicen cuáles están dirigidos.',
    tabla(['estrato','n','% del corpus'].concat(dims),filas),
    Math.max(220, 60+E.filas.length*28));
  init(id,{
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
    legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:18,type:'scroll',width:'92%'},
    toolbox:TOOLBOX,
    grid:{left:170,right:24,top:14,bottom:52},
    xAxis:Object.assign({type:'value',min:-1,max:1},EJE_X),
    yAxis:Object.assign({type:'category',
      data:E.filas.map(f=>String(f.estrato).slice(0,26))},EJE_Y),
    series:dims.map((d,k)=>({
      name:d,type:'bar',
      data:E.filas.map(f=>f.dimensiones[d]?f.dimensiones[d].valencia:null),
      itemStyle:{borderRadius:2},
      label:{show:false}}))
  });
}

/* ---------- F1 · dispersión entre idiomas ---------- */
function seccionDispersion(){
  const D=DATOS.dispersion; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>1 · ¿Mide lo mismo cuando el contenido es el mismo?</h2>');
  add('<p class="sub">El mismo contenido, en varios idiomas. Un instrumento válido debería darles '
    + 'aproximadamente el mismo valor. La dispersión —rango y desviación— es el número que se '
    + 'reporta: mide cuánto varía el instrumento cuando el contenido no varía. Idealmente, cero. '
    + 'Los idiomas salen de los datos cargados, no de una lista declarada.</p>');
  if(DATOS.idiomas_escasos && Object.keys(DATOS.idiomas_escasos).length){
    add('<div class="aviso">Excluidos por tener menos de '+DATOS.n_minimo_idioma
      + ' observaciones: '
      + Object.entries(DATOS.idiomas_escasos).map(([l,n])=>esc(l)+' (n='+n+')').join(', ')
      + '. Se nombran para que la exclusión se vea, en vez de dibujarlos como comparables.</div>');
  }
  if(DATOS.dispersion_excluidas && DATOS.dispersion_excluidas.length){
    let h='<div class="aviso"><b>Dimensiones que no pueden aparecer aquí:</b><ul>';
    DATOS.dispersion_excluidas.forEach(x=>h+='<li><code>'+esc(x.dimension)+'</code> — '+esc(x.razon)+'</li>');
    add(h+'</ul></div>');
  }
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f1_'+i;
    const metodos = v.metodos || ['actual','nuevo'];
    let nota='<b>Dispersión entre idiomas.</b> ';
    metodos.forEach(m=>{const st=v.stats[m];
      nota += NOMBRE_METODO(m)+': rango <span class="cifra">'+fix(st&&st.rango,4)+'</span>, '
            + 'σ <span class="cifra">'+fix(st&&st.sigma,4)+'</span>. ';});
    if(metodos.indexOf('traducido')>=0){
      nota += '<br><b>Cuidado con «léxico traducido»:</b> es el MISMO léxico corriendo sobre la '
            + 'columna de traducción, así que mide el léxico junto con la calidad de una '
            + 'traducción automática. Si gana, no se sabe cuánto mérito es de cada uno.';
    }
    const filas=v.langs.map((l,k)=>[l].concat(metodos.map(m=>v.series[m][k])));
    bloque(id,'<code>'+dim+'</code> por idioma',
      '¿Cuánto cambia la medición cuando solo cambia el idioma?',
      v.langs.length,nota,
      tabla(['idioma'].concat(metodos.map(NOMBRE_METODO)),filas),320);
    init(id,{
      tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
      legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:22,type:'scroll',width:'92%'},
      toolbox:TOOLBOX,
      grid:{left:52,right:20,top:18,bottom:52},
      xAxis:Object.assign({type:'category',data:v.langs},EJE_X),
      yAxis:Object.assign({type:'value',min:-1,max:1},EJE_Y),
      series:metodos.map((m,k)=>({
        name:NOMBRE_METODO(m),type:'bar',data:v.series[m],
        barGap:k===0?'12%':undefined,
        itemStyle:{color:COLOR_METODO(m),borderRadius:3,
                   decal:m==='traducido'?{symbol:'line',dashArrayX:[1,0],
                     dashArrayY:[2,4],rotation:-Math.PI/4,color:'rgba(0,0,0,.22)'}:null},
        label:{show:true,position:'top',fontSize:10,color:P.tinta2,
               formatter:p=>Number(p.value).toFixed(2)}}))
    });
  });
}

/* ---------- F2 · plano valencia × relevancia ---------- */
const SIMBOLOS=['circle','diamond','triangle','rect','pin'];
function seccionPlano(){
  const D=DATOS.plano; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>2 · ¿Cuántos de los ceros de hoy son «no habló»?</h2>');
  add('<p class="sub">Dos ejes que el método actual colapsa en uno. Un valor cercano a cero '
    + 'arriba a la derecha significa «habló del tema y su juicio fue neutro»; el mismo valor '
    + 'a la izquierda significa «no habló». Hoy ambos son el mismo <code>0.0000</code> y se '
    + 'promedian juntos. Las formas distinguen la frase de origen, para que la figura se lea '
    + 'también impresa en blanco y negro.</p>');
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f2_'+i;
    const grupos=Object.keys(v.grupos);
    const series=grupos.map((g,k)=>({
      name:g,type:'scatter',symbol:SIMBOLOS[k%SIMBOLOS.length],symbolSize:12,
      itemStyle:{color:P.actual,borderColor:P.sup,borderWidth:2},
      data:v.grupos[g].map(p=>[p.x,p.y,p.lang,p.texto])
    }));
    series[0].markArea={silent:true,itemStyle:{color:'rgba(137,135,129,0.13)'},
      label:{show:true,position:'insideTop',color:P.mudo,fontSize:11,
             formatter:'no habló del tema'},
      data:[[{xAxis:0},{xAxis:DATOS.umbral_mudez}]]};
    series[0].markLine={silent:true,symbol:'none',
      lineStyle:{color:P.eje,type:'dashed'},label:{show:false},
      data:[{yAxis:0}]};
    const filas=[];grupos.forEach(g=>v.grupos[g].forEach(p=>filas.push(
      [g,p.lang,p.x,p.y,p.texto])));
    const nota='<b>'+v.mudos+' de '+v.n+'</b> observaciones caen en la banda de mudez '
      + '(relevancia &lt; '+DATOS.umbral_mudez+'). No se filtran: dibujarlas <em>es</em> la '
      + 'evidencia. El umbral es un corte de lectura, no un filtro de datos.';
    bloque(id,'<code>'+dim+'</code> — valencia contra relevancia',
      '¿La observación no opinó, o no habló del tema?',
      v.n,nota,tabla(['frase','idioma','relevancia','valencia','texto'],filas),360);
    init(id,{
      tooltip:{trigger:'item',formatter:p=>
        '<b>'+esc(p.seriesName)+'</b> · '+esc(p.data[2]||'')+'<br>'
        +'relevancia '+fix(p.data[0],2)+' · valencia '+fix(p.data[1],2)
        +'<br><span style="color:#52514e">'+esc((p.data[3]||'').slice(0,70))+'…</span>'},
      legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemGap:22,type:'scroll',width:'92%'},
      toolbox:TOOLBOX,
      grid:{left:60,right:24,top:18,bottom:78},
      xAxis:Object.assign({type:'value',min:0,max:1,name:'relevancia',
        nameLocation:'middle',nameGap:30,nameTextStyle:{color:P.tinta2}},EJE_X,
        {splitLine:{lineStyle:{color:P.grid}}}),
      yAxis:Object.assign({type:'value',min:-1,max:1,name:'valencia',
        nameLocation:'middle',nameGap:38,nameTextStyle:{color:P.tinta2}},EJE_Y),
      series:series
    });
  });
  if(DATOS.plano_excluidas && DATOS.plano_excluidas.length){
    let h='<p class="nota"><b>Dimensiones excluidas de esta figura:</b> ';
    h+=DATOS.plano_excluidas.map(e=>'<code>'+esc(e.dimension)+'</code> — '+esc(e.razon)).join('; ');
    add(h+'.</p>');
  }
  if(DATOS.control_mudo){
    const UR=(DATOS.umbral_relevancia!=null?DATOS.umbral_relevancia:DATOS.umbral_mudez);
    let h='<div class="bien"><b>El control mudo.</b> Un enunciado logistico sin carga '
      + 'evaluativa, que no habla del animo, ni de comida, lugares o cultura, ni de '
      + 'autoridades. Si el eje de relevancia funciona, debe salir bajo en las tres:<ul>';
    DATOS.control_mudo.forEach(c=>{var bajo=c.relevancia_media<UR;
      h+='<li><code>'+esc(c.dimension)+'</code>: relevancia media '
      + '<span class="cifra">'+fix(c.relevancia_media,2)+'</span> '
      + (bajo?'<b style="color:var(--ok)">por debajo del umbral, como debe ser</b>'
            :'<b style="color:var(--alerta)">POR ENCIMA del umbral '+DATOS.umbral_mudez
              +': el modelo cree que este texto sí habla del tema</b>')
      + ', valencia media <span class="cifra">'+fix(c.valencia_media,2)+'</span></li>';});
    add(h+'</ul>El método actual asigna a este mismo texto un 0.0000 indistinguible del de un '
      + 'turista que habló del tema y lo juzgó neutro.</div>');
  }
}

/* ---------- F3 · valores posibles de cada método ---------- */
function seccionValores(){
  const D=DATOS.valores; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>3 · ¿Qué valores puede tomar cada método?</h2>');
  add('<p class="sub">Cada punto es un texto. La diagonal es el acuerdo perfecto entre ambos '
    + 'métodos: cuanto más lejos de ella, más discrepan.</p>');
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f3_'+i;
    const series=[
      {name:'acuerdo perfecto',type:'line',silent:true,data:[[-1,-1],[1,1]],symbol:'none',silent:true,
       lineStyle:{color:P.eje,type:'dashed',width:1},tooltip:{show:false}},
      {name:'textos',type:'scatter',symbolSize:12,
       itemStyle:{color:P.nuevo,borderColor:P.sup,borderWidth:2},
       data:v.puntos.map(p=>[p.x,p.y,p.lang,p.texto])}
    ];
    if(v.escalonada && v.marcas.length){
      series[1].markLine={silent:true,symbol:'none',
        lineStyle:{color:P.actual,type:'dotted',width:1,opacity:0.55},
        label:{show:false},
        data:v.marcas.map(m=>({xAxis:m}))};
    }
    const nota = v.escalonada
      ? '<b>Las líneas verticales son los únicos valores que el método actual puede tomar aquí</b> '
        + '('+v.marcas.length+' en total). Fórmula: '+esc(v.formula)
      : '<b>Esta dimensión no es escalonada.</b> Su fórmula actual '+esc(v.formula)
        + ', así que no tiene sentido marcar valores discretos: no existen.';
    bloque(id,'<code>'+dim+'</code> — actual contra rediseño',
      '¿Dónde discrepan los dos métodos, y qué valores puede producir cada uno?',
      v.puntos.length,nota,
      tabla(['idioma','actual','rediseño','texto'],
            v.puntos.map(p=>[p.lang,p.x,p.y,p.texto])),360);
    init(id,{
      tooltip:{trigger:'item',formatter:p=>p.seriesName==='textos'
        ? esc(p.data[2]||'')+'<br>actual '+fix(p.data[0],3)+' · rediseño '+fix(p.data[1],3)
          +'<br><span style="color:#52514e">'+esc((p.data[3]||'').slice(0,70))+'…</span>'
        : ''},
      toolbox:TOOLBOX,
      grid:{left:60,right:24,top:18,bottom:56},
      xAxis:Object.assign({type:'value',min:-1,max:1,name:'método actual',
        nameLocation:'middle',nameGap:30,nameTextStyle:{color:P.tinta2}},EJE_X,
        {splitLine:{lineStyle:{color:P.grid}}}),
      yAxis:Object.assign({type:'value',min:-1,max:1,name:'rediseño',
        nameLocation:'middle',nameGap:38,nameTextStyle:{color:P.tinta2}},EJE_Y),
      series:series
    });
  });
}

/* ---------- F4 · ausencia por dimensión ---------- */
function seccionAusencia(){
  const D=DATOS.ausencia; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>4 · ¿Esta dimensión mide algo en este corpus?</h2>');
  add('<p class="sub">Distribución <em>completa</em> de la relevancia, con el umbral superpuesto. '
    + 'Se muestra entera a propósito: el corte en '+DATOS.umbral_mudez+' es arbitrario y hay que '
    + 'poder juzgar qué tan sensible es la lectura al valor elegido.</p>');
  const etq=[];for(let k=0;k<10;k++)etq.push((k/10).toFixed(1)+'–'+((k+1)/10).toFixed(1));
  claves.forEach((dim,i)=>{
    const v=D[dim], id='f4_'+i;
    const corte=Math.round(DATOS.umbral_mudez*10);
    const muda = v.prop_muda > DATOS.umbral_dim_muda;
    const nota='<b>'+Math.round(v.prop_muda*100)+'%</b> de las observaciones quedan bajo el '
      + 'umbral de mudez. '+(muda
        ? '<span style="color:var(--alerta)">Supera el '
          + Math.round(DATOS.umbral_dim_muda*100)+'% configurado: en este corpus la dimensión '
          + 'apenas mide nada, y su promedio está dominado por textos que no hablaron del tema.</span>'
        : 'Por debajo del '+Math.round(DATOS.umbral_dim_muda*100)+'% configurado.')
      + ' La ausencia se reporta como magnitud propia, nunca apilada junto a la valencia.';
    bloque(id,'<code>'+dim+'</code> — distribución de relevancia',
      '¿Hablaron del tema, o la dimensión está muda?',
      v.n,nota,tabla(['rango de relevancia','observaciones'],
                     etq.map((e,k)=>[e,v.conteo[k]])),300);
    init(id,{
      tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
      toolbox:TOOLBOX,
      grid:{left:52,right:20,top:18,bottom:40},
      xAxis:Object.assign({type:'category',data:etq},EJE_X),
      yAxis:Object.assign({type:'value',minInterval:1},EJE_Y),
      series:[{type:'bar',data:v.conteo,
        itemStyle:{color:p=>p.dataIndex<corte?P.mudo:P.nuevo,borderRadius:3},
        label:{show:true,position:'top',fontSize:10,color:P.tinta2,
               formatter:p=>p.value>0?p.value:''},
        markArea:{silent:true,itemStyle:{color:'rgba(137,135,129,0.13)'},
          label:{show:true,position:'insideTopLeft',color:P.mudo,fontSize:11,
                 formatter:'zona muda'},
          data:corte>0?[[{xAxis:etq[0]},{xAxis:etq[Math.max(corte-1,0)]}]]:[]}
      }]
    });
  });
}

/* ---------- F5 · descomposición de probabilidades ---------- */
const COLOR_ETIQUETA={pos:'#2a78d6',na:'#898781',neg:'#e34948',
  safety:'#2a78d6',neutral:'#898781',insecurity:'#eda100',violence:'#e34948',
  atmosfera:'#2a78d6',imagen_cultural:'#eda100',perspectiva_politica:'#52514e'};
const ORDEN_ETIQUETA=['pos','safety','na','neutral','insecurity','neg','violence',
  'atmosfera','imagen_cultural','perspectiva_politica'];
function seccionProbs(){
  const D=DATOS.probs; const claves=Object.keys(D);
  if(!claves.length) return;
  add('<h2>5 · ¿A dónde se va la probabilidad del modelo?</h2>');
  add('<p class="sub">Las hipótesis de cada grupo compiten y suman 1. Si la hipótesis de '
    + 'irrelevancia se lo lleva casi todo en todas partes, las frases están mal calibradas: '
    + 'es un problema del instrumento, no del corpus. Las frases exactas están en la ficha '
    + 'técnica, al final.</p>');
  claves.forEach((grupo,i)=>{
    const v=D[grupo], id='f5_'+i;
    const etq=v.etiquetas.slice().sort((a,b)=>{
      const ia=ORDEN_ETIQUETA.indexOf(a),ib=ORDEN_ETIQUETA.indexOf(b);
      return (ia<0?99:ia)-(ib<0?99:ib);});
    const series=etq.filter(e=>v.valores[e]).map(e=>({
      name:e,type:'bar',stack:'p',
      data:v.valores[e],
      itemStyle:{color:COLOR_ETIQUETA[e]||P.mudo,borderColor:P.sup,borderWidth:2},
      label:{show:true,position:'inside',fontSize:10,color:'#fff',
             formatter:p=>p.value<=0.10?'':((p.dataIndex===0&&p.value>0.16)
               ? e+'  '+Number(p.value).toFixed(2)
               : Number(p.value).toFixed(2))}
    }));
    const filas=v.filas.map((f,k)=>[f].concat(etq.map(e=>v.valores[e]?v.valores[e][k]:null)));
    const nota='Promedio por <code>'+esc(v.agregado_por)+'</code>. Cada barra suma 1. '
      + 'Las etiquetas van en orden semántico: positiva, irrelevancia, negativa.';
    bloque(id,'grupo <code>'+grupo+'</code>',
      '¿Cómo reparte el modelo la probabilidad entre las hipótesis?',
      null,nota,tabla(['fila'].concat(etq),filas),
      Math.max(180,60+v.filas.length*42));
    init(id,{
      tooltip:{trigger:'axis',axisPointer:{type:'shadow'},
        valueFormatter:v2=>Number(v2).toFixed(3)},
      legend:{bottom:0,textStyle:{color:P.tinta2,fontSize:11},itemWidth:14,itemHeight:8,itemGap:22,type:'scroll',width:'92%'},
      toolbox:TOOLBOX,
      grid:{left:130,right:24,top:12,bottom:46},
      xAxis:Object.assign({type:'value',max:1},EJE_Y),
      yAxis:Object.assign({type:'category',data:v.filas},EJE_X,
        {axisLabel:{color:P.tinta2,fontSize:11}}),
      series:series
    });
  });
}

/* ---------- F6 · cobertura del léxico ---------- */
function seccionLexico(){
  const v=DATOS.lexico;
  if(!v) return;
  add('<h2>6 · ¿Llega a activarse el léxico del método actual?</h2>');
  add('<p class="sub">El método actual mide tres dimensiones contando palabras de listas en '
    + 'inglés y español. Donde no activa ninguna, las tres dimensiones valen lo mismo por '
    + 'construcción: no es una medición, es el instrumento.'
    + (v.por_idioma ? '' : ' <b>Los textos cargados no declaran idioma</b>, así que el '
        + 'desglose por idioma no aplica y todo cae en un solo grupo; el emparejamiento por '
        + 'subcadena se mide igual.')
    + '</p>');
  const langs=v.filas.map(f=>f.lang);
  const cob=v.filas.map(f=>f.cobertura);
  const nota_sub = v.total_matches===0
    ? '<b>No se midió en este corpus:</b> el léxico no activó ninguna palabra en ningún texto, '
      + 'así que no hay emparejamientos que clasificar. Eso <em>no</em> significa que el '
      + 'emparejamiento por subcadena no ocurra.'
    : '<b>'+v.total_subcadenas+' de '+v.total_matches+'</b> emparejamientos ocurrieron por '
      + 'subcadena y no por palabra completa. El método actual busca con <code>w in texto</code>, '
      + 'sin frontera de palabra: <code>safe</code> se activa dentro de <code>unsafe</code>, '
      + '<code>kind</code> dentro de <code>kinda</code>. Aquí solo se mide; corregirlo '
      + 'falsearía la comparación, porque dejaría de ser el método que está en producción.';
  let ej='';
  if(v.ejemplos && v.ejemplos.length){
    ej='<details><summary>emparejamientos por subcadena encontrados</summary>'
      + '<div class="tabla-scroll">'+tabla(['palabra','dimensión','idioma','texto'],
          v.ejemplos.map(e=>[e.palabra,e.dimension,e.lang,e.texto]))+'</div></details>';
  }
  bloque('f6','Cobertura del léxico por idioma',
    '¿En qué proporción de los textos de cada idioma el léxico activa alguna palabra?',
    v.filas.reduce((a,f)=>a+f.n_textos,0),
    nota_sub,
    tabla(['idioma','textos','cobertura','emparejamientos','por subcadena'],
      v.filas.map(f=>[f.lang,f.n_textos,f.cobertura,f.matches,f.subcadenas])),300);
  init('f6',{
    tooltip:{trigger:'axis',axisPointer:{type:'shadow'},
      valueFormatter:x=>Math.round(x*100)+'%'},
    toolbox:TOOLBOX,
    grid:{left:56,right:20,top:18,bottom:40},
    xAxis:Object.assign({type:'category',data:langs},EJE_X),
    yAxis:Object.assign({type:'value',min:0,max:1,
      axisLabel:{color:P.tinta2,fontSize:11,formatter:x=>Math.round(x*100)+'%'}},EJE_Y),
    series:[{type:'bar',data:cob,
      itemStyle:{color:p=>p.value===0?P.mudo:P.actual,borderRadius:3},
      label:{show:true,position:'top',fontSize:10,color:P.tinta2,
             formatter:p=>Math.round(p.value*100)+'%'}}]
  });
  if(ej) add(ej);
}

/* ---------- ficha técnica ---------- */
function seccionFicha(){
  const f=DATOS.ficha;
  add('<h2>Ficha técnica del instrumento</h2>');
  add('<p class="sub">Las hipótesis <em>son</em> el instrumento de medición: cambiarlas cambia '
    + 'los resultados. Van aquí textualmente para que la figura sea citable, igual que se cita '
    + 'un libro de códigos.</p>');
  let h='<section class="fig"><table><tbody>';
  h+='<tr><th>idioma de las hipótesis</th><td><code>'+esc(f.idioma_hipotesis)+'</code></td></tr>';
  h+='<tr><th>plantilla</th><td><code>'+esc(f.plantilla)+'</code></td></tr>';
  h+='<tr><th>modelo de sentimiento</th><td><code>'+esc(f.modelo_sentimiento)+'</code></td></tr>';
  h+='<tr><th>modelo NLI</th><td><code>'+esc(f.modelo_nli)+'</code></td></tr>';
  h+='<tr><th>pasadas de NLI por texto</th><td>'+f.pasadas_por_texto+'</td></tr>';
  if(f.tiempo_corrida_s!=null)
    h+='<tr><th>tiempo de la corrida</th><td>'+f.tiempo_corrida_s+' s</td></tr>';
  h+='<tr><th>fecha</th><td>'+esc(f.fecha)+'</td></tr>';
  if(f.huella_instrumento) h+='<tr><th>huella del instrumento</th><td><code>'+esc(f.huella_instrumento)+'</code></td></tr>';
  if(f.corrida_restaurada) h+='<tr><th>corrida restaurada de</th><td><code>'+esc(f.corrida_restaurada)+'</code></td></tr>';
  h+='</tbody></table>';
  h+='<h3 style="margin-top:16px">Hipótesis</h3>'+tabla(['grupo','etiqueta','frase'],
     f.hipotesis.map(x=>[x.grupo,x.etiqueta,x.frase]));
  h+='</section>';
  add(h);

  if(DATOS.bateria && DATOS.bateria.length){
    let b='<section class="fig"><h3>Batería completa</h3>'
      + '<p class="preg">Publicada íntegra para que la equivalencia entre traducciones se pueda '
      + 'auditar. Si una versión no dice lo mismo que las otras, la figura 1 lo estará contando '
      + 'como sesgo del instrumento cuando es error de traducción.</p>'
      + '<div class="tabla-scroll"><table><thead><tr><th>frase</th><th>idioma</th>'
      + '<th>texto</th><th>qué prueba</th></tr></thead><tbody>';
    DATOS.bateria.forEach(t=>{b+='<tr><td>'+esc(t.frase)+'</td><td>'+esc(t.lang)+'</td><td>'
      + rtl(t.lang,t.texto)+'</td><td>'+esc(t.que_prueba)+'</td></tr>';});
    add(b+'</tbody></table></div></section>');
  }
}

/* ---------- arranque ----------
   Va al FINAL a proposito. Las declaraciones de funcion se elevan, pero
   `const P` y `const TOOLBOX` no: llamar a dibujar() antes de ellas las deja
   en zona muerta temporal y revienta con "Cannot access 'P' before
   initialization" en la primera figura, dejando la pagina a medio dibujar. */
if (typeof echarts === 'undefined') {
  raiz.innerHTML = '<div class="fallo"><h1>No se pudo cargar la biblioteca de gráficas</h1>'
    + '<p>Esta página dibuja con Apache ECharts, que se carga desde un CDN. La petición a '
    + '<code>__CDN__</code> no llegó.</p>'
    + '<p>Causas habituales: no hay conexión a internet, una red corporativa bloquea el CDN, '
    + 'o el archivo se abrió sin red. Los datos <em>sí</em> están dentro de este archivo: '
    + 'recupera la conexión y recarga, y las figuras aparecen sin volver a ejecutar el notebook.</p></div>';
} else {
  dibujar();
}
</script>
</body>
</html>
"""

# ===========================================================================
# UNA PAGINA, DOS DESTINOS
#
# La misma funcion construye el documento completo. Dentro del notebook se
# entrega con IPython.display.HTML; al disco se escribe tal cual. No hay dos
# rutas de renderizado que puedan divergir.
#
# En Colab cada salida de celda vive en un iframe aislado y el .ipynb guarda
# el HTML sin el estado del JavaScript: al reabrir el notebook guardado las
# figuras dejan de ser interactivas hasta reejecutar esta celda. El archivo
# descargable es el artefacto que sobrevive, y es el que va a la tesis.
# ===========================================================================


def construir_pagina(datos):
    return PLANTILLA_HTML.replace("__CDN__", CDN_ECHARTS).replace(
        "__DATOS__", json.dumps(datos, ensure_ascii=False))


PAGINA = construir_pagina(DATOS_FIGURAS)

with open(RUTA_HTML, "w", encoding="utf-8") as _fh:
    _fh.write(PAGINA)

import os as _os
print("=" * 74)
print("FIGURAS GENERADAS")
print("=" * 74)
print("  archivo: {}  ({:.0f} KB)".format(RUTA_HTML,
                                          _os.path.getsize(RUTA_HTML) / 1024))
print("  Este archivo es el artefacto que sobrevive al cierre del notebook.")
print("  Cada figura se exporta como SVG desde su barra de herramientas.")
if DATOS_FIGURAS["alertas"]:
    print()
    print("  ALERTAS")
    for _a in DATOS_FIGURAS["alertas"]:
        print("     ·", _a)
else:
    print()
    print("  Ninguna dimension supero los umbrales configurados.")
print("=" * 74)

try:
    from google.colab import files as _files
    _files.download(RUTA_HTML)
except Exception as _exc:
    print("(descarga automatica no disponible: {})".format(_exc))

display(HTML(PAGINA))

## 14. Validación convergente: zero-shot contra la rúbrica

Dos instrumentos automáticos midieron el mismo corpus. Si miden lo mismo, deberían coincidir en
dos cosas:

```
 valencia
   +1 ┤ ░░░░░░░░░░░░░░░░░         ● 5         ← la rúbrica ordena en VERTICAL:
      │ ░ ausencia (nivel 0)    ● 4              1–2 abajo, 3 al centro, 4–5 arriba
    0 ┤ ░ = fondo gris            ◆ 3
      │ ░░░░░░░░░░░░░░░░░    ● 2
   -1 ┤                         ● 1
      └───────────────┬──────────────── relevancia
                  umbral           ← y en HORIZONTAL: lo calificado, a la derecha
```

- **Separación** (eje horizontal): ¿la relevancia del zero-shot distingue a los tuits con nivel > 0
  de los que la rúbrica dejó en ausencia? Se mide con el **AUC**: 0.5 es azar, 1 es separación
  perfecta.
- **Orden** (eje vertical): entre los tuits calificados, ¿la valencia sube con el nivel? Se mide con
  la **ρ de Spearman** entre nivel (1–5) y valencia.

Cada métrica lleva intervalo por remuestreo y su `n`. Los niveles con muy pocos tuits se dibujan
igual, con aviso.

**Antes de leer un número, lee el constructo.** La tabla de correspondencia de abajo dice qué mide
cada lado. Un AUC bajo puede venir de un instrumento malo **o** de dos instrumentos que miden
cosas distintas. En `perspectiva_politica` pasa lo segundo: la rúbrica incluye infraestructura,
logística y gestión de seguridad, y la compuerta solo pregunta por autoridades y gobierno.

**La saliencia de violencia no va en el plano**: es presencia, no valencia. Tiene su propia curva
ROC contra `violence_salience`.

> ⚠️ **Esto no es validación contra codificación humana.** La rúbrica la aplicó Gemini. Que dos
> modelos coincidan dice que convergen, no que acierten.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

# ===========================================================================
# VALIDACION CONVERGENTE
#
# Une por IDENTIFICADOR los valores del instrumento (LARGO, metodo 'nuevo') con
# la rubrica (RUBRICA_TUIT) y mide separacion (AUC) y orden (rho de Spearman).
# Nunca se une por texto: la vista previa del 11/09 que se unio por los primeros
# 120 caracteres perdio filas.
# ===========================================================================

N_BOOTSTRAP = 1000
SEMILLA_VALIDACION = 20260913
N_MINIMO_NIVEL = 5          # por debajo, el nivel se dibuja con aviso

# ---------------------------------------------------------------------------
# CORRESPONDENCIA DECLARADA. Que magnitud del instrumento se contrasta con
# cada criterio y que mide cada lado. Se escribe a mano a proposito: inferirla
# de los nombres diria que "perspectiva politica" y "perspectiva politica" son
# lo mismo, y no lo son.
# ---------------------------------------------------------------------------
CORRESPONDENCIA = {
    "rubrica_1_atmosfera_de_mexic": {
        "dimension": "atmosfera", "magnitud": "relevancia", "figura": "plano",
        "constructo_rubrica": "Afecto del emisor hacia México o su gente: de odio o desprecio (1) "
                              "a fascinación (5).",
        "constructo_zeroshot": "Ánimo vivido: ambiente alegre y emocionante contra desagradable y "
                               "estresante.",
        "desajuste": "La rúbrica mide afecto HACIA el país y su gente; el instrumento, el ánimo QUE SE "
                     "VIVE. Un tuit hostil hacia México que no describe ningún ambiente puede quedar "
                     "fuera de la compuerta."},
    "rubrica_2_imagen_cultural_de": {
        "dimension": "imagen_cultural", "magnitud": "relevancia", "figura": "plano",
        "constructo_rubrica": "Juicio estético sobre objetos culturales mexicanos: comida, arte, "
                              "historia, arquitectura, paisaje.",
        "constructo_zeroshot": "Cultura del lugar, como su comida y sus sitios: hermosa e impresionante "
                               "contra fea y decepcionante.",
        "desajuste": "Constructos cercanos. La rúbrica exige un juicio; la compuerta se activa con la "
                     "sola mención del tema."},
    "rubrica_3_perspectiva_politi": {
        "dimension": "perspectiva_politica", "magnitud": "relevancia", "figura": "plano",
        "constructo_rubrica": "Competencia funcional de México: economía, instituciones, "
                              "infraestructura, educación, organización, logística y gestión de "
                              "seguridad.",
        "constructo_zeroshot": "Autoridades y gobierno: competentes y honestos contra incompetentes y "
                               "corruptos.",
        "desajuste": "El instrumento cubre solo autoridades y gobierno. La rúbrica incluye "
                     "infraestructura, logística, organización y seguridad: el constructo zero-shot "
                     "es más estrecho, y un AUC bajo aquí es esperable por diseño."},
    "rubrica_4_saliencia_de_viole": {
        "dimension": "violence_salience", "magnitud": "prominencia", "figura": "roc",
        "constructo_rubrica": "Mención o alusión a violencia, crimen o inseguridad contemporáneas en "
                              "México, en cualquier postura (presencia binaria).",
        "constructo_zeroshot": "Prominencia = P(violencia) + 0.5·P(inseguridad) sobre cuatro "
                               "etiquetas en competencia.",
        "desajuste": "Constructos cercanos: ambos son presencia, no valencia."},
}


def _hipotesis_de(dim):
    """El texto real de las hipotesis, para que la tabla cite el instrumento."""
    idi = IDIOMA_HIPOTESIS
    if dim in DIMENSIONES:
        t = DIMENSIONES[dim][idi]
        partes = ["pos: «{}»".format(t["pos"]), "neg: «{}»".format(t["neg"])]
        if DISENO_RELEVANCIA == "compuerta" and dim in GATE[idi]:
            partes.insert(0, "compuerta: «{}»".format(GATE[idi][dim]))
        elif "na" in t:
            partes.append("na: «{}»".format(t["na"]))
        return " · ".join(partes)
    if dim in ("violence_salience", "safety_perceived"):
        return " · ".join("{}: «{}»".format(k, v) for k, v in VIOLENCIA[idi].items())
    return ""


def auc_mw(y, s):
    """AUC por Mann-Whitney: probabilidad de que un presente puntue mas alto
    que un ausente, con empates a la mitad."""
    y = np.asarray(y, dtype=bool)
    s = np.asarray(s, dtype=float)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    r = rankdata(s)
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def ic_bootstrap(fn, *arrays, n=N_BOOTSTRAP, semilla=SEMILLA_VALIDACION):
    rng = np.random.default_rng(semilla)
    N = len(arrays[0])
    vals = []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        v = fn(*[a[idx] for a in arrays])
        if v == v:                       # descarta NaN (remuestra sin una clase)
            vals.append(v)
    if len(vals) < n * 0.5:
        return [float("nan"), float("nan")]
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


def _rho(a, b):
    if len(a) < 3 or np.all(a == a[0]) or np.all(b == b[0]):
        return float("nan")
    return float(spearmanr(a, b).statistic)


def _caja(v):
    v = np.asarray(v, dtype=float)
    if len(v) == 0:
        return None
    q = np.percentile(v, [0, 25, 50, 75, 100])
    return [round(float(x), 4) for x in q]


VALIDACION = None
VALIDACION_METRICAS = None
DATOS_VALIDACION = None

_motivo = None
if not HAY_RUBRICA:
    _motivo = "la entrada no trae columnas de rubrica"
elif PROCEDENCIA != "corpus":
    _motivo = "FUENTE = {!r}: la rubrica solo existe para el corpus".format(PROCEDENCIA)
elif "LARGO" not in globals() or LARGO.empty:
    _motivo = "no hay valores del instrumento (seccion 10)"

if _motivo:
    print("Validacion convergente OMITIDA:", _motivo)
else:
    _n = LARGO[LARGO.metodo == "nuevo"]
    _val = _n.pivot_table(index="texto_id", columns="dimension", values="valencia", aggfunc="first")
    _rel = (_n.pivot_table(index="texto_id", columns="dimension", values="relevancia", aggfunc="first")
              .dropna(axis=1, how="all"))
    _zs = (_val.add_suffix("__valencia").join(_rel.add_suffix("__relevancia"))
               .reset_index().rename(columns={"texto_id": "id"}))
    _zs["id"] = _zs["id"].astype(str)

    _ids_r = set(RUBRICA_TUIT["id"])
    _ids_z = set(_zs["id"])
    VALIDACION = RUBRICA_TUIT.merge(_zs, on="id", how="inner")
    _orig = RUBRICA_ORIGEN_DF.set_index(RUBRICA_ORIGEN_DF[COL_ID].astype(str))
    VALIDACION["texto"] = VALIDACION["id"].map(_orig[COL_TEXTO])
    VALIDACION["traduccion"] = VALIDACION["id"].map(_orig[COL_TRADUCCION]) \
        if COL_TRADUCCION in _orig else ""

    print("=" * 78)
    print("UNION POR IDENTIFICADOR")
    print("   unidos: {}   solo en la rubrica: {}   solo en el instrumento: {}".format(
        len(VALIDACION), len(_ids_r - _ids_z), len(_ids_z - _ids_r)))
    if _ids_r - _ids_z:
        print("   (los tuits de la rubrica sin valor zero-shot quedan fuera: N_MUESTRA o DEDUPE)")
    print("=" * 78)

    _just = RUBRICA.pivot_table(index="id", columns="slug", values="justificacion", aggfunc="first")
    filas_met, planos, rocs, corresp = [], {}, {}, []

    for c in CRITERIOS:
        spec = CORRESPONDENCIA.get(c["slug"])
        if spec is None:
            print("   criterio {} sin correspondencia declarada: fuera de la validacion".format(c["slug"]))
            continue
        dim = spec["dimension"]
        col = dim + ("__relevancia" if spec["magnitud"] == "relevancia" else "__valencia")
        corresp.append({"criterio": c["etiqueta"], "dimension": dim, "magnitud": spec["magnitud"],
                        "rubrica": spec["constructo_rubrica"], "zeroshot": spec["constructo_zeroshot"],
                        "hipotesis": _hipotesis_de(dim), "desajuste": spec["desajuste"]})
        if col not in VALIDACION:
            print("   {}: el instrumento no produjo '{}' en esta corrida".format(c["etiqueta"], col))
            continue

        d = VALIDACION[VALIDACION["clase_" + c["corto"]] != "faltante"].copy()
        faltantes = int((VALIDACION["clase_" + c["corto"]] == "faltante").sum())
        y = d["presente_" + c["corto"]].to_numpy(bool)
        s = d[col].to_numpy(float)
        auc = auc_mw(y, s)
        auc_ic = ic_bootstrap(auc_mw, y, s)
        met = {"criterio": c["etiqueta"], "slug": c["slug"], "dimension": dim,
               "magnitud": spec["magnitud"], "n": int(len(d)), "presentes": int(y.sum()),
               "ausentes": int((~y).sum()), "faltantes_excluidos": faltantes,
               "auc": auc, "auc_ic_inf": auc_ic[0], "auc_ic_sup": auc_ic[1],
               "rho": float("nan"), "rho_ic_inf": float("nan"), "rho_ic_sup": float("nan"),
               "n_rho": 0}
        met["no_mejor_que_azar"] = bool(auc_ic[0] == auc_ic[0] and auc_ic[0] <= 0.5)

        if spec["figura"] == "plano":
            p = d[y]
            niv = p["nivel_" + c["corto"]].to_numpy(float)
            val = p[dim + "__valencia"].to_numpy(float)
            met["rho"] = _rho(niv, val)
            met["rho_ic_inf"], met["rho_ic_sup"] = ic_bootstrap(_rho, niv, val)
            met["n_rho"] = int(len(p))

            fondo = d[~y][[col, dim + "__valencia"]].round(3).to_numpy().tolist()
            niveles, cajas, tabla_niv = {}, [], []
            a = d[~y]
            cajas.append({"nivel": "ausencia", "n": int(len(a)), "caja": _caja(a[col])})
            tabla_niv.append(["ausencia", int(len(a)), float(a[col].median()) if len(a) else None,
                              float((a[col] >= UMBRAL_RELEVANCIA).mean()) if len(a) else None,
                              float(a[dim + "__valencia"].median()) if len(a) else None])
            for k in [1, 2, 3, 4, 5]:
                g = p[p["nivel_" + c["corto"]] == k]
                niveles[str(k)] = [{
                    "x": round(float(r[col]), 4), "y": round(float(r[dim + "__valencia"]), 4),
                    "id": r["id"], "lang": r["lang"], "texto": str(r["texto"])[:220],
                    "trad": str(r["traduccion"])[:220],
                    "just": str(_just.at[r["id"], c["slug"]])[:260] if r["id"] in _just.index else ""}
                    for _, r in g.iterrows()]
                cajas.append({"nivel": str(k), "n": int(len(g)), "caja": _caja(g[col])})
                tabla_niv.append([str(k), int(len(g)),
                                  float(g[col].median()) if len(g) else None,
                                  float((g[col] >= UMBRAL_RELEVANCIA).mean()) if len(g) else None,
                                  float(g[dim + "__valencia"].median()) if len(g) else None])
            planos[c["slug"]] = {"etiqueta": c["etiqueta"], "dimension": dim, "fondo": fondo,
                                 "niveles": niveles, "cajas": cajas, "tabla": tabla_niv,
                                 "desajuste": spec["desajuste"]}
        else:
            from sklearn.metrics import roc_curve
            fpr, tpr, _ = roc_curve(y, s)
            paso = max(1, len(fpr) // 250)
            curva = [[round(float(a_), 4), round(float(b_), 4)]
                     for a_, b_ in list(zip(fpr, tpr))[::paso] + [(fpr[-1], tpr[-1])]]
            rocs[c["slug"]] = {"etiqueta": c["etiqueta"], "dimension": dim, "curva": curva,
                               "cajas": [{"clase": "ausente", "n": int((~y).sum()), "caja": _caja(s[~y])},
                                         {"clase": "presente", "n": int(y.sum()), "caja": _caja(s[y])}],
                               "presentes": [{"x": round(float(r[col]), 4), "id": r["id"],
                                              "texto": str(r["texto"])[:220],
                                              "just": str(_just.at[r["id"], c["slug"]])[:260]
                                              if r["id"] in _just.index else ""}
                                             for _, r in d[y].iterrows()],
                               "desajuste": spec["desajuste"]}
        filas_met.append(met)

    VALIDACION_METRICAS = pd.DataFrame(filas_met)

    print()
    print("  {:<24} {:>6} {:>5}   {:>18}   {:>20}".format(
        "criterio", "n", "pres", "AUC [IC 95%]", "rho nivel-val [IC]"))
    print("  " + "-" * 82)
    for m in filas_met:
        rho_txt = ("{:+.2f} [{:+.2f},{:+.2f}] n={}".format(m["rho"], m["rho_ic_inf"], m["rho_ic_sup"], m["n_rho"])
                   if m["rho"] == m["rho"] else "--")
        print("  {:<24} {:>6} {:>5}   {:.2f} [{:.2f}, {:.2f}]   {:>20}".format(
            m["criterio"], m["n"], m["presentes"], m["auc"], m["auc_ic_inf"], m["auc_ic_sup"], rho_txt))
        if m["no_mejor_que_azar"]:
            print("     ! el intervalo del AUC incluye 0.5 o queda debajo: no separa mejor que el azar.")
            print("       Constructo:", CORRESPONDENCIA[m["slug"]]["desajuste"])
    print()
    print("  Contraste entre DOS instrumentos automaticos (la rubrica es Gemini).")
    print("  No es validacion contra codificacion humana.")

    DATOS_VALIDACION = {
        "correspondencia": corresp, "metricas": filas_met, "planos": planos, "rocs": rocs,
        "umbral_relevancia": UMBRAL_RELEVANCIA, "n_minimo_nivel": N_MINIMO_NIVEL,
        "paleta_valencia": PALETA_VALENCIA, "color_presencia": COLOR_PRESENCIA,
        "color_ausencia": COLOR_AUSENCIA, "n_bootstrap": N_BOOTSTRAP,
        "diseno_relevancia": DISENO_RELEVANCIA,
    }

JS_VALIDACION = r"""
const D = DATOS;
const SIMB = ['triangle','diamond','rect','circle','roundRect'];
RAIZ_SEC.appendChild(el('<h2>Validación convergente · zero-shot contra la rúbrica</h2>'));
RAIZ_SEC.appendChild(el('<p class="banner"><b>Contraste entre dos instrumentos automáticos.</b> La rúbrica la aplicó '
  + 'un modelo de lenguaje (Gemini): que coincidan dice que convergen, no que acierten. '
  + 'NO sustituye la validación contra codificación humana.</p>'));
panel(RAIZ_SEC,{titulo:'Qué se contrasta con qué',
  pregunta:'Leer antes que cualquier número: un AUC bajo puede ser un instrumento malo o dos constructos distintos.',
  alto:0, html: tabla(['criterio','dimensión zero-shot','magnitud','qué mide la rúbrica','qué mide el zero-shot','hipótesis','desajuste'],
    D.correspondencia.map(c=>[c.criterio,c.dimension,c.magnitud,c.rubrica,c.zeroshot,c.hipotesis,c.desajuste]))});

const metr = Object.fromEntries(D.metricas.map(m=>[m.slug,m]));
const txtAuc = m => 'AUC '+fix(m.auc,2)+' ['+fix(m.auc_ic_inf,2)+', '+fix(m.auc_ic_sup,2)+']';
const txtRho = m => m.rho==null?'ρ —':'ρ nivel–valencia '+fix(m.rho,2)+' ['+fix(m.rho_ic_inf,2)+', '+fix(m.rho_ic_sup,2)+'] · n='+m.n_rho;

Object.entries(D.planos).forEach(([slug,P])=>{
  const m = metr[slug];
  const escasos = P.cajas.filter(k=>k.nivel!=='ausencia' && k.n>0 && k.n<D.n_minimo_nivel).map(k=>'nivel '+k.nivel+' (n='+k.n+')');
  const alertas = m.no_mejor_que_azar ? ['<b>No separa mejor que el azar.</b> El intervalo del AUC incluye 0.5 o queda por debajo. '+esc(P.desajuste)] : [];
  const p = panel(RAIZ_SEC,{titulo:esc(P.etiqueta)+' · plano relevancia–valencia',
    pregunta:'¿Los tuits con nivel &gt; 0 caen a la derecha, y la valencia sube con el nivel? Fondo gris: ausencia (nivel 0). '
      +'Derecha: relevancia por nivel.',
    chips:[txtAuc(m), txtRho(m), 'presentes '+m.presentes, 'ausencia '+m.ausentes, 'bloqueados excluidos '+m.faltantes_excluidos],
    alertas, avisos: escasos.length?['Niveles con menos de '+D.n_minimo_nivel+' tuits: '+escasos.join(', ')+'. Se dibujan, pero no son interpretables solos.']:[],
    alto:440, nota:'Constructo: '+esc(P.desajuste)+' La línea vertical marca UMBRAL_RELEVANCIA = '+D.umbral_relevancia+'.',
    tabla: tabla(['nivel','n','relevancia mediana','proporción ≥ umbral','valencia mediana'],
      P.tabla.map(f=>[f[0],f[1],f[2]==null?'—':f[2],f[3]==null?'—':pct(f[3]),f[4]==null?'—':f[4]]))});
  const series = [{name:'ausencia (nivel 0)',type:'scatter',xAxisIndex:0,yAxisIndex:0,data:P.fondo,symbolSize:3,
      large:true,largeThreshold:4000,itemStyle:{color:D.color_ausencia,opacity:.28},silent:true,
      markLine:{silent:true,symbol:'none',lineStyle:{color:C.tinta2,type:'dashed'},label:{formatter:'umbral',color:C.tinta2},
        data:[{xAxis:D.umbral_relevancia}]}}];
  ['1','2','3','4','5'].forEach((k,i)=>{
    if(!P.niveles[k] || !P.niveles[k].length) return;
    series.push({name:'nivel '+k,type:'scatter',xAxisIndex:0,yAxisIndex:0,symbol:SIMB[i],symbolSize:10,
      data:P.niveles[k].map(o=>({value:[o.x,o.y],o})),
      itemStyle:{color:D.paleta_valencia[i],borderColor:'#0b0b0b',borderWidth:.6,opacity:.92}});
  });
  const cats = P.cajas.map(k=>(k.nivel==='ausencia'?'ausencia':'nivel '+k.nivel)+' (n='+k.n+')');
  series.push({name:'relevancia por nivel',type:'boxplot',xAxisIndex:1,yAxisIndex:1,
    data:P.cajas.map((k,i)=>({value:k.caja||[null,null,null,null,null],
      itemStyle:{color:i===0?'#e1e0d9':D.paleta_valencia[i-1],borderColor:'#52514e'}})),
    tooltip:{formatter:q=>{const v=q.value; return esc(q.name)+'<br>mín '+fix(v[1],2)+' · Q1 '+fix(v[2],2)
      +' · mediana '+fix(v[3],2)+' · Q3 '+fix(v[4],2)+' · máx '+fix(v[5],2);}}});
  init(p.lienzo,{
    toolbox:TOOLBOX, legend:{top:0,left:0,textStyle:{color:C.tinta2,fontSize:11},
      data:series.filter(s=>s.type==='scatter').map(s=>s.name)},
    tooltip:{trigger:'item',confine:true,extraCssText:'max-width:420px;white-space:normal',
      formatter:q=>{ if(!q.data||!q.data.o) return q.seriesName;
        const o=q.data.o; return '<b>'+esc(q.seriesName)+'</b> · relevancia '+fix(o.x,2)+' · valencia '+fix(o.y,2)
          +'<br>'+esc(o.texto)+(o.trad?'<br><i>'+esc(o.trad)+'</i>':'')+'<br><span style="color:#52514e">rúbrica: '+esc(o.just)+'</span>'; }},
    grid:[{left:48,right:'45%',top:58,bottom:44},{left:'68%',right:14,top:58,bottom:44}],
    xAxis:[Object.assign({type:'value',min:0,max:1,name:'relevancia',nameLocation:'middle',nameGap:26,gridIndex:0},EJE_X),
           Object.assign({type:'value',min:0,max:1,name:'relevancia',nameLocation:'middle',nameGap:26,gridIndex:1},EJE_X)],
    yAxis:[Object.assign({type:'value',min:-1,max:1,name:'valencia',gridIndex:0},EJE_Y),
           Object.assign({type:'category',data:cats,gridIndex:1,inverse:true},EJE_X)],
    series});
});

Object.entries(D.rocs).forEach(([slug,R])=>{
  const m = metr[slug];
  const alertas = m.no_mejor_que_azar ? ['<b>No separa mejor que el azar.</b> '+esc(R.desajuste)] : [];
  const p = panel(RAIZ_SEC,{titulo:esc(R.etiqueta)+' · curva ROC contra '+esc(R.dimension),
    pregunta:'¿La prominencia de violencia del instrumento distingue a los tuits que la rúbrica marcó con violencia presente? '
      +'Criterio de presencia: no se dibuja en el plano de valencia.',
    chips:[txtAuc(m),'presentes '+m.presentes,'ausentes '+m.ausentes,'bloqueados excluidos '+m.faltantes_excluidos],
    alertas, alto:360, nota:'Constructo: '+esc(R.desajuste),
    tabla: tabla(['prominencia','id','texto','justificación de la rúbrica'],
      R.presentes.slice().sort((a,b)=>b.x-a.x).map(o=>[o.x,o.id,o.texto,o.just]))});
  init(p.lienzo,{toolbox:TOOLBOX,
    tooltip:{trigger:'item'},
    legend:{top:0,left:0,textStyle:{color:C.tinta2,fontSize:11}},
    grid:[{left:52,right:'54%',top:58,bottom:44},{left:'62%',right:14,top:58,bottom:44}],
    xAxis:[Object.assign({type:'value',min:0,max:1,name:'1 − especificidad',nameLocation:'middle',nameGap:26,gridIndex:0},EJE_X),
           Object.assign({type:'value',min:0,max:1,name:R.dimension,nameLocation:'middle',nameGap:26,gridIndex:1},EJE_X)],
    yAxis:[Object.assign({type:'value',min:0,max:1,name:'sensibilidad',gridIndex:0},EJE_Y),
           Object.assign({type:'category',data:R.cajas.map(k=>k.clase+' (n='+k.n+')'),gridIndex:1},EJE_X)],
    series:[
      {name:'ROC',type:'line',xAxisIndex:0,yAxisIndex:0,data:R.curva,showSymbol:false,lineStyle:{color:D.color_presencia,width:2},
       areaStyle:{color:'rgba(74,58,167,.08)'}},
      {name:'azar',type:'line',xAxisIndex:0,yAxisIndex:0,data:[[0,0],[1,1]],showSymbol:false,lineStyle:{color:C.mudo,type:'dashed'}},
      {name:'prominencia por clase',type:'boxplot',xAxisIndex:1,yAxisIndex:1,
       data:R.cajas.map((k,i)=>({value:k.caja||[null,null,null,null,null],itemStyle:{color:i?D.color_presencia:'#e1e0d9',borderColor:'#52514e'}}))},
      {name:'presentes',type:'scatter',xAxisIndex:1,yAxisIndex:1,symbolSize:7,
       data:R.presentes.map(o=>({value:[o.x,1],o})),itemStyle:{color:D.color_presencia,opacity:.6},
       tooltip:{formatter:q=>esc(q.data.o.texto)+'<br><span style="color:#52514e">'+esc(q.data.o.just)+'</span>'}}]});
});

panel(RAIZ_SEC,{titulo:'Resumen de métricas',alto:0,
  pregunta:'Intervalos al 95% por remuestreo ('+D.n_bootstrap+' réplicas, semilla fija). Diseño de relevancia: '+esc(D.diseno_relevancia)+'.',
  html: tabla(['criterio','dimensión','n','presentes','AUC','IC inf','IC sup','ρ nivel–valencia','IC inf','IC sup','n ρ','¿no mejor que azar?'],
    D.metricas.map(m=>[m.criterio,m.dimension,m.n,m.presentes,m.auc,m.auc_ic_inf,m.auc_ic_sup,
      m.rho==null?'—':m.rho,m.rho_ic_inf==null?'—':m.rho_ic_inf,m.rho_ic_sup==null?'—':m.rho_ic_sup,m.n_rho,m.no_mejor_que_azar?'sí':'no']))});
"""

if DATOS_VALIDACION is not None:
    registrar_seccion("validacion", "Validación convergente", DATOS_VALIDACION, JS_VALIDACION)
    PAGINA_VALIDACION = mostrar_seccion("validacion")

## 15. Asociación entre criterios

¿Los tuits que hablan de violencia **también** expresan emoción hacia México? ¿La imagen cultural
aparece junto a la perspectiva política? Se mide con la **presencia** de cada criterio (nivel > 0)
en tablas 2×2, una por cada uno de los 6 pares.

### Por qué Fisher y no chi cuadrada

Perspectiva política y violencia tienen 25 tuits presentes cada una sobre 2 600. En **todas** las
tablas alguna frecuencia esperada queda por debajo de 5, justo donde la aproximación de la chi
cuadrada deja de valer. La prueba principal es la **exacta de Fisher**, con corrección de **Holm**
por las 6 comparaciones. La chi cuadrada aparece solo como columna de referencia, marcada cuando
no es válida.

Con 2 600 tuits casi cualquier asociación sale «significativa». Lo que se lee es el **tamaño**:
**φ** (de −1 a 1) y la **razón de momios** con su intervalo. Cuando una tabla tiene un cero se
aplica la corrección de Haldane–Anscombe (+0.5 a cada celda), y se declara.

### Tres lecturas, porque cada una tiene su trampa

| lectura | pregunta | trampa |
|---|---|---|
| **corpus completo** | ¿los temas aparecen juntos? | ~2 480 tuits sin ningún criterio inflan la celda «ninguno» y con ella cualquier asociación |
| **solo calificados** | entre quienes tocan algún criterio, ¿uno arrastra a otro? | **sesgo de Berkson**: condicionar a «al menos uno presente» induce asociación **negativa** artificial |
| **estratificada por consulta** | ¿la asociación sobrevive dentro de cada tipo de búsqueda? | ~1 080 tuits vienen de consultas de victimización: sin estratificar, la consulta puede fabricar la asociación |

La estratificada usa **Mantel–Haenszel** (razón de momios común) y la prueba de homogeneidad de
**Breslow–Day** entre estratos.

### Un par asociado por definición

La rúbrica incluye «gestión de seguridad» dentro de **perspectiva política**, y **saliencia de
violencia** es violencia e inseguridad. Parte de su asociación está escrita en el instrumento, no en
el discurso. Por eso la figura lo anota.

In [ ]:
import itertools
import math
import warnings

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import StratifiedTable
from statsmodels.stats.multitest import multipletests

# ===========================================================================
# ASOCIACION ENTRE CRITERIOS
#
# Presencia binaria (nivel > 0) por criterio. Cada tabla 2x2 se ordena asi:
#
#                    B presente   B ausente
#     A presente        n11          n10
#     A ausente         n01          n00
#
# razon de momios = n11*n00 / (n10*n01)   ·   phi con el mismo signo
# ===========================================================================

# Regla de estratos para Mantel-Haenszel. Dos estratos a proposito: con una
# consulta por estrato hay estratos de 1 o 2 tuits y tablas enteras en cero.
PATRON_ESTRATO_VICTIMIZACION = "victim"      # se busca en minusculas dentro de `base`
ALFA = 0.05

# Pares cuya asociacion esta en parte escrita en la rubrica.
TRASLAPES_DEFINICIONALES = {
    frozenset({"perspectiva_politica", "violencia"}):
        "La definición de perspectiva política incluye la «gestión de seguridad», y saliencia de "
        "violencia es violencia, crimen o inseguridad: parte de la asociación está en el instrumento, "
        "no en el discurso.",
}


def _tabla(a, b):
    a = np.asarray(a, bool)
    b = np.asarray(b, bool)
    return np.array([[np.sum(a & b), np.sum(a & ~b)],
                     [np.sum(~a & b), np.sum(~a & ~b)]], dtype=float)


def medir_par(t):
    """Fisher, phi, razon de momios con IC de Woolf y chi cuadrada de referencia."""
    n11, n10, n01, n00 = t[0, 0], t[0, 1], t[1, 0], t[1, 1]
    n = t.sum()
    r1, r0, c1, c0 = n11 + n10, n01 + n00, n11 + n01, n10 + n00
    phi = ((n11 * n00 - n10 * n01) / math.sqrt(r1 * r0 * c1 * c0)) if min(r1, r0, c1, c0) > 0 else float("nan")
    corregida = bool((t == 0).any())
    tc = t + 0.5 if corregida else t
    or_ = (tc[0, 0] * tc[1, 1]) / (tc[0, 1] * tc[1, 0])
    se = math.sqrt((1 / tc).sum())
    ic = (math.exp(math.log(or_) - 1.96 * se), math.exp(math.log(or_) + 1.96 * se))
    p_fisher = float(fisher_exact(t.astype(int))[1])
    if min(r1, r0, c1, c0) > 0:
        chi2, p_chi, _, esperados = chi2_contingency(t, correction=False)
        esp_min = float(esperados.min())
    else:
        chi2, p_chi, esp_min = float("nan"), float("nan"), 0.0
    return {"n": int(n), "n11": int(n11), "n10": int(n10), "n01": int(n01), "n00": int(n00),
            "esperado_min": esp_min, "phi": phi, "or": or_, "or_ic_inf": ic[0], "or_ic_sup": ic[1],
            "correccion_haldane": corregida, "p_fisher": p_fisher,
            "chi2": float(chi2), "p_chi2": float(p_chi), "chi2_valida": bool(esp_min >= 5)}


def asociacion(df, lectura, estratos=None):
    filas = []
    cortos = [c["corto"] for c in CRITERIOS]
    for x, y in itertools.combinations(cortos, 2):
        ok = (df["clase_" + x] != "faltante") & (df["clase_" + y] != "faltante")
        d = df[ok]
        excluidos = int((~ok).sum())
        if lectura == "estratificada":
            tablas, tam = [], {}
            for nombre, g in d.groupby(estratos[ok]):
                tb = _tabla(g["presente_" + x], g["presente_" + y])
                tablas.append(tb)
                tam[str(nombre)] = int(tb.sum())
            fila = medir_par(sum(tablas))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                st = StratifiedTable([tb for tb in tablas], shift_zeros=True)
                ic = st.oddsratio_pooled_confint()
                fila.update({"or_mh": float(st.oddsratio_pooled), "or_mh_ic_inf": float(ic[0]),
                             "or_mh_ic_sup": float(ic[1]), "p_mh": float(st.test_null_odds().pvalue),
                             "p_homogeneidad": float(st.test_equal_odds().pvalue),
                             "estratos": tam})
        else:
            fila = medir_par(_tabla(d["presente_" + x], d["presente_" + y]))
        et = {c["corto"]: c["etiqueta"] for c in CRITERIOS}
        fila.update({"lectura": lectura, "a": x, "b": y, "par": "{} × {}".format(et[x], et[y]),
                     "excluidos_por_faltante": excluidos,
                     "nota": TRASLAPES_DEFINICIONALES.get(frozenset({x, y}), "")})
        filas.append(fila)
    r = pd.DataFrame(filas)
    col_p = "p_mh" if lectura == "estratificada" else "p_fisher"
    r["p_holm"] = multipletests(r[col_p].fillna(1.0), method="holm")[1]
    r["significativa_holm"] = r["p_holm"] < ALFA
    return r


def _p(v):
    """Un p que se desborda a 0 no es 0: es menor que lo representable."""
    return "<1e-300" if v == 0 else "{:.1e}".format(v)


ASOCIACION = None
DATOS_ASOCIACION = None
if not HAY_RUBRICA:
    print("Asociacion entre criterios OMITIDA: la entrada no trae rubrica.")
else:
    _T = RUBRICA_TUIT
    _estrato = np.where(_T["base"].astype(str).str.lower().str.contains(PATRON_ESTRATO_VICTIMIZACION),
                        "consultas de victimizacion", "resto de consultas")
    _estrato = pd.Series(_estrato, index=_T.index)
    _lecturas = {
        "corpus": asociacion(_T, "corpus"),
        "calificados": asociacion(_T[_T.calificado], "calificados"),
        "estratificada": asociacion(_T, "estratificada", _estrato),
    }
    ASOCIACION = pd.concat(_lecturas.values(), ignore_index=True)

    print("=" * 96)
    print("ASOCIACION ENTRE CRITERIOS (presencia = nivel > 0)")
    print("   estratos: " + " · ".join("{} = {}".format(k, v) for k, v in _estrato.value_counts().items())
          + "   (regla: '{}' dentro de `base`)".format(PATRON_ESTRATO_VICTIMIZACION))
    print("=" * 96)
    for nombre, r in _lecturas.items():
        print("\n  LECTURA: {}   n = {}".format(nombre, int(r["n"].max())))
        if nombre == "corpus":
            print("  ! los tuits sin ningun criterio presente llenan la celda n00 e inflan toda asociacion")
        if nombre == "calificados":
            print("  ! SESGO DE BERKSON: condicionar a 'algun criterio presente' induce asociacion NEGATIVA")
        cab = "  {:<46} {:>5} {:>5} {:>5} {:>6} {:>7} {:>6} {:>18} {:>9} {:>9}"
        print(cab.format("par", "n11", "n10", "n01", "n00", "esp.min", "phi", "OR [IC 95%]",
                         "p exacto" if nombre != "estratificada" else "p MH", "p Holm"))
        for _, f in r.iterrows():
            orr = (f["or_mh"], f["or_mh_ic_inf"], f["or_mh_ic_sup"]) if nombre == "estratificada" \
                else (f["or"], f["or_ic_inf"], f["or_ic_sup"])
            print("  {:<46} {:>5} {:>5} {:>5} {:>6} {:>7.1f} {:>+6.2f} {:>6.1f} [{:.1f}, {:.1f}]{} {:>9} {:>9}".format(
                f["par"][:46], f["n11"], f["n10"], f["n01"], f["n00"], f["esperado_min"], f["phi"],
                orr[0], orr[1], orr[2], "*" if f["correccion_haldane"] else " ",
                _p(f["p_mh"] if nombre == "estratificada" else f["p_fisher"]), _p(f["p_holm"])))
            if nombre == "estratificada":
                print("  {:<46} homogeneidad entre estratos (Breslow-Day) p = {:.2g}".format("", f["p_homogeneidad"]))
        print("  chi cuadrada solo de referencia: valida en {} de {} tablas (esperado minimo >= 5)".format(
            int(r["chi2_valida"].sum()), len(r)))
        print("  * = correccion de Haldane-Anscombe (+0.5) por celda en cero")
    for k, v in TRASLAPES_DEFINICIONALES.items():
        print("\n  NOTA", " × ".join(sorted(k)) + ":", v)

    DATOS_ASOCIACION = {
        "criterios": [{"corto": c["corto"], "etiqueta": c["etiqueta"]} for c in CRITERIOS],
        "lecturas": {k: v.replace({np.nan: None}).to_dict("records") for k, v in _lecturas.items()},
        "estratos": {str(k): int(v) for k, v in _estrato.value_counts().items()},
        "alfa": ALFA,
    }

JS_ASOCIACION = r"""
const D = DATOS;
const pexp = v => v==null?'—':(v===0?'<1e-300':Number(v).toExponential(2));
const ET = D.criterios.map(c=>c.etiqueta), CO = D.criterios.map(c=>c.corto);
const NOMBRES = {corpus:'corpus completo', calificados:'solo calificados', estratificada:'estratificada por consulta'};
const AVISOS = {
  corpus:'Los tuits sin ningún criterio presente llenan la celda «ninguno» e inflan toda asociación.',
  calificados:'<b>Sesgo de Berkson.</b> Condicionar a «algún criterio presente» induce asociación negativa artificial: un φ negativo aquí no prueba exclusión.',
  estratificada:'Mantel–Haenszel sobre dos estratos: '+Object.entries(D.estratos).map(([k,v])=>esc(k)+' ('+v+')').join(' · ')
    +'. La matriz muestra φ de la tabla combinada; la razón de momios y la p son las de Mantel–Haenszel.'};
RAIZ_SEC.appendChild(el('<h2>Asociación entre criterios</h2>'));
RAIZ_SEC.appendChild(el('<p class="sub">Presencia = nivel &gt; 0. Prueba exacta de Fisher con corrección de Holm; '
  +'la chi cuadrada solo como referencia. Calificaciones de un modelo de lenguaje, no codificación humana.</p>'));
let lectura = 'corpus';
const botones = Object.keys(D.lecturas).map(k=>'<button class="chip" data-l="'+k+'" aria-pressed="'+(k===lectura)+'">'+NOMBRES[k]+'</button>').join(' ');
const p = panel(RAIZ_SEC,{titulo:'Matriz de φ entre criterios',
  pregunta:'Cada celda es un par. * = significativa tras Holm (α = '+D.alfa+').',
  controles:'<div class="grupo"><span>Lectura</span>'+botones+'</div>', alto:380, html:'<p class="aviso av"></p><div class="tb"></div>'});
const g = init(p.lienzo,{});
p.seccion.querySelectorAll('button[data-l]').forEach(b=>b.onclick=()=>{lectura=b.dataset.l;
  p.seccion.querySelectorAll('button[data-l]').forEach(x=>x.setAttribute('aria-pressed',String(x.dataset.l===lectura)));pintar();});
function pintar(){
  const filas = D.lecturas[lectura];
  const datos = [];
  filas.forEach(f=>{const i=CO.indexOf(f.a), j=CO.indexOf(f.b);
    const txt=(f.phi==null?'—':f.phi.toFixed(2))+(f.significativa_holm?'*':'');
    datos.push({value:[j,i,f.phi],label:{formatter:txt},f}); datos.push({value:[i,j,f.phi],label:{formatter:txt},f});});
  g.setOption({toolbox:TOOLBOX,
    tooltip:{confine:true,extraCssText:'max-width:380px;white-space:normal',formatter:q=>{const f=q.data.f; const mh=lectura==='estratificada';
      return '<b>'+esc(f.par)+'</b><br>n11 '+f.n11+' · n10 '+f.n10+' · n01 '+f.n01+' · n00 '+f.n00
        +'<br>φ '+fix(f.phi,2)+' · OR '+fix(mh?f.or_mh:f.or,2)+' ['+fix(mh?f.or_mh_ic_inf:f.or_ic_inf,2)+', '+fix(mh?f.or_mh_ic_sup:f.or_ic_sup,2)+']'
        +'<br>p '+(mh?'MH ':'exacto ')+pexp(mh?f.p_mh:f.p_fisher)+' · Holm '+pexp(f.p_holm)
        +(f.nota?'<br><i>'+esc(f.nota)+'</i>':'');}},
    grid:{left:150,right:90,top:20,bottom:90},
    xAxis:Object.assign({type:'category',data:ET,axisLabel:{color:C.tinta2,rotate:20,interval:0}},EJE_X),
    yAxis:Object.assign({type:'category',data:ET,inverse:true},EJE_X),
    visualMap:{min:-1,max:1,calculable:false,orient:'vertical',right:0,top:'middle',text:['+1','−1'],
      inRange:{color:['#c62f2e','#f3dcdb','#fcfcfb','#d6e4f7','#1c5cab']},textStyle:{color:C.tinta2}},
    series:[{type:'heatmap',data:datos,label:{show:true,color:'#0b0b0b',fontSize:12},
      itemStyle:{borderColor:'#fcfcfb',borderWidth:2}}]},true);
  p.seccion.querySelector('.av').innerHTML = AVISOS[lectura];
  const mh = lectura==='estratificada';
  p.seccion.querySelector('.tb').innerHTML = tabla(
    ['par','n','n11','n10','n01','n00','esperado mín.','φ',mh?'OR MH':'OR','IC inf','IC sup','Haldane',mh?'p MH':'p exacto','p Holm',
     mh?'p homogeneidad':'χ² p (ref.)', mh?'estratos':'χ² válida','nota'],
    filas.map(f=>[f.par,f.n,f.n11,f.n10,f.n01,f.n00,f.esperado_min,f.phi,mh?f.or_mh:f.or,mh?f.or_mh_ic_inf:f.or_ic_inf,
      mh?f.or_mh_ic_sup:f.or_ic_sup,f.correccion_haldane?'sí':'no',pexp(mh?f.p_mh:f.p_fisher),
      pexp(f.p_holm), mh?pexp(f.p_homogeneidad):pexp(f.p_chi2),
      mh?Object.entries(f.estratos).map(([k,v])=>k+': '+v).join(' · '):(f.chi2_valida?'sí':'NO (esperado < 5)'), f.nota]));
}
pintar();
"""

if DATOS_ASOCIACION is not None:
    registrar_seccion("asociacion", "Asociación entre criterios", DATOS_ASOCIACION, JS_ASOCIACION)
    PAGINA_ASOCIACION = mostrar_seccion("asociacion")

## 16. Análisis NLP del corpus

La rúbrica calificó 144 tuits y dejó 2 487 sin nivel > 0 en ningún criterio. Esta sección mira el
95% que nadie ha leído con tres preguntas, y pone a los 144 en el mismo mapa para comparar.

```
 texto ORIGINAL ─▶ embeddings multilingües ─▶ UMAP ─▶ HDBSCAN ─▶ clusters (+ ruido)
                    (la traducción no deforma los grupos)             │
 TRADUCCIÓN (es) ─▶ spaCy + diccionario de México                     │
                    ├─ lemas ─▶ c-TF-IDF por cluster ◀────────────────┘ ─▶ TÓPICOS
                    ├─ entidades ─▶ co-ocurrencia con PMI ─▶ GRAFO de entidades
                    └─ dependencias ─▶ sujeto–verbo–objeto ─▶ GRAFO de relaciones
```

| lectura | pregunta | cómo |
|---|---|---|
| **auditoría de la colecta** | ¿qué trajo cada consulta que no es imagen de México? | cluster × consulta × período |
| **candidatos a falso negativo** | ¿hay tuits sobre México que la rúbrica dejó en 0? | no calificado **y** menciona México **y** (relevancia zero-shot ≥ umbral **o** prominencia de violencia ≥ umbral) |
| **temas fuera de la rúbrica** | ¿de qué más se habla de México? | clusters con mucha mención de México y casi ningún calificado |

### Tres decisiones y su porqué

**Los clusters salen del texto original, no de la traducción.** Un modelo multilingüe pone el inglés
y el japonés en el mismo espacio, así que los errores de la traducción automática no deciden qué
tuits quedan juntos.

**spaCy trabaja sobre la traducción.** Da un solo vocabulario y un solo reconocedor de entidades para
todo el corpus: «Estadio Azteca» es un nodo del grafo y no tres. **Esto es un uso nuevo de la
traducción**, que EvaluadorTweets produjo como apoyo de lectura humana, y hay que declararlo. **Cinco
traducciones** (tuits que Gemini bloqueó) se completaron a mano con otro modelo el 13/09/2026.

**Un diccionario de México va antes que el reconocedor estadístico.** El NER en español solo tiene
cuatro etiquetas y falla con tuits. El diccionario —celda editable abajo— fija las entidades que
importan y es la señal de «menciona México» para los candidatos a falso negativo. Revísalo antes de
leer la lista de candidatos.

### Lo que esta sección NO dice

- Un **candidato a falso negativo no es un error de la rúbrica**. Es un tuit que merece que una
  persona lo lea. Por eso no se calcula ninguna tasa.
- Los **grafos son exploratorios**. Las dependencias sintácticas sobre traducciones de tuits son
  ruidosas: sirven para orientar la lectura, no para reportar relaciones.
- Los **clusters dependen de parámetros** (`HDBSCAN_MIN_CLUSTER`, `UMAP_VECINOS`). Con la semilla fija
  son reproducibles, pero otros valores dan otros grupos: todos quedan en la ficha técnica.

In [ ]:
# === BLOQUE: configuracion-nlp ===
# ===========================================================================
# CONFIGURACION DEL ANALISIS NLP
# ===========================================================================

# --- modelos ------------------------------------------------------------------
# Embeddings multilingues sobre el texto ORIGINAL. mpnet es el mejor de la
# familia paraphrase-multilingual; MiniLM (mas chico) sirve si falta memoria.
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
LOTE_EMBEDDINGS = 64
# spaCy sobre la TRADUCCION. Si cambias el modelo, esta seccion lo descarga.
MODELO_SPACY = "es_core_news_lg"
LOTE_SPACY = 64
# Los modelos NLI ya no se usan despues de la seccion 11: liberarlos deja la
# memoria de GPU para los embeddings.
LIBERAR_MODELOS_NLI = True

SEMILLA_NLP = 42

# --- texto util ---------------------------------------------------------------
# Tras quitar enlaces y menciones, un texto con menos letras que esto no dice
# nada que agrupar (son tuits que solo traen un enlace o una imagen).
MIN_LETRAS_TEXTO_UTIL = 3

# --- reduccion y agrupamiento --------------------------------------------------
UMAP_VECINOS = 15            # mas alto -> estructura mas global, clusters mas grandes
UMAP_DIM_CLUSTER = 10        # dimensiones para agrupar; la figura usa 2
UMAP_MIN_DIST = 0.0
HDBSCAN_MIN_CLUSTER = 15     # tamano minimo de un cluster
HDBSCAN_MIN_SAMPLES = 5      # mas alto -> mas tuits declarados ruido

# --- topicos y grafos ----------------------------------------------------------
TOP_TERMINOS = 10
TOP_NODOS = 80               # nodos maximos por grafo: mas se vuelve ilegible
MIN_COOCURRENCIA = 3         # tuits en que dos entidades aparecen juntas
MIN_PMI = 0.0                # informacion mutua puntual minima de una arista
MIN_TRIPLETA = 2             # repeticiones minimas de una relacion sujeto-verbo-objeto
TOP_RELACIONES = 60
MIN_FRECUENCIA_LOGODDS = 3

# --- nubes de palabras ---------------------------------------------------------
# Frecuencia minima de un lema en el PRIOR de cada nube (la union de los lados
# que se comparan) para entrar en ella. Mismo papel que MIN_FRECUENCIA_LOGODDS,
# con perilla propia porque el prior de las nubes de polaridad son 140 tuits y
# no 2 631: el mismo numero no filtra lo mismo.
MIN_FRECUENCIA_NUBE = 3
# Terminos como maximo por nube. Por encima de ~120 la nube deja de leerse y
# empieza a ser una textura.
TOP_TERMINOS_NUBE = 120
# Por debajo de esto la nube se dibuja con AVISO: no es comparable con las
# demas. El grupo ambivalente cae aqui a proposito.
MIN_TERMINOS_NUBE = 25
# Ocurrencias minimas de un termino DENTRO de su grupo para dibujarlo.
#
# MIN_FRECUENCIA_NUBE filtra sobre el PRIOR (los dos lados juntos), que es lo
# que el estimador necesita; pero un termino puede pasar ese filtro con una
# sola aparicion en el grupo que se dibuja. Medido sobre el corpus: sin este
# umbral la nube ambivalente sale con 39 terminos, 22 de ellos de UNA sola
# ocurrencia en sus 19 tuits, y queda por encima de MIN_TERMINOS_NUBE, asi que
# el aviso de escasez no se disparaba nunca. Con 3:
#     positivo 65 -> 50 terminos     negativo 44 -> 17 (avisa)
#     ambivalente 39 -> 8 (avisa)
MIN_SOPORTE_NUBE = 3

# Trazado DETERMINISTA. wordcloud2 coloca por espiral sobre el lienzo: con el
# alto fijo, la rejilla fija y la rotacion apagada, la misma entrada da siempre
# la misma figura. Sin esto la nube se recompone al redimensionar la ventana y
# deja de poder citarse.
NUBE_ALTO = 380
NUBE_REJILLA = 8
NUBE_ROTACION = False

# --- lecturas analiticas ---------------------------------------------------------
# Umbral de prominencia de violencia para candidatos a falso negativo. SE FIJA
# MIRANDO LA ROC DE LA SECCION 14 Y SE CONGELA ANTES DE LISTAR CANDIDATOS: no se
# mueve segun cuantos candidatos salgan. La celda de lecturas imprime el punto
# de operacion que implica antes de listar.
UMBRAL_SALIENCE_FN = 0.50
# La relevancia usa UMBRAL_RELEVANCIA de la seccion 7.

# Temas fuera de la rubrica: clusters donde se habla de Mexico y casi nadie
# fue calificado.
UMBRAL_TEMA_MENCION_MEXICO = 0.30     # proporcion minima de tuits que mencionan Mexico
UMBRAL_TEMA_CALIFICADOS = 0.05        # proporcion maxima de calificados
MIN_TAMANO_TEMA = 15

# ---------------------------------------------------------------------------
# DICCIONARIO DE MEXICO. EDITABLE: revisalo antes de leer los candidatos a
# falso negativo, porque es la senal de "menciona Mexico".
#
#   categoria -> cuenta_como_mexico: si su mencion basta para decir que el
#                tuit habla de Mexico. El crimen organizado y el torneo NO:
#                "cartel" o "world cup" no localizan un tuit en Mexico.
#                terminos: {nombre canonico: [variantes]}
#
# Las variantes se buscan en MINUSCULAS en la traduccion (con spaCy) y en el
# texto original (por expresion regular). Por eso van en espanol, ingles y
# japones. La etiqueta de la entidad es "MX_" + categoria en mayusculas.
# ---------------------------------------------------------------------------
MEXICO = {
    "pais": {"cuenta_como_mexico": True, "terminos": {
        "México": ["méxico", "mexico", "méjico", "メキシコ"],
        "mexicanos": ["mexicano", "mexicana", "mexicanos", "mexicanas", "mexican", "mexicans",
                      "メキシコ人"],
    }},
    "ciudad_sede": {"cuenta_como_mexico": True, "terminos": {
        "Ciudad de México": ["ciudad de méxico", "ciudad de mexico", "cdmx", "mexico city",
                             "méxico df", "メキシコシティ"],
        "Guadalajara": ["guadalajara", "グアダラハラ"],
        "Monterrey": ["monterrey", "モンテレイ", "モンテレー"],
    }},
    "destino": {"cuenta_como_mexico": True, "terminos": {
        "Cancún": ["cancún", "cancun", "カンクン"],
        "Tulum": ["tulum", "トゥルム"],
        "Playa del Carmen": ["playa del carmen"],
        "Riviera Maya": ["riviera maya"],
        "Los Cabos": ["los cabos", "cabo san lucas", "ロスカボス"],
        "Puerto Vallarta": ["puerto vallarta", "vallarta"],
        "Acapulco": ["acapulco"],
        "Oaxaca": ["oaxaca", "オアハカ"],
        "Chichén Itzá": ["chichén itzá", "chichen itza", "チチェン・イッツァ"],
        "Teotihuacán": ["teotihuacán", "teotihuacan", "テオティワカン"],
        "Tijuana": ["tijuana", "ティフアナ"],
        "Quintana Roo": ["quintana roo"],
        "Jalisco": ["jalisco"],
        "Nuevo León": ["nuevo león", "nuevo leon"],
        "Sinaloa": ["sinaloa"],
        "Yucatán": ["yucatán", "yucatan", "ユカタン"],
        "Chiapas": ["chiapas"],
        "Veracruz": ["veracruz"],
        "Puebla": ["puebla"],
        "San Miguel de Allende": ["san miguel de allende"],
        "Baja California": ["baja california"],
    }},
    "estadio": {"cuenta_como_mexico": True, "terminos": {
        "Estadio Azteca": ["estadio azteca", "azteca stadium", "estadio banorte",
                           "estadio ciudad de méxico", "アステカ"],
        "Estadio Akron": ["estadio akron", "estadio guadalajara"],
        "Estadio BBVA": ["estadio bbva", "estadio monterrey"],
    }},
    "institucion": {"cuenta_como_mexico": True, "terminos": {
        "Guardia Nacional": ["guardia nacional", "national guard of mexico"],
        "SEDENA": ["sedena"],
        "SEMAR": ["semar"],
        "FGR": ["fgr", "fiscalía general de la república"],
        "INM": ["instituto nacional de migración"],
        "Profeco": ["profeco"],
        "Pemex": ["pemex"],
    }},
    "persona_publica": {"cuenta_como_mexico": True, "terminos": {
        "Claudia Sheinbaum": ["claudia sheinbaum", "sheinbaum", "シェインバウム"],
        "López Obrador": ["lópez obrador", "lopez obrador", "amlo"],
        "Clara Brugada": ["clara brugada"],
    }},
    "crimen_organizado": {"cuenta_como_mexico": False, "terminos": {
        "Cártel de Sinaloa": ["cártel de sinaloa", "cartel de sinaloa", "sinaloa cartel"],
        "CJNG": ["cjng", "cártel jalisco nueva generación", "jalisco new generation cartel"],
        "cártel": ["cártel", "cartel", "cárteles", "carteles", "cartels", "narco", "narcos"],
    }},
    "torneo": {"cuenta_como_mexico": False, "terminos": {
        "Mundial 2026": ["mundial", "copa del mundo", "copa mundial", "world cup", "ワールドカップ"],
        "FIFA": ["fifa"],
        "Selección Mexicana": ["selección mexicana", "seleccion mexicana", "el tri"],
    }},
}

# Expresiones que contienen una variante pero NO son Mexico. Se borran del texto
# antes de buscar: sin esto, "New Mexico" contaria como mencion de Mexico.
EXCLUSIONES_MEXICO = ["new mexico", "nuevo méxico", "nuevo mexico", "golfo de méxico",
                      "gulf of mexico", "golfo de america", "gulf of america"]

_n_term = sum(len(v["terminos"]) for v in MEXICO.values())
_n_var = sum(len(x) for v in MEXICO.values() for x in v["terminos"].values())
print("diccionario MEXICO: {} categorias · {} entidades · {} variantes".format(len(MEXICO), _n_term, _n_var))
for _k, _v in MEXICO.items():
    print("   {:<18} {:>3} entidades   cuenta como Mexico: {}".format(
        _k, len(_v["terminos"]), "si" if _v["cuenta_como_mexico"] else "no"))
print("umbral de prominencia para candidatos a FN: {}  (congelado)".format(UMBRAL_SALIENCE_FN))

In [ ]:
# === BLOQUE: preparacion-texto ===
import gc
import re
import importlib.metadata as _md_nlp

import numpy as np
import pandas as pd

# ===========================================================================
# PREPARACION Y AGRUPAMIENTO
#
#   1. limpieza minima y texto util
#   2. embeddings multilingues del texto ORIGINAL
#   3. UMAP: 10 dimensiones para agrupar, 2 para dibujar
#   4. HDBSCAN: clusters y ruido (-1), renumerados por tamano
#
# Un solo ajuste para todo el corpus: calificados y no calificados comparten
# espacio, clusters y topicos.
# ===========================================================================

NLP_TUIT = None
FICHA_NLP = {}

_URL = re.compile(r"https?://\S+|www\.\S+", re.I)
_MENCION = re.compile(r"(?<!\w)@\w+")
_LETRA = re.compile(r"[^\W\d_]", re.U)


def limpiar(texto):
    """Enlaces fuera y menciones fuera: no dicen de que habla el tuit y en
    las respuestas dominan el principio del texto."""
    t = _URL.sub(" ", str(texto or ""))
    t = _MENCION.sub(" ", t)
    t = t.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    return re.sub(r"\s+", " ", t).strip()


def _version(paquete):
    try:
        return _md_nlp.version(paquete)
    except _md_nlp.PackageNotFoundError:
        return None


if not HAY_RUBRICA:
    print("Analisis NLP OMITIDO: sin rubrica no hay grupos que comparar.")
else:
    _orig = RUBRICA_ORIGEN_DF
    NLP_TUIT = RUBRICA_TUIT[["id", "grupo", "calificado", "con_faltante", "polaridad", "lang",
                             "idioma", "base", "periodo", "fecha_str"]].copy()
    NLP_TUIT["texto"] = _orig[COL_TEXTO].astype(str).values
    NLP_TUIT["traduccion"] = (_orig[COL_TRADUCCION].astype(str).values
                              if COL_TRADUCCION in _orig else "")
    NLP_TUIT["texto_limpio"] = NLP_TUIT["texto"].map(limpiar)
    NLP_TUIT["traduccion_limpia"] = NLP_TUIT["traduccion"].map(limpiar)
    NLP_TUIT["texto_util"] = NLP_TUIT["texto_limpio"].map(lambda t: len(_LETRA.findall(t)) >= MIN_LETRAS_TEXTO_UTIL)

    print("=" * 78)
    print("TEXTO UTIL (tras quitar enlaces y menciones, al menos {} letras)".format(MIN_LETRAS_TEXTO_UTIL))
    for g, sub in NLP_TUIT.groupby("grupo"):
        print("   {:<14} {:>5} tuits · {:>4} sin texto util, excluidos del agrupamiento".format(
            g, len(sub), int((~sub.texto_util).sum())))
    print("=" * 78)


In [ ]:
# === BLOQUE: embeddings-y-clusters ===
# Desde aqui se necesita GPU (o mucha paciencia): embeddings, UMAP y
# HDBSCAN. El script de frecuencias NO ejecuta esta celda.
if NLP_TUIT is not None:
    # --- liberar la GPU ---------------------------------------------------
    if LIBERAR_MODELOS_NLI:
        for _nombre in ("nli_pipe", "sentiment_pipe"):
            if globals().get(_nombre) is not None:
                globals()[_nombre] = None
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    # --- embeddings ---------------------------------------------------------
    import torch
    from sentence_transformers import SentenceTransformer

    _util = NLP_TUIT[NLP_TUIT.texto_util].reset_index(drop=True)
    _disp = "cuda" if torch.cuda.is_available() else "cpu"
    print("embeddings: {} sobre {} textos originales ({})...".format(MODELO_EMBEDDINGS, len(_util), _disp))
    _modelo_emb = SentenceTransformer(MODELO_EMBEDDINGS, device=_disp)
    EMBEDDINGS = _modelo_emb.encode(_util["texto_limpio"].tolist(), batch_size=LOTE_EMBEDDINGS,
                                    show_progress_bar=False, normalize_embeddings=True,
                                    convert_to_numpy=True)
    del _modelo_emb
    gc.collect()

    # --- UMAP ---------------------------------------------------------------
    try:
        import umap
        _reductor = "umap-learn " + str(_version("umap-learn"))
        _r10 = umap.UMAP(n_neighbors=UMAP_VECINOS, n_components=UMAP_DIM_CLUSTER, min_dist=UMAP_MIN_DIST,
                         metric="cosine", random_state=SEMILLA_NLP).fit_transform(EMBEDDINGS)
        _r2 = umap.UMAP(n_neighbors=UMAP_VECINOS, n_components=2, min_dist=0.1,
                        metric="cosine", random_state=SEMILLA_NLP).fit_transform(EMBEDDINGS)
    except ImportError:
        from sklearn.decomposition import PCA
        print("   AVISO: umap-learn no esta instalado; se usa PCA. Los clusters seran peores y la")
        print("   ficha tecnica lo registra.")
        _reductor = "PCA (respaldo: umap-learn no disponible)"
        _r10 = PCA(n_components=UMAP_DIM_CLUSTER, random_state=SEMILLA_NLP).fit_transform(EMBEDDINGS)
        _r2 = PCA(n_components=2, random_state=SEMILLA_NLP).fit_transform(EMBEDDINGS)

    # --- HDBSCAN ------------------------------------------------------------
    try:
        import hdbscan
        _agr = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER, min_samples=HDBSCAN_MIN_SAMPLES,
                               metric="euclidean", cluster_selection_method="eom").fit(_r10)
        _agrupador = "hdbscan " + str(_version("hdbscan"))
    except ImportError:
        from sklearn.cluster import HDBSCAN as _SkHDBSCAN
        _agr = _SkHDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER, min_samples=HDBSCAN_MIN_SAMPLES).fit(_r10)
        _agrupador = "sklearn.cluster.HDBSCAN (respaldo) · scikit-learn " + str(_version("scikit-learn"))
    _etq = np.asarray(_agr.labels_)
    _prob = np.asarray(getattr(_agr, "probabilities_", np.ones(len(_etq))))

    # Renumerar por tamano: el cluster 0 es el mas grande. El ruido sigue en -1.
    _tam = pd.Series(_etq[_etq >= 0]).value_counts()
    _mapa = {viejo: nuevo for nuevo, viejo in enumerate(_tam.index)}
    _mapa[-1] = -1
    _util["cluster"] = [_mapa[int(e)] for e in _etq]
    _util["prob_cluster"] = _prob
    _util["umap_x"] = _r2[:, 0]
    _util["umap_y"] = _r2[:, 1]
    UMAP10 = _r10

    NLP_TUIT = NLP_TUIT.merge(_util[["id", "cluster", "prob_cluster", "umap_x", "umap_y"]], on="id", how="left")
    NLP_TUIT["cluster"] = NLP_TUIT["cluster"].fillna(-2).astype(int)   # -2 = sin texto util
    IDS_UTIL = _util["id"].tolist()

    FICHA_NLP = {
        "modelo_embeddings": MODELO_EMBEDDINGS, "dispositivo": _disp,
        "sentence_transformers": _version("sentence-transformers"),
        "reductor": _reductor, "agrupador": _agrupador, "semilla": SEMILLA_NLP,
        "umap_vecinos": UMAP_VECINOS, "umap_dim_cluster": UMAP_DIM_CLUSTER, "umap_min_dist": UMAP_MIN_DIST,
        "hdbscan_min_cluster": HDBSCAN_MIN_CLUSTER, "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
        "min_letras_texto_util": MIN_LETRAS_TEXTO_UTIL,
    }

    _n_cl = int(_tam.shape[0])
    print()
    print("AGRUPAMIENTO: {} clusters · reductor {} · {}".format(_n_cl, _reductor, _agrupador))
    for g, sub in NLP_TUIT[NLP_TUIT.cluster >= -1].groupby("grupo"):
        print("   {:<14} ruido {:>5.1%}   ({} de {} con texto util)   en {} clusters distintos".format(
            g, (sub.cluster == -1).mean(), int((sub.cluster == -1).sum()), len(sub),
            sub.loc[sub.cluster >= 0, "cluster"].nunique()))
    print("   tamanos:", ", ".join("C{}={}".format(_mapa[k], v) for k, v in _tam.items()))


In [ ]:
# === BLOQUE: spacy-lemas-entidades ===
import importlib
import itertools
import math
import re
import subprocess
import sys
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# ===========================================================================
# spaCy SOBRE LA TRADUCCION: entidades, lemas, dependencias
#
# El diccionario MEXICO entra como EntityRuler ANTES del reconocedor
# estadistico: sus entidades quedan fijas y el NER solo completa el resto.
# ===========================================================================

POS_LEMAS = {"NOUN", "PROPN", "ADJ"}
DEP_SUJETO = {"nsubj", "nsubj:pass"}
DEP_OBJETO = {"obj", "iobj", "obl", "obl:arg"}
PREFIJO_MX = "MX_"


def cargar_spacy(nombre):
    import spacy
    if not spacy.util.is_package(nombre):
        print("descargando el modelo de spaCy {}...".format(nombre))
        subprocess.run([sys.executable, "-m", "spacy", "download", nombre], check=True)
        importlib.invalidate_caches()
    nlp = spacy.load(nombre)
    patrones = []
    for cat, spec in MEXICO.items():
        for canon, variantes in spec["terminos"].items():
            for v in variantes:
                patrones.append({"label": PREFIJO_MX + cat.upper(), "pattern": v, "id": canon})
    ruler = nlp.add_pipe("entity_ruler", before="ner",
                         config={"phrase_matcher_attr": "LOWER", "overwrite_ents": True})
    ruler.add_patterns(patrones)
    return nlp, len(patrones)


def _sin_exclusiones(texto):
    t = texto
    for e in EXCLUSIONES_MEXICO:
        t = re.sub(re.escape(e), " ", t, flags=re.I)
    return t


def _regex_mexico():
    """Las variantes de las categorias que cuentan como Mexico, para buscarlas
    en el texto ORIGINAL (ingles y japones no pasan por spaCy)."""
    latinas, cjk = [], []
    for spec in MEXICO.values():
        if not spec["cuenta_como_mexico"]:
            continue
        for variantes in spec["terminos"].values():
            for v in variantes:
                (latinas if re.search(r"[a-záéíóúñü]", v) else cjk).append(v)
    patron_lat = re.compile(r"(?<!\w)(" + "|".join(re.escape(v) for v in sorted(latinas, key=len, reverse=True))
                            + r")(?!\w)", re.I)
    return patron_lat, cjk


def _normalizar_entidad(ent):
    if ent.ent_id_:
        return ent.ent_id_
    t = re.sub(r"\s+", " ", ent.text.strip(" .,:;!?¡¿\"'#()[]«»“”"))
    return t


NLP_DOCS = None
FICHA_SPACY = {}
if NLP_TUIT is not None:
    import spacy

    nlp_es, _n_pat = cargar_spacy(MODELO_SPACY)
    _cats_mx = {PREFIJO_MX + k.upper() for k, v in MEXICO.items() if v["cuenta_como_mexico"]}
    _patron_lat, _cjk = _regex_mexico()

    _textos = [_sin_exclusiones(t) for t in NLP_TUIT["traduccion_limpia"]]
    ents_tuit, lemas_tuit, tripletas_tuit, mx_trad = [], [], [], []
    # La categoria gramatical se filtraba y se tiraba. Guardarla en paralelo
    # no cuesta nada y separa lo que EVALUA (ADJ) de lo que nombra el tema
    # (NOUN) y de los lugares y nombres propios (PROPN).
    lemas_pos_tuit = []
    print("spaCy {} ({}) sobre {} traducciones...".format(MODELO_SPACY, spacy.__version__, len(_textos)))
    for doc in nlp_es.pipe(_textos, batch_size=LOTE_SPACY):
        ents = []
        tok_ent = {}
        for e in doc.ents:
            clave = _normalizar_entidad(e)
            if len(clave) < 2 or not re.search(r"[^\W\d_]", clave):
                continue
            ents.append((clave, e.label_))
            for tk in e:
                tok_ent[tk.i] = clave
        ents_tuit.append(ents)
        mx_trad.append(any(lab in _cats_mx for _, lab in ents))
        _lemas = [t for t in doc
                  if t.pos_ in POS_LEMAS and t.is_alpha and not t.is_stop and len(t.text) > 2]
        lemas_tuit.append([t.lemma_.lower() for t in _lemas])
        lemas_pos_tuit.append([t.pos_ for t in _lemas])

        def termino(tk):
            if tk.i in tok_ent:
                return tok_ent[tk.i], True
            if tk.pos_ in ("NOUN", "PROPN") and tk.is_alpha and not tk.is_stop:
                return tk.lemma_.lower(), False
            return None, False

        trip = []
        for tk in doc:
            if tk.pos_ != "VERB":
                continue
            sujetos = [c for c in tk.children if c.dep_ in DEP_SUJETO]
            objetos = [c for c in tk.children if c.dep_ in DEP_OBJETO]
            for s_, o_ in itertools.product(sujetos, objetos):
                ts, es_s = termino(s_)
                to, es_o = termino(o_)
                if ts and to and ts != to and (es_s or es_o):
                    trip.append((ts, tk.lemma_.lower(), to))
        tripletas_tuit.append(trip)

    mx_orig = []
    for t in NLP_TUIT["texto"]:
        t2 = _sin_exclusiones(t)
        mx_orig.append(bool(_patron_lat.search(t2)) or any(v in t2 for v in _cjk))

    NLP_TUIT["entidades"] = ents_tuit
    NLP_TUIT["lemas"] = lemas_tuit
    NLP_TUIT["lemas_pos"] = lemas_pos_tuit      # paralelo a "lemas", misma longitud
    NLP_TUIT["tripletas"] = tripletas_tuit
    NLP_TUIT["menciona_mexico"] = np.array(mx_trad) | np.array(mx_orig)

    FICHA_SPACY = {"modelo_spacy": MODELO_SPACY, "spacy": spacy.__version__,
                   "patrones_diccionario": _n_pat, "exclusiones": EXCLUSIONES_MEXICO}

    print("   entidades reconocidas: {} · patrones del diccionario: {}".format(
        sum(len(e) for e in ents_tuit), _n_pat))
    print("   mencionan Mexico (diccionario en la traduccion o en el original):")
    for g, sub in NLP_TUIT.groupby("grupo"):
        print("      {:<14} {:>5} de {:>5}  ({:.1%})".format(g, int(sub.menciona_mexico.sum()), len(sub),
                                                         sub.menciona_mexico.mean()))
    print("   solo por el original: {} · solo por la traduccion: {}".format(
        int((np.array(mx_orig) & ~np.array(mx_trad)).sum()), int((np.array(mx_trad) & ~np.array(mx_orig)).sum())))


In [ ]:
# === BLOQUE: vocabulario-grafos-nubes ===
# Nada de aqui depende de los clusters: solo de los lemas, las entidades
# y la polaridad. Por eso el script de frecuencias puede ejecutar esta
# celda sin GPU, saltandose los embeddings.
if NLP_TUIT is not None:
    # ------------------------------------------------------------------------
    # GRAFO DE CO-OCURRENCIA DE ENTIDADES, uno por grupo
    #   PMI(a, b) = log( p(a,b) / (p(a) p(b)) ), probabilidades sobre tuits con entidades
    # ------------------------------------------------------------------------
    def grafo_entidades(df):
        conj = [set(k for k, _ in lst) for lst in df["entidades"]]
        conj = [s for s in conj if s]
        N = len(conj) or 1
        freq = Counter(itertools.chain.from_iterable(conj))
        etiqueta = {}
        for lst in df["entidades"]:
            for k, lab in lst:
                etiqueta.setdefault(k, Counter())[lab] += 1
        nodos = [k for k, _ in freq.most_common(TOP_NODOS)]
        dentro = set(nodos)
        pares = Counter()
        for s in conj:
            for a, b in itertools.combinations(sorted(s & dentro), 2):
                pares[(a, b)] += 1
        aristas = []
        for (a, b), n in pares.items():
            if n < MIN_COOCURRENCIA:
                continue
            pmi = math.log((n / N) / ((freq[a] / N) * (freq[b] / N)))
            if pmi >= MIN_PMI:
                aristas.append({"source": a, "target": b, "n": n, "pmi": round(pmi, 3)})
        conectados = {x for e in aristas for x in (e["source"], e["target"])}
        return {"nodos": [{"id": k, "n": int(freq[k]), "cat": etiqueta[k].most_common(1)[0][0],
                           "aislado": k not in conectados} for k in nodos],
                "aristas": aristas, "tuits_con_entidades": len(conj)}

    GRAFOS_ENTIDADES = {"todos": grafo_entidades(NLP_TUIT),
                        "calificado": grafo_entidades(NLP_TUIT[NLP_TUIT.calificado]),
                        "no_calificado": grafo_entidades(NLP_TUIT[~NLP_TUIT.calificado])}

    # ------------------------------------------------------------------------
    # GRAFO DE RELACIONES sujeto -> verbo -> objeto (con al menos una entidad)
    # ------------------------------------------------------------------------
    def grafo_relaciones(df):
        cnt = Counter(itertools.chain.from_iterable(df["tripletas"]))
        cnt = [(t, n) for t, n in cnt.most_common() if n >= MIN_TRIPLETA][:TOP_RELACIONES]
        nodos = Counter()
        for (s_, v, o_), n in cnt:
            nodos[s_] += n
            nodos[o_] += n
        ents_conocidas = set(k for lst in df["entidades"] for k, _ in lst)
        return {"nodos": [{"id": k, "n": int(n), "entidad": k in ents_conocidas} for k, n in nodos.items()],
                "aristas": [{"source": s_, "target": o_, "verbo": v, "n": int(n)} for (s_, v, o_), n in cnt]}

    GRAFOS_RELACIONES = {"todos": grafo_relaciones(NLP_TUIT),
                         "calificado": grafo_relaciones(NLP_TUIT[NLP_TUIT.calificado]),
                         "no_calificado": grafo_relaciones(NLP_TUIT[~NLP_TUIT.calificado])}

    # ------------------------------------------------------------------------
    # LOG-ODDS CON PRIOR INFORMATIVO (Monroe, Colaresi y Quinn, 2008)
    #   prior = frecuencias de los dos lados juntos; z > 0 = caracteristico del
    #   primer lado, z < 0 = del segundo.
    #
    # Los ROTULOS de los dos lados son parametros. El estimador no sabe que
    # compara y no debe saberlo: la seccion de las nubes lo usa para polaridad,
    # y un retorno con las llaves "calificados"/"no_calificados" cableadas haria
    # que la pagina dijera "calificados" donde quiere decir "positivo".
    # ------------------------------------------------------------------------
    def log_odds(cnt1, cnt2, lado1="calificados", lado2="no_calificados",
                 minimo=MIN_FRECUENCIA_LOGODDS, top=20):
        prior = cnt1 + cnt2
        a0 = sum(prior.values())
        n1, n2 = sum(cnt1.values()), sum(cnt2.values())
        filas = []
        for w, aw in prior.items():
            if aw < minimo:
                continue
            y1, y2 = cnt1.get(w, 0), cnt2.get(w, 0)
            d = (math.log((y1 + aw) / (n1 + a0 - y1 - aw)) - math.log((y2 + aw) / (n2 + a0 - y2 - aw)))
            z = d / math.sqrt(1 / (y1 + aw) + 1 / (y2 + aw))
            filas.append({"termino": w, "z": round(z, 3),
                          "n_" + lado1: int(y1), "n_" + lado2: int(y2)})
        # Desempate alfabetico: dos terminos con la misma z no deben
        # intercambiarse entre corridas, que es lo que pide el trazado
        # determinista de las nubes.
        r = sorted(filas, key=lambda f: (-f["z"], f["termino"]))
        return {"lados": [lado1, lado2],
                lado1: r[:top],
                lado2: list(reversed(r[-top:]))}

    _cal, _nocal = NLP_TUIT[NLP_TUIT.calificado], NLP_TUIT[~NLP_TUIT.calificado]
    LOG_ODDS = {
        "lemas": log_odds(Counter(itertools.chain.from_iterable(_cal["lemas"])),
                          Counter(itertools.chain.from_iterable(_nocal["lemas"])),
                          "calificados", "no_calificados"),
        "entidades": log_odds(Counter(k for lst in _cal["entidades"] for k, _ in set(lst)),
                              Counter(k for lst in _nocal["entidades"] for k, _ in set(lst)),
                              "calificados", "no_calificados"),
    }

    # ------------------------------------------------------------------------
    # NUBES DE PALABRAS
    #
    #   descriptivas   peso = frecuencia de lema        (corpus, calificados)
    #   de polaridad   peso = z de log-odds uno contra  (positivo, negativo,
    #                  el resto, prior = los tuits       ambivalente)
    #                  CON polaridad
    #
    # Dos pesos porque son dos preguntas. Con 19 a 76 tuits la frecuencia cruda
    # mide sobre todo el tamano del grupo, y ademas "Mexico" esta en todos los
    # grupos por construccion: con z se encoge solo, sin lista negra que
    # mantener.
    # ------------------------------------------------------------------------
    def _cuenta_lemas(sub):
        return Counter(itertools.chain.from_iterable(sub["lemas"]))

    def _ficha_nube(clave, universo, peso, n_tuits, terminos):
        return {"clave": clave, "universo": universo, "peso": peso,
                "n_tuits": int(n_tuits), "n_terminos": len(terminos),
                "escasa": len(terminos) < MIN_TERMINOS_NUBE,
                "terminos": terminos}

    def nube_frecuencia(sub, clave, universo):
        cnt = _cuenta_lemas(sub)
        t = [(w, int(n)) for w, n in cnt.items() if n >= MIN_FRECUENCIA_NUBE]
        # Desempate alfabetico: el trazado tiene que ser el mismo cada corrida.
        t.sort(key=lambda p: (-p[1], p[0]))
        return _ficha_nube(clave, universo, "frecuencia", len(sub),
                           [[w, n] for w, n in t[:TOP_TERMINOS_NUBE]])

    def nube_logodds(sub, resto, clave, universo):
        r = log_odds(_cuenta_lemas(sub), _cuenta_lemas(resto), clave, "resto",
                     minimo=MIN_FRECUENCIA_NUBE, top=TOP_TERMINOS_NUBE)
        # Solo lo CARACTERISTICO del grupo (z > 0) y con SOPORTE en el grupo.
        # Sin el soporte, mas de la mitad de la nube ambivalente serian
        # terminos de una sola aparicion dibujados como si dijeran algo.
        t = [[f["termino"], f["z"], f["n_" + clave], f["n_resto"]]
             for f in r[clave] if f["z"] > 0 and f["n_" + clave] >= MIN_SOPORTE_NUBE]
        return _ficha_nube(clave, universo, "z", len(sub), t)

    _POLOS_NUBE = ("positivo", "negativo", "ambivalente")
    _con_polaridad = NLP_TUIT[NLP_TUIT.polaridad.isin(_POLOS_NUBE)]

    NUBES = {
        "descriptivas": [
            nube_frecuencia(NLP_TUIT, "corpus", "todos los tuits del corpus"),
            nube_frecuencia(NLP_TUIT[NLP_TUIT.calificado], "calificados",
                            "tuits con algun criterio de la rubrica en nivel > 0"),
        ],
        "polaridad": [
            nube_logodds(_con_polaridad[_con_polaridad.polaridad == p],
                         _con_polaridad[_con_polaridad.polaridad != p], p,
                         "contra los otros {} tuits con polaridad".format(
                             len(_con_polaridad) - int((_con_polaridad.polaridad == p).sum())))
            for p in _POLOS_NUBE
        ],
        "n_con_polaridad": int(len(_con_polaridad)),
        "n_sin_polaridad": int((NLP_TUIT.polaridad == "sin_polaridad").sum()),
    }

    # Reparto de los criterios de PRESENCIA por polaridad. Se deriva de la
    # escala declarada, nunca del nombre del criterio. La nota del panel lo usa
    # para ENSENAR que presencia no es valencia en vez de afirmarlo: de los 25
    # tuits con violencia saliente, 4 son positivos o ambivalentes.
    NUBES["reparto_presencia"] = [
        {"criterio": c["etiqueta"],
         "n": int(RUBRICA_TUIT["presente_" + c["corto"]].sum()),
         "por_polaridad": {p: int((RUBRICA_TUIT.loc[RUBRICA_TUIT["presente_" + c["corto"]],
                                                    "polaridad"] == p).sum())
                           for p in POLARIDADES}}
        for c in CRITERIOS if c["escala"] == "presencia"]
    print()
    print("GRAFOS: entidades {} nodos / {} aristas · relaciones {} aristas (todos los tuits)".format(
        len(GRAFOS_ENTIDADES["todos"]["nodos"]), len(GRAFOS_ENTIDADES["todos"]["aristas"]),
        len(GRAFOS_RELACIONES["todos"]["aristas"])))
    print("LOG-ODDS · lemas mas caracteristicos de los calificados:",
          ", ".join(f["termino"] for f in LOG_ODDS["lemas"]["calificados"][:10]))
    print()
    print("NUBES DE PALABRAS (lemas de la traduccion, frecuencia minima {})".format(
        MIN_FRECUENCIA_NUBE))
    for _n in NUBES["descriptivas"] + NUBES["polaridad"]:
        print("   {:<14} {:>5} tuits · {:>4} terminos · peso {}{}".format(
            _n["clave"], _n["n_tuits"], _n["n_terminos"], _n["peso"],
            "   ESCASA (< {})".format(MIN_TERMINOS_NUBE) if _n["escasa"] else ""))
    for _r in NUBES["reparto_presencia"]:
        print("   {} presente en {} tuits: {}".format(
            _r["criterio"], _r["n"],
            ", ".join("{} {}".format(v, k) for k, v in _r["por_polaridad"].items() if v)))


In [ ]:
# === BLOQUE: topicos-por-cluster ===
# Esto SI depende de los clusters de la celda de embeddings.
if NLP_TUIT is not None:
    # ------------------------------------------------------------------------
    # TOPICOS POR CLUSTER: c-TF-IDF sobre lemas
    #   tf(w, c)  = frecuencia de w en el cluster / palabras del cluster
    #   idf(w)    = log(1 + A / frecuencia total de w),  A = palabras medias por cluster
    # ------------------------------------------------------------------------
    _en_cl = NLP_TUIT[NLP_TUIT.cluster >= -1]
    _conteos = {c: Counter(itertools.chain.from_iterable(g["lemas"])) for c, g in _en_cl.groupby("cluster")}
    _tot_w = Counter()
    for cnt in _conteos.values():
        _tot_w.update(cnt)
    _A = np.mean([sum(c.values()) for c in _conteos.values()]) if _conteos else 1.0

    def _ctfidf(cnt):
        n = sum(cnt.values()) or 1
        return sorted(((w, (f / n) * math.log(1 + _A / _tot_w[w])) for w, f in cnt.items()),
                      key=lambda p: -p[1])

    CLUSTERS = []
    for c, g in _en_cl.groupby("cluster"):
        terminos = [w for w, _ in _ctfidf(_conteos[c])[:TOP_TERMINOS]]
        ents = Counter(k for lst in g["entidades"] for k, _ in set(lst)).most_common(8)
        ej = g.sort_values("prob_cluster", ascending=False).head(4)
        CLUSTERS.append({
            "cluster": int(c), "n": int(len(g)), "calificados": int(g.calificado.sum()),
            "prop_calificados": float(g.calificado.mean()), "prop_mexico": float(g.menciona_mexico.mean()),
            "terminos": terminos, "topico": ("ruido · " if c == -1 else "") + ", ".join(terminos[:4]),
            "entidades": [[k, int(v)] for k, v in ents],
            "consultas": [[str(k), int(v)] for k, v in g["base"].value_counts().head(4).items()],
            "periodos": {str(k): int(v) for k, v in g["periodo"].value_counts().items()},
            "idiomas": {str(k): int(v) for k, v in g["lang"].value_counts().head(4).items()},
            "ejemplos": [{"id": r["id"], "texto": r["texto"][:240], "trad": r["traduccion"][:240]}
                         for _, r in ej.iterrows()],
        })
    _topico = {c["cluster"]: c["topico"] for c in CLUSTERS}
    NLP_TUIT["topico"] = NLP_TUIT["cluster"].map(_topico).fillna("")
    print()
    print("TOPICOS POR CLUSTER (c-TF-IDF sobre lemas de la traduccion)")
    for c in CLUSTERS[:30]:
        print("   C{:<3} n={:<5} calif {:>5.1%} · Mexico {:>5.1%} · {}".format(
            c["cluster"], c["n"], c["prop_calificados"], c["prop_mexico"], ", ".join(c["terminos"][:7])))


In [ ]:
import numpy as np
import pandas as pd

# ===========================================================================
# TRES LECTURAS ANALITICAS
#
#   1. auditoria de la colecta      cluster x consulta x periodo
#   2. candidatos a falso negativo  para REVISION MANUAL, nunca una tasa
#   3. temas fuera de la rubrica    clusters con Mexico y sin calificados
# ===========================================================================

AUDITORIA = None
CANDIDATOS_FN = None
TEMAS_NUEVOS = None

if NLP_TUIT is not None:
    # --- 1 · auditoria de la colecta ------------------------------------------
    _a = NLP_TUIT.copy()
    _a["cluster_txt"] = _a["cluster"].map(lambda c: "sin texto útil" if c == -2 else ("ruido" if c == -1 else "C{}".format(c)))
    AUDITORIA = (_a.groupby(["cluster", "cluster_txt", "base", "periodo"], dropna=False)
                   .agg(n=("id", "size"), calificados=("calificado", "sum"),
                        mencionan_mexico=("menciona_mexico", "sum"))
                   .reset_index().sort_values(["cluster", "n"], ascending=[True, False]))
    _por_consulta = (_a.groupby("base").agg(n=("id", "size"), calificados=("calificado", "mean"),
                                             mexico=("menciona_mexico", "mean"),
                                             ruido=("cluster", lambda s: float((s < 0).mean())))
                       .sort_values("n", ascending=False))
    print("=" * 90)
    print("AUDITORIA DE LA COLECTA · por consulta")
    print("  {:<52} {:>6} {:>11} {:>9} {:>14}".format("consulta (base)", "tuits", "calificados", "Mexico", "ruido/sin txt"))
    for b, f in _por_consulta.iterrows():
        print("  {:<52} {:>6} {:>10.1%} {:>9.1%} {:>13.1%}".format(str(b)[:52], int(f["n"]), f["calificados"],
                                                                 f["mexico"], f["ruido"]))

    # --- 2 · candidatos a falso negativo ----------------------------------------
    print()
    print("=" * 90)
    print("CANDIDATOS A FALSO NEGATIVO")
    if VALIDACION is None:
        print("   OMITIDO: no hay valores zero-shot unidos a la rubrica (seccion 14).")
    else:
        _dims_rel = [c for c in VALIDACION.columns if c.endswith("__relevancia")]
        _v = NLP_TUIT.merge(VALIDACION[["id"] + _dims_rel +
                                       [c for c in VALIDACION.columns if c == "violence_salience__valencia"]],
                            on="id", how="left")
        # Punto de operacion del umbral de prominencia, ANTES de listar: la
        # proporcion de presentes y de ausentes de R4 que lo superan.
        _slug_v = next((c["corto"] for c in CRITERIOS if c["escala"] == "presencia"), None)
        if _slug_v and "violence_salience__valencia" in _v:
            _vv = _v.merge(RUBRICA_TUIT[["id", "presente_" + _slug_v, "clase_" + _slug_v]], on="id")
            _vv = _vv[_vv["clase_" + _slug_v] != "faltante"]
            _sup = _vv["violence_salience__valencia"] >= UMBRAL_SALIENCE_FN
            print("   umbral de prominencia congelado en {}: lo supera el {:.1%} de los tuits con violencia "
                  "presente en la rubrica y el {:.1%} de los ausentes (ver la ROC de la seccion 14).".format(
                      UMBRAL_SALIENCE_FN, _sup[_vv["presente_" + _slug_v]].mean(),
                      _sup[~_vv["presente_" + _slug_v]].mean()))

        def _motivos(r):
            m = []
            for c in _dims_rel:
                if pd.notna(r[c]) and r[c] >= UMBRAL_RELEVANCIA:
                    m.append("relevancia {} {:.2f}".format(c.replace("__relevancia", ""), r[c]))
            if "violence_salience__valencia" in r and pd.notna(r["violence_salience__valencia"]) \
                    and r["violence_salience__valencia"] >= UMBRAL_SALIENCE_FN:
                m.append("prominencia de violencia {:.2f}".format(r["violence_salience__valencia"]))
            return "; ".join(m)

        _base = _v[(~_v.calificado) & _v.menciona_mexico].copy()
        _base["motivo_fn"] = _base.apply(_motivos, axis=1)
        CANDIDATOS_FN = _base[_base.motivo_fn != ""].copy()
        CANDIDATOS_FN["relevancia_max"] = CANDIDATOS_FN[_dims_rel].max(axis=1)
        CANDIDATOS_FN = CANDIDATOS_FN.sort_values("relevancia_max", ascending=False)
        _just = RUBRICA.pivot_table(index="id", columns="corto", values="justificacion", aggfunc="first")
        _just.columns = ["justificacion_" + c for c in _just.columns]
        CANDIDATOS_FN = CANDIDATOS_FN.merge(_just.reset_index(), on="id", how="left")
        NLP_TUIT["candidato_fn"] = NLP_TUIT["id"].isin(set(CANDIDATOS_FN["id"]))
        NLP_TUIT["motivo_fn"] = NLP_TUIT["id"].map(CANDIDATOS_FN.set_index("id")["motivo_fn"]).fillna("")
        print("   no calificados que mencionan Mexico: {} · de ellos candidatos: {}".format(
            len(_base), len(CANDIDATOS_FN)))
        print("   {} con algun criterio bloqueado (su calificacion esta incompleta)".format(
            int(CANDIDATOS_FN.con_faltante.sum())))
        print("   Son CANDIDATOS PARA REVISION MANUAL, no errores de la rubrica: no se calcula ninguna tasa.")
        for _, r in CANDIDATOS_FN.head(8).iterrows():
            print("   · [{}] {} -> {}".format(r["id"], r["texto"][:90].replace("\n", " "), r["motivo_fn"]))

    if "candidato_fn" not in NLP_TUIT:
        NLP_TUIT["candidato_fn"] = False
        NLP_TUIT["motivo_fn"] = ""

    # --- 3 · temas fuera de la rubrica ------------------------------------------
    print()
    print("=" * 90)
    print("TEMAS FUERA DE LA RUBRICA (Mexico >= {:.0%}, calificados <= {:.0%}, n >= {})".format(
        UMBRAL_TEMA_MENCION_MEXICO, UMBRAL_TEMA_CALIFICADOS, MIN_TAMANO_TEMA))
    TEMAS_NUEVOS = [c for c in CLUSTERS
                    if c["cluster"] >= 0 and c["n"] >= MIN_TAMANO_TEMA
                    and c["prop_mexico"] >= UMBRAL_TEMA_MENCION_MEXICO
                    and c["prop_calificados"] <= UMBRAL_TEMA_CALIFICADOS]
    if not TEMAS_NUEVOS:
        print("   ningun cluster cumple los dos umbrales")
    for c in TEMAS_NUEVOS:
        print("   C{:<3} n={:<4} Mexico {:>5.1%} · calificados {:>5.1%} · {}".format(
            c["cluster"], c["n"], c["prop_mexico"], c["prop_calificados"], ", ".join(c["terminos"][:8])))
    print("   Candidatos a dimension: se leen con sus ejemplos antes de proponer nada.")

In [ ]:
# ===========================================================================
# FIGURAS DEL ANALISIS NLP
# ===========================================================================

DATOS_NLP = None
if NLP_TUIT is not None:
    _pts = NLP_TUIT[NLP_TUIT.cluster >= -1]
    _audit_tree = {}
    for _, f in AUDITORIA[AUDITORIA.cluster >= -1].iterrows():
        _audit_tree.setdefault(f["cluster_txt"], {}).setdefault(str(f["base"]), {})[str(f["periodo"])] = int(f["n"])

    _cols_fn = ["id", "lang", "base", "periodo", "texto", "traduccion", "motivo_fn", "cluster", "topico",
                "con_faltante"]
    DATOS_NLP = {
        "ficha": dict(FICHA_NLP, **FICHA_SPACY, top_nodos=TOP_NODOS, min_coocurrencia=MIN_COOCURRENCIA,
                      min_pmi=MIN_PMI, min_tripleta=MIN_TRIPLETA, umbral_relevancia=UMBRAL_RELEVANCIA,
                      umbral_salience_fn=UMBRAL_SALIENCE_FN, umbral_tema_mexico=UMBRAL_TEMA_MENCION_MEXICO,
                      umbral_tema_calificados=UMBRAL_TEMA_CALIFICADOS, min_tamano_tema=MIN_TAMANO_TEMA,
                      min_frecuencia_nube=MIN_FRECUENCIA_NUBE, top_terminos_nube=TOP_TERMINOS_NUBE,
                      min_terminos_nube=MIN_TERMINOS_NUBE, min_soporte_nube=MIN_SOPORTE_NUBE,
                      nube_alto=NUBE_ALTO,
                      nube_rejilla=NUBE_REJILLA, nube_rotacion=NUBE_ROTACION),
        # Colores DE LA PALETA declarada, no inventados. Rojo/ambar/azul es el
        # mismo orden divergente que usan los niveles de la rubrica.
        "nubes": dict(NUBES, colores={"positivo": PALETA["actual"], "negativo": PALETA["neg"],
                                      "ambivalente": PALETA["medio"],
                                      "descriptiva": PALETA["tinta2"]}),
        "grupos": {g: {"n": int(len(s)), "texto_util": int(s.texto_util.sum()),
                       "ruido": int((s.cluster == -1).sum()), "sin_texto": int((s.cluster == -2).sum()),
                       "mexico": int(s.menciona_mexico.sum())}
                   for g, s in NLP_TUIT.groupby("grupo")},
        "puntos": [[round(float(r.umap_x), 3), round(float(r.umap_y), 3), int(r.cluster), 1 if r.calificado else 0,
                    r.id, r.texto[:160], r.traduccion[:160]] for r in _pts.itertuples()],
        "clusters": CLUSTERS,
        "auditoria": _audit_tree,
        "grafos_entidades": GRAFOS_ENTIDADES,
        "grafos_relaciones": GRAFOS_RELACIONES,
        "log_odds": LOG_ODDS,
        "candidatos": (CANDIDATOS_FN[[c for c in _cols_fn if c in CANDIDATOS_FN]].head(300)
                       .assign(texto=lambda d: d.texto.str[:260], traduccion=lambda d: d.traduccion.str[:260])
                       .to_dict("records") if CANDIDATOS_FN is not None else None),
        "n_candidatos": int(len(CANDIDATOS_FN)) if CANDIDATOS_FN is not None else None,
        "temas": TEMAS_NUEVOS,
        "categorias_mx": ["MX_" + k.upper() for k in MEXICO],
    }

JS_NLP = r"""
const D = DATOS;
const PAL = ['#2a78d6','#eb6834','#1baf7a','#eda100','#e87ba4','#4a3aa7','#0d9488','#b45309','#7c3aed','#be123c',
             '#0369a1','#65a30d','#c2410c','#9333ea','#0f766e'];
RAIZ_SEC.appendChild(el('<h2>Análisis NLP del corpus</h2>'));
RAIZ_SEC.appendChild(el('<p class="banner"><b>Traducción automática.</b> Tópicos, entidades y relaciones salen de la '
  + 'traducción al español que EvaluadorTweets produjo como apoyo de lectura humana (5 filas completadas a mano con otro '
  + 'modelo). Los clusters salen del texto original. Grupos definidos por la rúbrica, que es un modelo de lenguaje.</p>'));
RAIZ_SEC.appendChild(el('<p class="sub">'+Object.entries(D.grupos).map(([g,v])=>'<b>'+esc(g)+'</b>: '+v.n+' tuits · '
  +v.texto_util+' con texto útil · '+v.ruido+' ruido · '+v.mexico+' mencionan México').join(' &nbsp;|&nbsp; ')+'</p>'));

/* ---- nubes de palabras ---- */
/* Las nubes van PRIMERO: son la entrada barata a la seccion. El mapa UMAP
   exige entender que es un embedding antes de ver nada. */
const NB = D.nubes;

function nube(o){
  const vacia = !o.terminos.length;
  const p = panel(RAIZ_SEC,{titulo:o.titulo, pregunta:o.pregunta, chips:o.chips,
    alto: vacia ? 0 : D.ficha.nube_alto, avisos:o.avisos||[], nota:o.nota});
  if(vacia) return p;
  /* Sin la extension no se dibuja nada, pero el panel dice por que en vez de
     dejar un lienzo en blanco. */
  if(!window.__HAY_NUBE__){ p.lienzo.innerHTML = '<p class="aviso">'+window.__AVISO_NUBE__+'</p>'; return p; }
  const max = Math.max.apply(null, o.terminos.map(t=>t[1]));
  init(p.lienzo,{toolbox:TOOLBOX,
    tooltip:{formatter:q=>esc(q.name)+' · '+o.rotulo+' '+(o.decimales?fix(q.value,o.decimales):q.value)},
    series:[{type:'wordCloud', shape:'circle', drawOutOfBound:false,
      gridSize:D.ficha.nube_rejilla, sizeRange:[12,56],
      /* Determinista: rotacion apagada y rejilla fija, o la figura se
         recompone al redimensionar y deja de poder citarse. */
      rotationRange:D.ficha.nube_rotacion?[-90,90]:[0,0], rotationStep:90,
      width:'96%', height:'92%', left:'center', top:'center',
      data:o.terminos.map(t=>({name:t[0], value:t[1],
        textStyle:{color:o.color, fontWeight:t[1]>=max*0.6?600:400}}))}]});
  return p;
}

function chipsNube(n){
  const c=[n.n_tuits.toLocaleString('es')+' tuits', n.n_terminos+' términos'];
  if(n.escasa) c.push('vocabulario escaso');
  return c;
}
function avisosNube(n){
  if(!n.n_terminos) return ['Omitida: ningún lema alcanza la frecuencia mínima ('
    +D.ficha.min_frecuencia_nube+') en este grupo. '+n.n_tuits+' tuits no dan para una nube.'];
  if(n.escasa) return ['<b>Vocabulario escaso.</b> '+n.n_tuits+' tuits y '+n.n_terminos
    +' términos, por debajo de los '+D.ficha.min_terminos_nube+' declarados: <b>no es comparable</b> con las otras nubes.'];
  return [];
}

const TRAD_NUBE = 'Los términos son lemas de spaCy sobre la <b>traducción automática</b> al español, '
  + 'que es el único espacio donde un corpus multilingüe es comparable.';

RAIZ_SEC.appendChild(el('<h3 style="font-size:14px;margin:18px 0 2px">Nubes descriptivas · el tamaño es la frecuencia del lema</h3>'));
RAIZ_SEC.appendChild(el('<p class="nota" style="margin:0 0 10px">'+TRAD_NUBE+'</p>'));

const nDesc = Object.fromEntries(NB.descriptivas.map(n=>[n.clave,n]));
nube({titulo:'Nube del corpus completo', terminos:nDesc.corpus.terminos, color:NB.colores.descriptiva,
  rotulo:'aparece', chips:chipsNube(nDesc.corpus), avisos:avisosNube(nDesc.corpus),
  pregunta:'De qué habla el corpus tal como lo trajo la colecta, con los no calificados incluidos.'});
nube({titulo:'Nube de los tuits calificados', terminos:nDesc.calificados.terminos, color:NB.colores.descriptiva,
  rotulo:'aparece', chips:chipsNube(nDesc.calificados), avisos:avisosNube(nDesc.calificados),
  pregunta:'Qué reconoció la rúbrica: sólo los tuits con algún criterio en nivel &gt; 0.'});
RAIZ_SEC.appendChild(el('<p class="nota" style="margin:-6px 0 4px">La diferencia entre las dos es la distancia entre '
  + '<b>lo que la consulta pescó</b> y <b>lo que la rúbrica reconoció</b>. El color no codifica nada: es tinta neutra a propósito.</p>'));

RAIZ_SEC.appendChild(el('<h3 style="font-size:14px;margin:22px 0 2px">Nubes de polaridad · el tamaño es la z de log-odds</h3>'));
RAIZ_SEC.appendChild(el('<p class="nota" style="margin:0 0 10px">Log-odds con prior informativo (Monroe, Colaresi y Quinn, 2008), '
  + 'cada grupo contra el resto de los '+NB.n_con_polaridad+' tuits con polaridad. <b>No es frecuencia:</b> con grupos de 19 a 76 tuits '
  + 'la frecuencia cruda mide sobre todo el tamaño del grupo. Un término repartido por igual entre los grupos —«México», sin ir más lejos— '
  + 'obtiene z cercana a cero y se encoge solo. '+TRAD_NUBE+'</p>'));

NB.polaridad.forEach(n=>{
  nube({titulo:'Vocabulario característico · '+n.clave, terminos:n.terminos.map(t=>[t[0],t[1]]),
    color:NB.colores[n.clave]||NB.colores.descriptiva, rotulo:'z', decimales:2,
    chips:chipsNube(n), avisos:avisosNube(n),
    pregunta:'Lemas con z &gt; 0 para el grupo <b>'+esc(n.clave)+'</b>, '+esc(n.universo)+'.'});
});

/* Nota de los tuits sin polaridad: no la afirma, la ENSEÑA con el reparto. */
(function(){
  const filas = NB.reparto_presencia.map(r=>{
    const partes = Object.entries(r.por_polaridad).filter(([k,v])=>v).map(([k,v])=>v+' '+k.replace(/_/g,' '));
    return '<b>'+esc(r.criterio)+'</b> está presente en '+r.n+' tuits: '+partes.join(', ')+'.';
  }).join(' ');
  panel(RAIZ_SEC,{titulo:'Los tuits que no entran en ninguna nube de polaridad', alto:0,
    chips:[NB.n_sin_polaridad+' sin polaridad'],
    pregunta:'Tuits calificados <b>sólo</b> por un criterio de presencia. Quedan fuera de las tres nubes y de su prior.',
    html:'<p>'+filas+'</p>'
      + '<p><b>Presencia no es valencia.</b> Que la mayoría de los tuits con violencia saliente sean negativos es '
      + 'correlación, no definición: los que no lo son dicen que México es maravilloso <i>a pesar de</i> o <i>si evitas</i> '
      + 'el peligro. Si la presencia sumara a la polaridad, esos tuits se arrastrarían a negativo, que sería falso.</p>'});
})();

/* ---- mapa ---- */
const nCl = D.clusters.filter(c=>c.cluster>=0).length;
const topCl = D.clusters.filter(c=>c.cluster>=0).slice(0,PAL.length).map(c=>c.cluster);
const nombreCl = c => c===-1?'ruido':'C'+c;
const topicoDe = Object.fromEntries(D.clusters.map(c=>[c.cluster,c.topico]));
const pm = panel(RAIZ_SEC,{titulo:'Mapa del corpus · UMAP de los embeddings del texto original',
  pregunta:'Cada punto es un tuit; el color es su cluster. Los <b>calificados</b> llevan un rombo con borde negro: ¿viven en '
    +'clusters propios o dispersos entre el resto?',
  chips:[nCl+' clusters','los '+topCl.length+' más grandes con color propio; el resto en gris claro'],alto:560});
const series=[];
const grupoCl = {};
D.puntos.forEach(p=>{ const k = p[2]===-1?'ruido':(topCl.includes(p[2])?nombreCl(p[2]):'otros clusters');
  (grupoCl[k]=grupoCl[k]||[]).push(p); });
Object.entries(grupoCl).forEach(([k,ps])=>{
  const i = topCl.indexOf(Number(k.slice(1)));
  series.push({name:k+(i>=0?' · '+topicoDe[Number(k.slice(1))]:''),type:'scatter',symbolSize:4,large:ps.length>2000,
    data:ps.filter(p=>!p[3]).map(p=>({value:[p[0],p[1]],p})),
    itemStyle:{color:k==='ruido'?'#c3c2b7':(i>=0?PAL[i]:'#dcdad2'),opacity:k==='ruido'?.45:.75}});
});
series.push({name:'calificados (rúbrica)',type:'scatter',symbol:'diamond',symbolSize:9,
  data:D.puntos.filter(p=>p[3]).map(p=>({value:[p[0],p[1]],p})),
  itemStyle:{color:p=>{const c=p.data.p[2]; const i=topCl.indexOf(c); return c===-1?'#c3c2b7':(i>=0?PAL[i]:'#dcdad2');},
    borderColor:'#0b0b0b',borderWidth:1.2}});
init(pm.lienzo,{toolbox:TOOLBOX,
  legend:{type:'scroll',orient:'vertical',right:0,top:10,bottom:10,textStyle:{color:C.tinta2,fontSize:11},width:240},
  tooltip:{confine:true,extraCssText:'max-width:420px;white-space:normal',formatter:q=>{const p=q.data.p;
    return '<b>'+nombreCl(p[2])+'</b> · '+esc(topicoDe[p[2]]||'')+(p[3]?' · <b>calificado</b>':'')+'<br>'+esc(p[5])
      +(p[6]?'<br><i>'+esc(p[6])+'</i>':'');}},
  grid:{left:10,right:260,top:10,bottom:10},
  xAxis:{type:'value',show:false,scale:true}, yAxis:{type:'value',show:false,scale:true}, series});

/* ---- tópicos ---- */
panel(RAIZ_SEC,{titulo:'Tópicos por cluster',alto:0,
  pregunta:'Términos más distintivos de cada cluster (c-TF-IDF sobre lemas de la traducción), con las consultas que lo aportan y ejemplos.',
  html: '<div style="max-height:620px;overflow:auto">'+tabla(['cluster','n','calificados','menciona México','términos distintivos','entidades','consultas principales','ejemplos'],
    D.clusters.map(c=>[nombreCl(c.cluster),c.n,pct(c.prop_calificados),pct(c.prop_mexico),c.terminos.join(', '),
      c.entidades.map(e=>e[0]+' ('+e[1]+')').join(', '),c.consultas.map(q=>q[0]+' ('+q[1]+')').join(' · '),
      c.ejemplos.slice(0,2).map(e=>e.texto).join('  ‖  ')]))+'</div>'});

/* ---- auditoría ---- */
const tree = Object.entries(D.auditoria).map(([cl,bases])=>({name:cl,
  children:Object.entries(bases).map(([b,pers])=>({name:b,children:Object.entries(pers).map(([p,n])=>({name:p,value:n}))}))}));
const clInfo = Object.fromEntries(D.clusters.map(c=>[nombreCl(c.cluster),c]));
const pa = panel(RAIZ_SEC,{titulo:'Auditoría de la colecta · cluster → consulta → período',
  pregunta:'¿Qué consulta trajo cada grupo de tuits, y en qué período? El área es el número de tuits. Haz clic para entrar en un cluster.',
  alto:520, tabla: tabla(['cluster','tópico','calificados','consulta','período','tuits'],
    [].concat(...Object.entries(D.auditoria).map(([cl,bases])=>[].concat(...Object.entries(bases).map(([b,pers])=>
      Object.entries(pers).map(([p,n])=>[cl,(clInfo[cl]||{}).topico||'',clInfo[cl]?pct(clInfo[cl].prop_calificados):'—',b,p,n]))))))});
init(pa.lienzo,{toolbox:TOOLBOX,tooltip:{formatter:q=>{const ruta=(q.treePathInfo||[]).slice(1).map(x=>x.name).join(' → ');
    const info=clInfo[(q.treePathInfo||[])[1]?q.treePathInfo[1].name:''];
    return esc(ruta)+'<br>'+q.value+' tuits'+(info?'<br>'+esc(info.topico)+' · calificados '+pct(info.prop_calificados):'');}},
  series:[{type:'treemap',data:tree,leafDepth:1,roam:false,width:'100%',height:'92%',top:0,
    breadcrumb:{show:true,bottom:0},
    label:{show:true,formatter:q=>q.name+'\n'+q.value,fontSize:11},upperLabel:{show:true,height:20},
    levels:[{itemStyle:{borderColor:'#fcfcfb',borderWidth:3,gapWidth:3},color:PAL},
            {itemStyle:{borderColor:'#fcfcfb',borderWidth:1,gapWidth:1},colorSaturation:[0.35,0.6]},
            {colorSaturation:[0.3,0.5]}]}]});

/* ---- grafos ---- */
function grafo(titulo,pregunta,datos,esRel,nota){
  let grupo='todos';
  const botones=['todos','calificado','no_calificado'].map(k=>'<button class="chip" data-g="'+k+'" aria-pressed="'+(k===grupo)+'">'+k.replace('_',' ')+'</button>').join(' ');
  const p=panel(RAIZ_SEC,{titulo,pregunta,controles:'<div class="grupo"><span>Tuits</span>'+botones+'</div>',alto:600,nota,html:'<p class="aviso av" hidden></p>'});
  const g=init(p.lienzo,{});
  const pintar=()=>{
    const G=datos[grupo];
    const av=p.seccion.querySelector('.av');
    if(!G.aristas.length){av.hidden=false;av.textContent='Con los umbrales actuales no queda ninguna arista para este grupo.';} else av.hidden=true;
    const cats = esRel ? ['entidad','sustantivo'] : [...new Set(G.nodos.map(n=>n.cat))];
    const nmax = Math.max(1,...G.nodos.map(n=>n.n));
    const nodos = G.nodos.filter(n=>esRel || !n.aislado || G.aristas.length===0).map(n=>({id:n.id,name:n.id,value:n.n,
      symbolSize:8+26*Math.sqrt(n.n/nmax),category:esRel?(n.entidad?0:1):cats.indexOf(n.cat),
      label:{show:n.n>=nmax*0.12}}));
    const emax = Math.max(1,...G.aristas.map(e=>esRel?e.n:e.pmi));
    const aristas = G.aristas.map(e=>({source:e.source,target:e.target,value:esRel?e.n:e.pmi,
      lineStyle:{width:esRel?1+3*e.n/emax:0.6+2.4*Math.max(0,e.pmi)/emax,opacity:.55,curveness:esRel?.15:0},
      label:esRel?{show:true,formatter:e.verbo,fontSize:9,color:C.tinta2}:undefined, e}));
    g.setOption({toolbox:TOOLBOX,
      legend:{top:0,left:0,data:esRel?['entidad','sustantivo']:cats,textStyle:{color:C.tinta2,fontSize:11}},
      tooltip:{formatter:q=>q.dataType==='edge'?(esRel?esc(q.data.e.source)+' —'+esc(q.data.e.verbo)+'→ '+esc(q.data.e.target)+' · '+q.data.e.n+' veces'
        :esc(q.data.e.source)+' — '+esc(q.data.e.target)+' · '+q.data.e.n+' tuits · PMI '+fix(q.data.e.pmi,2))
        :esc(q.name)+' · '+q.value+(esRel?' apariciones':' tuits')},
      series:[{type:'graph',layout:'force',roam:true,draggable:true,data:nodos,links:aristas,
        categories:(esRel?['entidad','sustantivo']:cats).map((c,i)=>({name:c,itemStyle:{color:c.startsWith('MX_')?['#c62f2e','#eb6834','#eda100','#b45309','#be123c','#9333ea','#4a3aa7','#0369a1'][i%8]:PAL[i%PAL.length]}})),
        edgeSymbol:esRel?['none','arrow']:['none','none'],edgeSymbolSize:7,
        force:{repulsion:esRel?260:140,edgeLength:esRel?[80,180]:[40,150],gravity:.08},
        label:{position:'right',fontSize:10,color:C.tinta},emphasis:{focus:'adjacency'}}]},true);
  };
  p.seccion.querySelectorAll('button[data-g]').forEach(b=>b.onclick=()=>{grupo=b.dataset.g;
    p.seccion.querySelectorAll('button[data-g]').forEach(x=>x.setAttribute('aria-pressed',String(x.dataset.g===grupo)));pintar();});
  pintar();
}
grafo('Grafo de co-ocurrencia de entidades',
  '¿Qué entidades aparecen juntas en el mismo tuit? Nodo = entidad (tamaño por frecuencia, color por tipo; MX_* viene del diccionario de México). '
  +'Arista = co-ocurrencia en al menos '+D.ficha.min_coocurrencia+' tuits, grosor por PMI.',
  D.grafos_entidades,false,'Se muestran como máximo '+D.ficha.top_nodos+' entidades por grupo. Arrastra los nodos; la rueda hace zoom.');
grafo('Grafo de relaciones sujeto → verbo → objeto',
  'Tripletas extraídas de las dependencias sintácticas de la traducción, con al menos una entidad en un extremo y repetidas al menos '
  +D.ficha.min_tripleta+' veces.',D.grafos_relaciones,true,
  '<b>Exploratorio.</b> El análisis de dependencias sobre traducciones automáticas de tuits es ruidoso: sirve para orientar la lectura, no para reportar relaciones.');

/* ---- log-odds ---- */
/* Los rótulos de los dos lados vienen en los datos (L.lados). El panel no sabe
   qué se compara: así el mismo bloque sirve para calificados contra no
   calificados y para cualquier otro par sin nombrar mal los grupos. */
['lemas','entidades'].forEach(tipo=>{
  const L=D.log_odds[tipo];
  const [LA,LB]=L.lados, nomA=LA.replace(/_/g,' '), nomB=LB.replace(/_/g,' ');
  const filas=[].concat(L[LA].map(f=>Object.assign({lado:nomA},f)),L[LB].map(f=>Object.assign({lado:nomB},f)))
    .sort((a,b)=>a.z-b.z);
  const p=panel(RAIZ_SEC,{titulo:'Vocabulario característico · '+tipo,
    pregunta:'Log-odds con prior informativo (Monroe et al., 2008): z &gt; 0 es característico de '+esc(nomA)+'; z &lt; 0, de '+esc(nomB)+'.',
    alto:Math.max(260,filas.length*16+60),
    tabla:tabla(['término','z','en '+nomA,'en '+nomB],filas.map(f=>[f.termino,f.z,f['n_'+LA],f['n_'+LB]]))});
  init(p.lienzo,{toolbox:TOOLBOX,tooltip:{trigger:'axis',axisPointer:{type:'shadow'}},
    grid:{left:150,right:30,top:10,bottom:30},
    xAxis:Object.assign({type:'value',name:'z',nameLocation:'middle',nameGap:22},EJE_Y),
    yAxis:Object.assign({type:'category',data:filas.map(f=>f.termino)},EJE_X),
    series:[{type:'bar',data:filas.map(f=>({value:f.z,itemStyle:{color:f.z>0?'#1c5cab':'#898781'}}))}]});
});

/* ---- candidatos a falso negativo ---- */
if(D.candidatos===null){
  panel(RAIZ_SEC,{titulo:'Candidatos a falso negativo',alto:0,avisos:['Omitido: no hay valores zero-shot unidos a la rúbrica.']});
} else {
  panel(RAIZ_SEC,{titulo:'Candidatos a falso negativo · para revisión manual',alto:0,
    pregunta:'No calificados por la rúbrica que <b>mencionan México</b> (diccionario) y superan la relevancia '+D.ficha.umbral_relevancia
      +' en alguna dimensión o la prominencia de violencia '+D.ficha.umbral_salience_fn+'.',
    chips:[D.n_candidatos+' candidatos'+(D.n_candidatos>300?' (se muestran 300; el CSV de la sección 17 los trae todos)':'')],
    avisos:['<b>No son errores de la rúbrica.</b> Son tuits que una persona debe leer. No se calcula ninguna tasa de falsos negativos.'],
    html: tabla(['id','idioma','consulta','período','cluster','motivo','texto','traducción','bloqueado'],
      D.candidatos.map(c=>[c.id,c.lang,c.base,c.periodo,'C'+c.cluster+' · '+(c.topico||''),c.motivo_fn,c.texto,c.traduccion,c.con_faltante?'sí':'']))});
}

/* ---- temas ---- */
panel(RAIZ_SEC,{titulo:'Temas fuera de la rúbrica',alto:0,
  pregunta:'Clusters con al menos '+pct(D.ficha.umbral_tema_mexico,0)+' de tuits que mencionan México, como máximo '
    +pct(D.ficha.umbral_tema_calificados,0)+' de calificados y al menos '+D.ficha.min_tamano_tema+' tuits: candidatos a dimensión.',
  html: D.temas.length ? tabla(['cluster','n','menciona México','calificados','términos','entidades','ejemplos'],
      D.temas.map(c=>[nombreCl(c.cluster),c.n,pct(c.prop_mexico),pct(c.prop_calificados),c.terminos.join(', '),
        c.entidades.map(e=>e[0]).join(', '),c.ejemplos.map(e=>e.texto+(e.trad?' — '+e.trad:'')).join('  ‖  ')]))
    : '<p class="nota">Ningún cluster cumple los umbrales.</p>'});

/* ---- ficha ---- */
panel(RAIZ_SEC,{titulo:'Ficha técnica del análisis NLP',alto:0,
  html:tabla(['parámetro','valor'],Object.entries(D.ficha).map(([k,v])=>[k,Array.isArray(v)?v.join(', '):String(v)]))});
"""

if DATOS_NLP is not None:
    registrar_seccion("nlp", "Análisis NLP del corpus", DATOS_NLP, JS_NLP)
    PAGINA_NLP = mostrar_seccion("nlp")
    print()
    print("FICHA TECNICA NLP")
    for _k, _v in DATOS_NLP["ficha"].items():
        print("   {:<26} {}".format(_k, _v))

## 17. Descarga de resultados

Todo va a `salida/`, nunca al directorio de entrada.

| archivo | qué trae | para qué |
|---|---|---|
| `resultados_enriquecidos.csv` | **una fila por tuit**: las 68 columnas del archivo + zero-shot + probabilidades crudas + rúbrica codificada + NLP | abrir en Excel, compartir, cruzar |
| `resultados_largo.csv` | una fila por texto × método × dimensión | analizar y graficar |
| `candidatos_fn.csv` | los candidatos a falso negativo, con una columna vacía `decision_revisor` | **revisión manual** |
| `asociacion_criterios.csv` | una fila por par × lectura | la tabla de la sección 15 |
| `figuras_v3.html` | las secciones 2, 14, 15 y 16, autocontenidas | la tesis |
| `figuras_dimensiones.html` | las figuras del instrumento (sección 13) | la tesis |
| `zeroshot_corrida.json.gz` | la corrida completa | `CARGAR_ZEROSHOT` en la siguiente sesión |

### Columnas nuevas del ancho

```
  … las 68 originales (incluidas las 16 de rúbrica, sin tocar)
  + atmosfera__nuevo · atmosfera__relevancia · atmosfera__actual · atmosfera__traducido · …
  + p__compuerta__atmosfera · p__atmosfera__pos · p__violencia__violence · …   (probabilidades)
  + rubrica__atmosfera__clase · rubrica__atmosfera__nivel_num · …             (codificación única)
  + grupo_rubrica · con_faltante
  + nlp__cluster · nlp__topico · nlp__menciona_mexico · nlp__umap_x · nlp__umap_y
  + nlp__candidato_fn · nlp__motivo_fn
  + huella_instrumento
```

`rubrica__*__nivel_num` vale **0 para la ausencia** (tanto `NO` como nivel 0) y queda **vacío para los
bloqueados**. `nlp__cluster` vale −1 para el ruido y −2 para los tuits sin texto útil.

### Lo que el ancho no trae, a propósito

**Ninguna columna multiplica valencia por relevancia.** «Habló y juzgó neutro» (0.02 × 0.91) y «no
habló del tema» (0.60 × 0.03) dan el mismo 0.018. Filtra por `relevancia` y **después** lee la
valencia.

In [ ]:
import os as _os
import pandas as pd

# ===========================================================================
# EXPORTACION DE RESULTADOS
#
# Sale de LARGO y de PROBS (el punto de extraccion sigue siendo uno solo), mas
# los marcos de la rubrica y del NLP. Se une SIEMPRE por identificador.
# ===========================================================================

SEP = "__"   # separador de nombres compuestos: dimension__metodo


def _tabla_ancha():
    piv = (LARGO.pivot_table(index="texto_id", columns=["dimension", "metodo"],
                             values="valencia", aggfunc="first"))
    piv.columns = [SEP.join(c) for c in piv.columns]

    rel = (LARGO[LARGO.metodo == "nuevo"]
           .pivot_table(index="texto_id", columns="dimension", values="relevancia", aggfunc="first"))
    rel = rel.dropna(axis=1, how="all")
    rel.columns = [c + SEP + "relevancia" for c in rel.columns]
    ancha = piv.join(rel, how="outer")

    # probabilidades crudas: con la compuerta incluyen p__compuerta__<dimension>,
    # de la que sale la relevancia
    if EXPORTAR_PROBABILIDADES and len(PROBS):
        pr = PROBS.pivot_table(index="texto_id", columns=["grupo", "etiqueta"],
                               values="probabilidad", aggfunc="first")
        pr.columns = ["p" + SEP + SEP.join(c) for c in pr.columns]
        ancha = ancha.join(pr, how="outer")

    ancha = ancha.round(6).reset_index()

    if CORPUS_DF is not None and COLUMNAS.get("tweet_id"):
        # El archivo ENTERO, con todas sus columnas, mas los resultados. Por
        # identificador y no por posicion: el dedupe y el muestreo mueven filas.
        col_id = COLUMNAS["tweet_id"]
        origen = CORPUS_DF.copy()
        origen[col_id] = origen[col_id].astype(str)
        ancha = origen.merge(ancha, left_on=col_id, right_on="texto_id", how="inner")
        ancha = ancha.drop(columns=["texto_id"])
        clave = col_id
    else:
        attrs = (LARGO[["texto_id", "texto", "frase", "lang", "objetivo"]]
                 .drop_duplicates("texto_id"))
        ancha = attrs.merge(ancha, on="texto_id", how="right")
        clave = "texto_id"

    # --- rubrica con la codificacion unica -------------------------------
    if HAY_RUBRICA and clave != "texto_id":
        cod = RUBRICA_TUIT[["id"]].copy()
        for c in CRITERIOS:
            cod["rubrica" + SEP + c["corto"] + SEP + "clase"] = RUBRICA_TUIT["clase_" + c["corto"]]
            cod["rubrica" + SEP + c["corto"] + SEP + "nivel_num"] = RUBRICA_TUIT["nivel_" + c["corto"]]
        cod["grupo_rubrica"] = RUBRICA_TUIT["grupo"]
        cod["con_faltante"] = RUBRICA_TUIT["con_faltante"]
        ancha = ancha.merge(cod, left_on=clave, right_on="id", how="left",
                            suffixes=("", "_rub")).drop(columns=["id_rub"] if "id_rub" in ancha else [])
        if "id" in ancha.columns and clave != "id":
            ancha = ancha.drop(columns=["id"])

    # --- NLP ---------------------------------------------------------------
    if globals().get("NLP_TUIT") is not None and clave != "texto_id":
        n = NLP_TUIT[["id", "cluster", "topico", "menciona_mexico", "umap_x", "umap_y",
                      "candidato_fn", "motivo_fn"]].copy()
        n.columns = ["id"] + ["nlp" + SEP + c for c in n.columns[1:]]
        ancha = ancha.merge(n, left_on=clave, right_on="id", how="left", suffixes=("", "_nlp"))
        if "id_nlp" in ancha.columns:
            ancha = ancha.drop(columns=["id_nlp"])

    ancha["huella_instrumento"] = HUELLA_INSTRUMENTO
    return ancha, clave


TABLA_ANCHA, _CLAVE = _tabla_ancha()

_cols_largo = ["texto_id", "tweet_id", "created_at", "lang", "lang_grupo",
               "estrato", "modalidad", "frase", "objetivo",
               "metodo", "origen_texto", "dimension", "escala", "grupo",
               "valencia", "relevancia", "engagement_weight", "label_sentimiento"]
TABLA_LARGA = LARGO[[c for c in _cols_largo if c in LARGO.columns]].round(6)

# ---------------------------------------------------------------------------
# GUARDIA. La misma de los marcos, aplicada a lo que SALE del notebook.
#   hablo y juzgo neutro   0.02 * 0.91 = 0.018
#   no hablo del tema      0.60 * 0.03 = 0.018
# ---------------------------------------------------------------------------
_sospechosas = [c for c in TABLA_ANCHA.columns
                if any(k in str(c).lower() for k in
                       ("ponderad", "x_relev", "por_relev", "score_final",
                        "valencia_relev", "compuest"))]
assert not _sospechosas, (
    "La tabla ancha tiene columnas que parecen combinar valencia y relevancia: "
    "{}. Ese producto reintroduce el cero ambiguo; quitalas.".format(_sospechosas))

# --- escritura ---------------------------------------------------------------
# utf-8-sig: sin el BOM, Excel abre los acentos y el japones como basura.
TABLA_ANCHA.to_csv(RUTA_CSV_ANCHO, index=False, encoding="utf-8-sig")
_escritos = [RUTA_CSV_ANCHO]
if EXPORTAR_LARGO:
    TABLA_LARGA.to_csv(RUTA_CSV_LARGO, index=False, encoding="utf-8-sig")
    _escritos.append(RUTA_CSV_LARGO)

# candidatos a falso negativo: material de revision, con su columna de decision
RUTA_CANDIDATOS = _os.path.join(SALIDA_DIR, "candidatos_fn.csv")
if globals().get("CANDIDATOS_FN") is not None and len(CANDIDATOS_FN):
    _cols = ["id", "lang", "base", "periodo", "fecha_str", "texto", "traduccion", "motivo_fn",
             "cluster", "topico", "con_faltante"]
    _cols += [c for c in CANDIDATOS_FN.columns if c.endswith("__relevancia")]
    _cols += [c for c in CANDIDATOS_FN.columns if c == "violence_salience__valencia"]
    _cols += [c for c in CANDIDATOS_FN.columns if c.startswith("justificacion_")]
    _cand = CANDIDATOS_FN[[c for c in _cols if c in CANDIDATOS_FN.columns]].copy()
    _cand.insert(0, "revision", "CANDIDATO A FALSO NEGATIVO: requiere revision manual; no es un error de la rubrica")
    _cand["decision_revisor"] = ""
    _cand.to_csv(RUTA_CANDIDATOS, index=False, encoding="utf-8-sig")
    _escritos.append(RUTA_CANDIDATOS)
elif globals().get("CANDIDATOS_FN") is not None:
    print("Sin candidatos a falso negativo: no se escribe un archivo vacio.")

RUTA_ASOCIACION = _os.path.join(SALIDA_DIR, "asociacion_criterios.csv")
if globals().get("ASOCIACION") is not None:
    _asoc = ASOCIACION.copy()
    if "estratos" in _asoc:
        _asoc["estratos"] = _asoc["estratos"].map(
            lambda d: " · ".join("{}: {}".format(k, v) for k, v in d.items()) if isinstance(d, dict) else "")
    _asoc.to_csv(RUTA_ASOCIACION, index=False, encoding="utf-8-sig")
    _escritos.append(RUTA_ASOCIACION)

RUTA_FIGURAS_V3 = _os.path.join(SALIDA_DIR, "figuras_v3.html")
if SECCIONES_V3:
    with open(RUTA_FIGURAS_V3, "w", encoding="utf-8") as _fh:
        _fh.write(pagina_v3(titulo="Rúbrica, validación, asociación y NLP · v3"))
    _escritos.append(RUTA_FIGURAS_V3)
if _os.path.exists(RUTA_HTML):
    _escritos.append(RUTA_HTML)

# volcado completo: la demostracion y el A/B del ancla ya existen
if not CARGAR_ZEROSHOT or VOLCADO_PREVIO is not None:
    escribir_volcado({"filas_demo": globals().get("filas_demo") or [],
                      "comparacion_ancla": globals().get("COMPARACION_ANCLA")})
    _escritos.append(RUTA_VOLCADO)

print("=" * 78)
print("RESULTADOS EXPORTADOS")
print("=" * 78)
for _f in _escritos:
    print("  {:<44} {:>9.1f} KB".format(_f, _os.path.getsize(_f) / 1024))
print()
print("  ancho:  {} filas x {} columnas   (clave: {})".format(len(TABLA_ANCHA), len(TABLA_ANCHA.columns), _CLAVE))
if CORPUS_DF is not None:
    print("          {} columnas originales del archivo + {} añadidas".format(
        len(CORPUS_DF.columns), len(TABLA_ANCHA.columns) - len(CORPUS_DF.columns)))
print("  largo:  {} filas x {} columnas".format(len(TABLA_LARGA), len(TABLA_LARGA.columns)))
print()
print("  PROCEDENCIA: {}".format(PROCEDENCIA))
if PROCEDENCIA == "corpus":
    print("  Origen: {} · huella del instrumento {}".format(CORPUS_RUTA, HUELLA_INSTRUMENTO))
    print("  Salen las filas EVALUADAS ({}).".format(len(TABLA_ANCHA)))
    if N_MUESTRA:
        print("  N_MUESTRA = {} esta puesto: ponlo en None para el corpus completo.".format(N_MUESTRA))
    print("  Valores DESCRIPTIVOS, contrastados contra una rubrica aplicada por un modelo de")
    print("  lenguaje: NO validados contra codificacion humana.")
print()
print("  Recuerda: valencia y relevancia NO se multiplican. Filtra por relevancia y")
print("  DESPUES lee la valencia.")
print("=" * 78)

try:
    from google.colab import files as _files
    for _f in _escritos:
        _files.download(_f)
except Exception as _e:
    print()
    print("(descarga automatica no disponible: {})".format(_e))
    print("Descargalos desde el panel de archivos de Colab, a la izquierda (carpeta {}/).".format(SALIDA_DIR))

## 18. Cómo leer esto

**En la sección 11** está el argumento contra el método léxico. La misma frase en seis idiomas, y el
método actual sale **positivo en inglés** para un texto que dice que la policía era corrupta. La
dispersión es el número que se reporta, y como esa dimensión no incorpora el sentimiento, es exacto.

**La columna `relevancia`** es información que el método actual no tiene. Un `perspectiva_politica`
de 0.00 con relevancia 0.05 significa «no habló de autoridades». Con relevancia 0.80 significa
«habló de autoridades y juzgó neutro».

### La rúbrica (sección 2)

- **Ausencia = nivel 0 = `aplicable NO`.** Es una sola clase. La bandera la agregó el motor de
  calificación, no la rúbrica, y el modelo la usó en lugar del 0. Los **bloqueados** son dato
  faltante: no cuentan como ausencia ni como nivel bajo.
- En estos datos, «ausencia» significa **«el modelo juzgó que el tuit es ajeno al criterio»**. Es
  equivalente en la práctica al descriptor 0 de la rúbrica, pero conviene decirlo así en la tesis.
- **Grupo calificado** = algún criterio con nivel > 0: 144 tuits en esta entrada.

### La validación (sección 14)

- **Es validez convergente entre dos instrumentos automáticos.** La rúbrica también es un modelo de
  lenguaje. Que coincidan no prueba que acierten.
- **Lee el constructo antes que el AUC.** `perspectiva_politica` mide autoridades y gobierno; la
  rúbrica, competencia funcional (incluye infraestructura, logística y seguridad). Un AUC bajo ahí
  es esperable por diseño.
- **Busca los puntos ciegos en el plano.** En la vista previa del 11/09, el nivel 1 de atmósfera
  (hostilidad hacia México) caía con relevancia ≈ 0.07: la compuerta pregunta por el ánimo vivido, no
  por el afecto hacia el país.

### La asociación (sección 15)

- **Lee φ y la razón de momios, no la p.** Con 2 600 tuits todo sale significativo.
- **Corpus completo** infla la asociación por los tuits sin ningún criterio. **Solo calificados** la
  vuelve negativa por **sesgo de Berkson**. **Estratificada** es la lectura más defendible.
- **Perspectiva política × violencia** está asociada en parte por definición: la rúbrica mete la
  gestión de seguridad en perspectiva política.

### El NLP (sección 16)

- **Candidatos a falso negativo ≠ falsos negativos.** Revísalos en `candidatos_fn.csv` y llena
  `decision_revisor`. Solo después de esa revisión se puede hablar de una tasa.
- **Tópicos, entidades y grafos salen de una traducción automática.** Hay que declararlo, junto con
  las 5 traducciones completadas a mano con otro modelo.
- **Los clusters dependen de sus parámetros.** Repórtalos con la ficha técnica y, si una conclusión
  importa, comprueba que sobrevive a otro `HDBSCAN_MIN_CLUSTER`.

### Antes de adoptar el instrumento, sigue haciendo falta la validación humana

1. Codifica a mano una muestra (≈200 tuits), estratificada por grupo de la rúbrica e incluyendo
   candidatos a falso negativo.
2. Calcula el acuerdo de cada instrumento —léxico, zero-shot y rúbrica— contra esos códigos (alfa de
   Krippendorff; la infraestructura está en `wc2026_analyzer/validation/agreement.py`, umbral 0.67).
3. Adopta el que gane, con el número en la mano.